<div style="border-bottom:4px solid #238B45;padding:16px 0">
  <h1 style="color:#166534"><strong>RUTA SILENCIO</strong></h1>
  <h2>Contexto, fundamentos y ETL auditable</h2>
  <p>
    Data Challenge 2026 | SOBUSA | Extreme Technologies<br>
  </p>
</div>


## **1. Contexto y problema**
La operación requiere anticipar pasajeros y recomendar separación entre despachos, buses por hora y flota, respetando capacidad y restricciones. El reto contempla horizontes de 30, 60 y 120 minutos, incertidumbre, explicabilidad, alertas, trazabilidad y actualización incremental. Python, FastAPI y PostgreSQL/PostGIS forman parte de la arquitectura del producto completo; esta fase prepara sus insumos.

Los datos combinan registros de sensores, programaciones de viajes, catálogos y agregados históricos. Estas unidades no son intercambiables. Una programación puede cancelarse; un agregado agrupa muchos viajes; las subidas totales de un recorrido no representan personas simultáneamente a bordo.

**Problema de datos:** establecer qué mediciones son utilizables, con qué cobertura, semántica y restricciones. Una recomendación de frecuencia basada en conteos incompletos puede confundir menor oferta o fallas de medición con menor demanda.

Los resultados preliminares adjuntos reportan cobertura desigual, pasajeros negativos por segmento, capacidad nominal en cero y calendarios de otro proceso. Son antecedentes por verificar, no conclusiones de esta ejecución.

## **2. Objetivos y preguntas**
**Objetivo general:** construir una capa ETL reproducible para SILENCIO que preserve fuentes, valide integridad y documente aptitud por uso antes del EDA estadístico y el pronóstico.

1. Inventariar fuentes, esquema, llaves e índices sin descargar tablas completas innecesariamente.
2. Extraer programaciones 2023–2025 y catálogos relacionados con filtros parametrizados.
3. Medir faltantes, duplicados, invalidez temporal, conteos negativos e inconsistencias relacionales.
4. Producir diccionario operacional y reglas versionadas, con semántica pendiente identificada.
5. Integrar viajes con rutas y vehículos sin multiplicar filas y conservar evidencia de errores.
6. Exportar tablas originales, transformadas, incidencias, cuarentenas, consultas y manifiesto.

**Preguntas de calidad:** ¿qué periodo cubre cada fuente?, ¿las llaves son únicas?, ¿qué valores son ausentes o inválidos?, ¿las fechas son coherentes?, ¿los promedios reconcilian con totales?, ¿las referencias pertenecen al mismo proceso?, ¿qué impide analizar carga a bordo y demanda por franja?

**Criterio verificable:** cada tabla consultada tiene estado explícito, cada regla genera un conteo y un tratamiento, cada extracción registra filtros y huella, y cada unión exige cardinalidad válida.

## **3. Marco teórico aplicado**
### 3.1 Demanda observada y oferta
Una subida registrada mide un abordaje, no una persona única ni toda la demanda potencial. La demanda no atendida por falta de oferta, cancelaciones o saturación no se observa necesariamente. Los conteos de viajes describen oferta. La terminología de abordajes y conteos automáticos está respaldada por la FTA [1]; sus criterios no se adoptan como regulación colombiana.

### 3.2 Flujo y carga
Para mediciones compatibles, la carga puede expresarse como \(L_k=L_{k-1}+S_k-B_k\). La fórmula exige condición inicial, orden, sentido y conteos confiables. No se aplica automáticamente al campo `pasajeros` negativo de segmentos. Ocupación relativa = carga simultánea / capacidad válida; pasajeros totales del viaje / capacidad no mide saturación simultánea.

### 3.3 Intervalo y ciclo
El intervalo separa despachos consecutivos comparables. `fechasalida` está documentada como programada y `fechainicio` como inicio efectivo; ambas se conservan. Un tiempo de viaje no prueba por sí solo el ciclo completo. La configuración vigente de ruta o vehículo no reconstruye necesariamente la historia.

### 3.4 Granularidad e integridad
Un JOIN uno-a-muchos puede repetir medidas del viaje y producir doble conteo. Se valida unicidad antes de integrar. NULL representa ausencia; cero es un valor observado cuya semántica requiere contraste. Restricciones SQL y validaciones analíticas cubren dimensiones diferentes.

### 3.5 Faltantes y errores
MCAR supone ausencia independiente de valores; MAR permite dependencia de variables observadas; MNAR depende del valor no observado [4]. Este ETL reporta faltantes y cobertura, pero no declara el mecanismo a partir de una tabla de porcentajes. Un valor extremo válido se conserva; una inconsistencia se marca, y no se convierte en cero por conveniencia.

### 3.6 Tiempo y reproducibilidad
La dependencia temporal y la validación con información pasada se desarrollan en Hyndman y Athanasopoulos [2]. Las fuentes agregadas pueden contener información futura respecto a una predicción histórica; en esta fase se preservan como perfiles, no se usan como features. Cada extracción usa una transacción de solo lectura; las consultas de un bloque pueden compartir un snapshot de lectura repetible [3]. Los bloques se ejecutan en momentos diferentes y el manifiesto lo registra: no se afirma una fotografía global idéntica.

### 3.7 ETL y arquitectura
**Extract:** lectura limitada y parametrizada. **Transform:** tipado, reglas y uniones verificadas en capa propia. **Load:** archivos locales con manifiesto; no INSERT/UPDATE en tablas operativas. Las marcas `ultima_actualizacion` sirven para planificar lecturas incrementales solo cuando se conoce su contrato. No equivalen a fecha de demanda ni autorizan eliminar datos históricos.

## **4. Instalación y configuración privada**
Ejecutar la siguiente celda una vez si faltan dependencias. Reiniciar el kernel si el entorno lo requiere. No se lee el `.env` adjunto de terceros ni se muestran credenciales. Este notebook escribe un ejemplo sin secretos en la carpeta de trabajo.

Completar `.env` junto al notebook con `DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD` y `DB_SSLMODE`. El puerto local 5432 no implica que el servidor remoto sea el PostgreSQL de tu Mac. El acceso ocurre desde el equipo donde corre el notebook.

Una ejecución crea una carpeta nueva, nunca reemplaza una ejecución anterior. El reporte HTML usa verde como color principal y magenta como complemento visual de incidencias; no incluye gráficos de distribución.

### **4.1. Entorno**

In [51]:
# Ejecutar manualmente solo si faltan paquetes; no instala al ejecutar todo.
# %pip install "pandas>=2.1,<3" "numpy>=1.26,<3" "SQLAlchemy>=2,<3" "psycopg[binary]>=3.1,<4" "python-dotenv>=1,<2" "ipython>=8,<10" "pyarrow>=14,<24" "nbformat>=5,<6"
import os, json, hashlib, uuid, html, platform, time, re
from pathlib import Path
from datetime import datetime
from contextlib import contextmanager
from importlib.metadata import version
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text, URL
from dotenv import load_dotenv
from IPython.display import display, HTML
from datetime import datetime
from zoneinfo import ZoneInfo
from uuid import uuid4
import pyarrow.parquet as pq
import shutil

### **4.2. Configuración y credenciales**

El periodo usa la salida **programada**, no el inicio real. Los viajes sin `fechasalida` quedan fuera y se registran por separado. No interpretarlos como inexistentes. Los agregados históricos se conservan por ruta sin recortar el año para poder detectar diferencias de cobertura.


In [53]:
CFG = {
    "id_ruta": 2, "nombre_ruta": "SILENCIO", "schema": "public", "proceso": "sobusa",
    "inicio": "2023-01-01", "fin_exclusivo": "2026-01-01",
    "zona": "America/Bogota", "lote": 10000, "timeout_ms": 120000,
    "max_filas": None, "extraer_segmentos": True, "max_segmentos": 2000000,
}
if CFG["id_ruta"] not in (2, 16, 21):
    raise ValueError("Ruta fuera del alcance del equipo")
if pd.Timestamp(CFG["inicio"]) >= pd.Timestamp(CFG["fin_exclusivo"]):
    raise ValueError("Periodo inválido")
load_dotenv(Path.cwd() / ".env", override=False)
faltan = [k for k in ("DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD") if not os.getenv(k)]
if faltan:
    raise ValueError("Completar .env: " + ", ".join(faltan))
RUN_ID = datetime.now(ZoneInfo(CFG["zona"])).strftime("%Y%m%dT%H%M%S") + "_" + uuid4().hex[:8]
BASE = Path("salidas_extraccion") / f"ruta_{CFG['id_ruta']}" / RUN_ID
for nombre in ("raw", "reportes", "sql"):
    (BASE / nombre).mkdir(parents=True, exist_ok=False)
RAW, REGISTRO = {}, []
def escribir_json(path, objeto):
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(objeto, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)
escribir_json(BASE / "configuracion.json", CFG)
(BASE / "requirements_ejecutados.txt").write_text(
    "\n".join(f"{p}=={version(p)}" for p in ["pandas", "SQLAlchemy", "psycopg", "python-dotenv", "pyarrow"]) + "\n",
    encoding="utf-8",
)
print("Ejecución:", RUN_ID, "| Salidas:", BASE.resolve())


Ejecución: 20261010T182243_04b6bd0f | Salidas: /Users/leoporojano/Documents/Data Challenge/salidas_extraccion/ruta_2/20261010T182243_04b6bd0f


### **4.3. Conexión y transacciones breves**
`URL.create` maneja las credenciales sin construir una URL manual. Cada bloque usa solo lectura, límite de tiempo por sentencia y zona horaria explícita. `REPEATABLE READ` garantiza una instantánea dentro del bloque; distintas transacciones pueden ver cambios distintos.

Si ocurre un error, no continuar usando la misma transacción: PostgreSQL puede haberla dejado abortada. La extracción crítica detiene el notebook; las fuentes opcionales se ejecutan en bloques independientes y registran su fallo. El cierre del contexto devuelve la conexión al pool; no abre una sesión durante todo el notebook.

In [54]:
engine = create_engine(
    URL.create("postgresql+psycopg", username=os.environ["DB_USER"],
               password=os.environ["DB_PASSWORD"], host=os.environ["DB_HOST"],
               port=int(os.getenv("DB_PORT", "5432")), database=os.environ["DB_NAME"]),
    pool_pre_ping=True,
    connect_args={"connect_timeout": 15, "sslmode": os.getenv("DB_SSLMODE", "require"),
                  "application_name": "sobusa_extraccion"},
)
@contextmanager
def lectura():
    with engine.connect().execution_options(isolation_level="REPEATABLE READ") as conn:
        with conn.begin():
            conn.execute(text("SET TRANSACTION READ ONLY"))
            for clave, valor in [("statement_timeout", str(CFG["timeout_ms"])),
                                 ("lock_timeout", "5000"), ("TimeZone", CFG["zona"])]:
                conn.execute(text("SELECT set_config(:clave, :valor, true)"),
                             {"clave": clave, "valor": valor})
            yield conn
with lectura() as conn:
    prueba = conn.execute(text("""SELECT current_database() AS base,
        current_setting('transaction_read_only') AS solo_lectura,
        current_setting('TimeZone') AS zona, version() AS version""")).mappings().one()
    if prueba["solo_lectura"] != "on":
        raise RuntimeError("La transacción no está en solo lectura")
    print(dict(prueba))


{'base': 'dataset', 'solo_lectura': 'on', 'zona': 'America/Bogota', 'version': 'PostgreSQL 14.22 on aarch64-unknown-linux-gnu, compiled by aarch64-unknown-linux-gnu-gcc (GCC) 12.4.0, 64-bit'}


### **4.4. Inventario real: consultas pequeñas**
No hacer `COUNT(*)` sobre todas las tablas. `n_live_tup` es aproximado; un conteo exacto solo se justifica para un filtro concreto. Las restricciones visibles dependen de los permisos del usuario: una lista vacía no demuestra que no existan. Se inventarían columnas y restricciones, no datos sensibles.


In [55]:
def consulta_pequena(nombre, sql, params=None, conn=None):
    # Solo metadatos o agregados de una fila. Nunca usar para tablas de hechos.
    if conn is None:
        with lectura() as c:
            return consulta_pequena(nombre, sql, params, c)
    inicio = time.perf_counter()
    (BASE / "sql" / f"{nombre}.sql").write_text(sql, encoding="utf-8")
    with conn.execution_options(stream_results=False).execute(text(sql), params or {}) as result:
        filas = result.fetchmany(20001)
        if len(filas) > 20000:
            raise ValueError("Consulta de metadatos supera 20000 filas")
        df = pd.DataFrame(filas, columns=result.keys())
    REGISTRO.append({"fuente": nombre, "estado": "metadatos", "filas": len(df),
                     "segundos": time.perf_counter() - inicio, "parametros": params or {}})
    escribir_json(BASE / "reportes" / "registro.json", REGISTRO)
    return df
with lectura() as c:
    ESQUEMA = consulta_pequena("esquema", """SELECT table_name AS tabla,column_name AS columna,data_type,udt_name,
        is_nullable,ordinal_position,numeric_precision,numeric_scale
        FROM information_schema.columns WHERE table_schema=:s ORDER BY table_name,ordinal_position""", {"s":CFG["schema"]}, conn=c)
    INVENTARIO = consulta_pequena("inventario", """SELECT relname AS tabla,n_live_tup AS filas_aproximadas,
        pg_total_relation_size(relid) AS bytes_total,last_analyze,last_autoanalyze
        FROM pg_stat_user_tables WHERE schemaname=:s ORDER BY relname""", {"s":CFG["schema"]}, conn=c)
    INDICES = consulta_pequena("indices", "SELECT tablename AS tabla,indexname,indexdef FROM pg_indexes WHERE schemaname=:s", {"s":CFG["schema"]}, conn=c)
    LLAVES = consulta_pequena("llaves", """SELECT tc.table_name AS tabla,tc.constraint_name,tc.constraint_type,
        kcu.column_name,kcu.ordinal_position
        FROM information_schema.table_constraints tc LEFT JOIN information_schema.key_column_usage kcu
        ON tc.constraint_catalog=kcu.constraint_catalog AND tc.constraint_schema=kcu.constraint_schema
        AND tc.constraint_name=kcu.constraint_name AND tc.table_name=kcu.table_name
        WHERE tc.table_schema=:s ORDER BY tc.table_name,tc.constraint_name,kcu.ordinal_position""", {"s":CFG["schema"]}, conn=c)
COLS = {t:set(g.columna) for t,g in ESQUEMA.groupby("tabla")}
for name,df in [("esquema",ESQUEMA),("inventario",INVENTARIO),("indices",INDICES),("llaves",LLAVES)]: df.to_csv(BASE / "reportes" / f"{name}.csv", index=False)

# Copia para presentación: no modifica los metadatos originales.
inventario_visible = INVENTARIO.copy()

inventario_visible["filas_aproximadas"] = (
    inventario_visible["filas_aproximadas"]
    .map(
        lambda valor: (
            "Sin estimación validada"
            if pd.isna(valor) or valor == 0
            else f"{int(valor):,}"
        )
    )
)

for columna in ["last_analyze", "last_autoanalyze"]:
    inventario_visible[columna] = (
        inventario_visible[columna]
        .astype("string")
        .fillna("Sin fecha registrada")
    )

display(inventario_visible)

inventario_visible.to_csv(
    BASE / "reportes" / "inventario_presentacion.csv",
    index=False
)

def qident(valor):
    return '"' + str(valor).replace('"', '""') + '"'
def qtable(tabla):
    return qident(CFG["schema"]) + "." + qident(tabla)
def exigir(tabla, campos):
    faltantes = set(campos) - COLS.get(tabla, set())
    if faltantes:
        raise ValueError(f"Esquema incompatible en {tabla}: faltan {sorted(faltantes)}")
def seleccion(tabla, campos):
    tipos = (
        ESQUEMA.loc[ESQUEMA["tabla"].eq(tabla)]
        .set_index("columna")["udt_name"]
        .to_dict()
    )

    expresiones = []

    for columna in campos:
        if columna not in COLS.get(tabla, set()):
            continue

        nombre = qident(columna)

        if tipos.get(columna) in {"geometry", "geography"}:
            expresiones.append(f"{nombre}::text AS {nombre}")
        else:
            expresiones.append(nombre)

    if not expresiones:
        raise ValueError(f"Sin columnas disponibles: {tabla}")

    return ", ".join(expresiones)


,tabla,filas_aproximadas,bytes_total,last_analyze,last_autoanalyze
0,Procesos,Sin estimación validada,32768,Sin fecha registrada,Sin fecha registrada
1,conteo_pasajeros_destino,Sin estimación validada,44084854784,Sin fecha registrada,Sin fecha registrada
2,dias_semana,Sin estimación validada,40960,Sin fecha registrada,Sin fecha registrada
3,estacionalidades,Sin estimación validada,32768,Sin fecha registrada,Sin fecha registrada
4,estacionalidades_anio,Sin estimación validada,65536,Sin fecha registrada,Sin fecha registrada
5,novedades_viales,Sin estimación validada,65536,Sin fecha registrada,Sin fecha registrada
6,novedades_viales_fecha,Sin estimación validada,24576,Sin fecha registrada,Sin fecha registrada
7,novedades_viales_ruta,Sin estimación validada,8192,Sin fecha registrada,Sin fecha registrada
8,progvehiculos,Sin estimación validada,1021837312,Sin fecha registrada,Sin fecha registrada
9,rutas,Sin estimación validada,16384,Sin fecha registrada,Sin fecha registrada


Las filas aproximadas provienen de estadísticas del servidor. Los valores cero se presentan como "Sin estimación validada", porque no permiten concluir que la tabla esté vacía. Los conteos de extracción se reportarán por separado para la ruta y el periodo seleccionados.

### **4.5. Extracción a disco: un lote a la vez**
Cada tabla se guarda como carpeta de partes Parquet. No hay `concat`, lista de DataFrames ni copia de cada tabla. Pandas usa tipos Arrow para conservar enteros anulables sin convertirlos a flotantes.

Los archivos temporales se publican solo al completar la extracción y cerrar la transacción. Si falla, se eliminan las partes incompletas y se conserva el registro. Los límites detienen la extracción; nunca se presenta una muestra truncada como tabla completa. El tamaño de lote limita filas, no bytes: reducirlo si existen columnas muy grandes.

In [56]:
def extraer(tabla, where="TRUE", p=None, conn=None, max_filas=None, obligatoria=False):
    params = p or {}
    limite = CFG["max_filas"] if max_filas is None else max_filas
    if tabla not in COLS:
        REGISTRO.append({"fuente": tabla, "estado": "no_disponible", "filas": None})
        escribir_json(BASE / "reportes" / "registro.json", REGISTRO)
        if obligatoria:
            raise ValueError("Fuente crítica ausente: " + tabla)
        return None
    campos = CAMPOS[tabla]
    sql = f"SELECT {seleccion(tabla, campos)} FROM {qtable(tabla)} WHERE {where}"
    destino = BASE / "raw" / tabla
    temporal = BASE / "raw" / (tabla + ".incompleta")
    if destino.exists() or temporal.exists():
        raise FileExistsError(f"{tabla} ya fue procesada: iniciar una ejecución nueva")
    temporal.mkdir()
    (BASE / "sql" / f"{tabla}.sql").write_text(sql, encoding="utf-8")
    inicio = time.perf_counter()
    n, partes = 0, 0
    def recibir(c):
        nonlocal n, partes
        # La opción se restablece antes de ejecutar otros comandos SQL en la conexión.
        c.execution_options(stream_results=True)
        try:
            lotes = pd.read_sql_query(text(sql), c, params=params,
                chunksize=CFG["lote"], coerce_float=False, dtype_backend="pyarrow")
            try:
                for lote in lotes:
                    n += len(lote)
                    if limite is not None and n > limite:
                        raise ValueError(f"Extracción supera el límite de {limite} filas")
                    if len(lote):
                        lote.to_parquet(temporal / f"parte-{partes:05d}.parquet", index=False)
                        partes += 1
                    del lote
            finally:
                lotes.close()
        finally:
            c.execution_options(stream_results=False)
    estado = {"fuente": tabla, "parametros": params,
              "sql_sha256": hashlib.sha256(sql.encode()).hexdigest(),
              "columnas_ausentes": sorted(set(campos) - COLS[tabla]),
              "inicio": datetime.now(ZoneInfo(CFG["zona"])).isoformat()}
    try:
        if conn is None:
            with lectura() as c:
                recibir(c)
        else:
            recibir(conn)
        if obligatoria and n == 0:
            raise ValueError(f"Fuente crítica vacía: {tabla}. Revisar filtros")
        temporal.replace(destino)
        RAW[tabla] = destino
        estado.update(estado="extraida" if n else "vacia_en_filtro", filas=n, partes=partes)
    except Exception as exc:
        shutil.rmtree(temporal, ignore_errors=True)
        original = getattr(exc, "orig", exc)
        # No registrar cadenas de conexión ni mensajes con parámetros privados.
        estado.update(estado="fallida", filas=None, filas_recibidas=n,
                      error=type(exc).__name__, sqlstate=getattr(original, "sqlstate", None))
        REGISTRO.append({**estado, "segundos": time.perf_counter() - inicio})
        escribir_json(BASE / "reportes" / "registro.json", REGISTRO)
        if obligatoria or conn is not None:
            raise
        print(f"Fuente opcional fallida: {tabla}; revisar registro y causa en entorno local")
        return None
    estado["segundos"] = time.perf_counter() - inicio
    REGISTRO.append(estado)
    escribir_json(BASE / "reportes" / "registro.json", REGISTRO)
    print(f"{tabla}: {n:,} filas · {partes} partes")
    return destino


### **4.6. Fuentes y filtros**
Viajes y vehículos son críticos. Los vehículos se seleccionan por su participación histórica en los viajes, aunque su ruta actual sea otra. No filtrar pasajeros nulos, ceros, cancelaciones ni duraciones: esa evaluación pertenece a la limpieza.

Los calendarios se conservan completos para revisar referencias entre procesos; no asignar automáticamente etiquetas SOBUSA a referencias ajenas. Las tablas consolidadas no representan observaciones equivalentes a viajes. La tabla de sensores queda pendiente: no se descarga en este notebook. Su consulta debe diseñarse con ventana temporal y revisión de índices/plan antes de activarla.


In [58]:
# Todas las tablas y columnas del esquema consultado.
CAMPOS = {
    tabla: grupo.sort_values("ordinal_position")["columna"].tolist()
    for tabla, grupo in ESQUEMA.groupby("tabla", sort=True)
}

params = {
    "ruta": CFG["id_ruta"],
    "inicio": CFG["inicio"],
    "fin": CFG["fin_exclusivo"],
    "proceso": CFG["proceso"],
}

viajes_where = """
    "idruta" = :ruta
    AND "fechasalida" >= CAST(:inicio AS timestamp)
    AND "fechasalida" < CAST(:fin AS timestamp)
"""

viajes_ruta_periodo = f"""
    SELECT "idprogramacion"
    FROM {qtable("progvehiculos")}
    WHERE {viajes_where}
"""

# Relaciones comprobadas mediante los campos del esquema.
filtros_especiales = {
    "progvehiculos": viajes_where,

    "vehiculos": f"""
        "idvehiculo" IN (
            SELECT "idvehiculo"
            FROM {qtable("progvehiculos")}
            WHERE {viajes_where}
        )
    """,

    "timbradas_segmentos": f"""
        "id_segmento" IN (
            SELECT "id"
            FROM {qtable("rutas_segmentos")}
            WHERE "idruta" = :ruta
        )
        AND "fecha" >= CAST(:inicio AS date)
        AND "fecha" < CAST(:fin AS date)
    """,

    "conteo_pasajeros_destino": f"""
        "idprogramacion" IN ({viajes_ruta_periodo})
        AND "fecha" >= CAST(:inicio AS timestamp)
        AND "fecha" < CAST(:fin AS timestamp)
    """,

    "timbradas_anio_trazabilidad": f"""
        "id_timbrada_anio" IN (
            SELECT "id_timbrada_anio"
            FROM {qtable("timbradas_anio")}
            WHERE "id_ruta" = :ruta
        )
    """,

    "timbradas_anio_zonas": f"""
        "id_timbrada_anio" IN (
            SELECT "id_timbrada_anio"
            FROM {qtable("timbradas_anio")}
            WHERE "id_ruta" = :ruta
        )
    """,

    "timbradas_anio_zonas_trazabilidad": f"""
        "id_timbrada_anio_zona" IN (
            SELECT z."id_timbrada_anio_zona"
            FROM {qtable("timbradas_anio_zonas")} AS z
            JOIN {qtable("timbradas_anio")} AS a
              ON a."id_timbrada_anio" = z."id_timbrada_anio"
            WHERE a."id_ruta" = :ruta
        )
    """,

    "timbradas_consolidado_zonas": f"""
        "id_timbrada_consolidado" IN (
            SELECT "id_timbrada_consolidado"
            FROM {qtable("timbradas_consolidado")}
            WHERE "id_ruta" = :ruta
        )
    """,

    "novedades_viales_fecha": f"""
        "id_novedad_vial_fecha" IN (
            SELECT "id_novedad_vial_fecha"
            FROM {qtable("novedades_viales_ruta")}
            WHERE "id_ruta" = :ruta
        )
        AND "fecha" >= CAST(:inicio AS date)
        AND "fecha" < CAST(:fin AS date)
    """,
}

plan = []

for tabla in CAMPOS:
    columnas = COLS[tabla]

    if tabla in filtros_especiales:
        filtro = filtros_especiales[tabla]
        alcance = "Filtro específico; consultar SQL"

    elif "idruta" in columnas:
        filtro = '"idruta" = :ruta'
        alcance = "Ruta SILENCIO; sin filtro temporal adicional"

    elif "id_ruta" in columnas:
        filtro = '"id_ruta" = :ruta'
        alcance = "Ruta SILENCIO; sin filtro temporal adicional"

    elif "proceso" in columnas:
        filtro = '"proceso" = :proceso'
        alcance = "Proceso SOBUSA; puede incluir otras rutas"

    else:
        filtro = "TRUE"
        alcance = "Tabla completa; sin atribución exclusiva a SILENCIO"

    plan.append({
        "tabla": tabla,
        "columnas": len(CAMPOS[tabla]),
        "alcance": alcance,
        "filtro_sql": filtro,
    })

PLAN_EXTRACCION = pd.DataFrame(plan)

PLAN_EXTRACCION.to_csv(
    BASE / "reportes" / "plan_extraccion.csv",
    index=False,
)

display(PLAN_EXTRACCION[["tabla", "columnas", "alcance"]])

# Tablas documentadas cuya descarga se difiere intencionalmente.
TABLAS_DIFERIDAS = {"conteo_pasajeros_destino"}

for fila in plan:
    tabla = fila["tabla"]

    if tabla in TABLAS_DIFERIDAS:
        print(f"{tabla}: descarga diferida; consulta bajo demanda.")
        continue

    # Evita repetir tablas completadas en esta sesión.
    if tabla in RAW:
        print(f"{tabla}: ya extraída en esta sesión.")
        continue

    print(f"Extrayendo: {tabla}", flush=True)

    extraer(
        tabla,
        where=fila["filtro_sql"],
        p=params,
    )

# Las tablas diferidas no se consideran fallos de extracción.
pendientes = sorted(
    set(CAMPOS) - set(RAW) - TABLAS_DIFERIDAS
)

if pendientes:
    raise RuntimeError(
        "Extracción incompleta. Revisar REGISTRO para: "
        + ", ".join(pendientes)
    )

print(
    "Extracción prevista completada. "
    "Sensores disponibles para consulta bajo demanda."
)


,tabla,columnas,alcance
0,Procesos,88,Proceso SOBUSA; puede incluir otras rutas
1,conteo_pasajeros_destino,25,Filtro específico; consultar SQL
2,dias_semana,2,Tabla completa; sin atribución exclusiva a SIL...
3,estacionalidades,5,Proceso SOBUSA; puede incluir otras rutas
4,estacionalidades_anio,6,Proceso SOBUSA; puede incluir otras rutas
5,geography_columns,7,Tabla completa; sin atribución exclusiva a SIL...
6,geometry_columns,7,Tabla completa; sin atribución exclusiva a SIL...
7,novedades_viales,5,Proceso SOBUSA; puede incluir otras rutas
8,novedades_viales_fecha,8,Filtro específico; consultar SQL
9,novedades_viales_ruta,3,Ruta SILENCIO; sin filtro temporal adicional


Procesos: ya extraída en esta sesión.
conteo_pasajeros_destino: descarga diferida; consulta bajo demanda.
Extrayendo: dias_semana
dias_semana: 7 filas · 1 partes
Extrayendo: estacionalidades
estacionalidades: 24 filas · 1 partes
Extrayendo: estacionalidades_anio
estacionalidades_anio: 0 filas · 0 partes
Extrayendo: geography_columns
geography_columns: 0 filas · 0 partes
Extrayendo: geometry_columns
geometry_columns: 4 filas · 1 partes
Extrayendo: novedades_viales
novedades_viales: 59 filas · 1 partes
Extrayendo: novedades_viales_fecha
novedades_viales_fecha: 0 filas · 0 partes
Extrayendo: novedades_viales_ruta
novedades_viales_ruta: 0 filas · 0 partes
Extrayendo: progvehiculos
progvehiculos: 162,678 filas · 17 partes
Extrayendo: raster_columns
raster_columns: 0 filas · 0 partes
Extrayendo: raster_overviews
raster_overviews: 0 filas · 0 partes
Extrayendo: rutas
rutas: 1 filas · 1 partes
Extrayendo: rutas_segmentos
rutas_segmentos: 135 filas · 1 partes
Extrayendo: rutas_segmentos_config


### **4.7. Cobertura y verificación de archivos**

In [59]:
COBERTURA = consulta_pequena("cobertura_ruta", f"""SELECT
    COUNT(*) AS programaciones_ruta,
    COUNT(*) FILTER (WHERE fechasalida IS NULL) AS salida_nula,
    MIN(fechasalida) AS primera_salida, MAX(fechasalida) AS ultima_salida
    FROM {qtable('progvehiculos')} WHERE idruta=:ruta""", {"ruta": CFG["id_ruta"]})
COBERTURA.to_csv(BASE / "reportes" / "cobertura_ruta.csv", index=False)
display(COBERTURA)
RESUMEN = pd.DataFrame(REGISTRO)
RESUMEN.to_csv(BASE / "reportes" / "registro.csv", index=False)
# Validar conteos leyendo únicamente metadatos Parquet, sin cargar los datos.
for tabla, carpeta in RAW.items():
    filas_archivos = sum(pq.ParquetFile(p).metadata.num_rows for p in carpeta.glob("parte-*.parquet"))
    esperado = next(r["filas"] for r in reversed(REGISTRO)
                    if r["fuente"] == tabla and r["estado"] in ("extraida", "vacia_en_filtro"))
    if filas_archivos != esperado:
        raise RuntimeError(f"Conteo de archivos incompatible: {tabla}")
escribir_json(BASE / "fuentes.json", {t: str(p.relative_to(BASE)) for t, p in RAW.items()})
display(RESUMEN[["fuente", "estado", "filas", "segundos"]])

print("Conteos de archivos verificados contra el registro de extracción. ")


,programaciones_ruta,salida_nula,primera_salida,ultima_salida
0,162678,0,2023-01-01 05:00:00,2025-12-31 19:00:00


,fuente,estado,filas,segundos
0,esquema,metadatos,587,0.406492
1,inventario,metadatos,25,0.121363
2,indices,metadatos,63,0.099867
3,llaves,metadatos,0,0.108505
4,Procesos,extraida,1,1.167034
5,dias_semana,extraida,7,2.628876
6,estacionalidades,extraida,24,1.460169
7,estacionalidades_anio,vacia_en_filtro,0,1.244914
8,geography_columns,vacia_en_filtro,0,1.339910
9,geometry_columns,extraida,4,1.440552


Conteos de archivos verificados contra el registro de extracción. 


In [60]:
def iterar_tabla(tabla, columnas=None):
    if tabla not in RAW:
        raise KeyError(f"Tabla no extraída: {tabla}")
    for archivo in sorted(RAW[tabla].glob("parte-*.parquet")):
        yield pd.read_parquet(archivo, columns=columnas, dtype_backend="pyarrow")
def muestra(tabla, n=5, columnas=None):
    # Lee solo las primeras n filas de la primera parte.
    if n < 1 or n > 100:
        raise ValueError("Elegir entre 1 y 100 filas para la muestra")
    archivo = next(iter(sorted(RAW[tabla].glob("parte-*.parquet"))), None)
    if archivo is None:
        return pd.DataFrame(columns=columnas)
    lote = next(pq.ParquetFile(archivo).iter_batches(batch_size=n, columns=columnas))
    return lote.to_pandas(types_mapper=pd.ArrowDtype)
display(muestra("progvehiculos", columnas=["idprogramacion", "idruta", "fechasalida", "numeropasajeros"]))
# Al terminar las consultas. Los archivos siguen disponibles y los lectores funcionan.
engine.dispose()


,idprogramacion,idruta,fechasalida,numeropasajeros
0,5061797,2,2023-01-01 05:36:00,63
1,5061896,2,2023-01-01 08:45:00,74
2,5061990,2,2023-01-01 12:30:00,106
3,5062047,2,2023-01-01 17:30:00,130
4,5061793,2,2023-01-01 05:00:00,<NA>


### **4.8 Fichas de tablas y diccionario de variables**

In [69]:
# Consulta catálogos SQL y cuenta filas para la dimensión general; no descarga tablas completas.
if 'consulta_pequena' not in globals():
    raise RuntimeError('Ejecuta primero las celdas de conexión y consulta_pequena.')

OBJETOS_BD = consulta_pequena('objetos_fichas', '''
SELECT c.relname AS tabla,
       c.reltuples AS filas_estimadas_catalogo,
       CASE c.relkind WHEN 'r' THEN 'Tabla' WHEN 'p' THEN 'Tabla particionada'
            WHEN 'v' THEN 'Vista' WHEN 'm' THEN 'Vista materializada'
            WHEN 'f' THEN 'Tabla externa' END AS tipo_objeto
FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace
WHERE n.nspname=:s AND c.relkind IN ('r','p','v','m','f')
ORDER BY c.relname
''', {'s':CFG['schema']})

LLAVES = consulta_pequena('llaves_fichas', '''
SELECT c.relname AS tabla, con.conname AS constraint_name,
       CASE con.contype WHEN 'p' THEN 'PRIMARY KEY' WHEN 'f' THEN 'FOREIGN KEY'
            WHEN 'u' THEN 'UNIQUE' WHEN 'c' THEN 'CHECK'
            WHEN 'x' THEN 'EXCLUDE' END AS constraint_type,
       a.attname AS column_name, k.orden AS ordinal_position,
       pg_get_constraintdef(con.oid,true) AS definicion
FROM pg_constraint con
JOIN pg_class c ON c.oid=con.conrelid
JOIN pg_namespace n ON n.oid=c.relnamespace
LEFT JOIN LATERAL unnest(con.conkey) WITH ORDINALITY AS k(attnum,orden) ON true
LEFT JOIN pg_attribute a ON a.attrelid=c.oid AND a.attnum=k.attnum
WHERE n.nspname=:s AND con.contype IN ('p','f','u','c','x')
ORDER BY c.relname,con.conname,k.orden
''', {'s':CFG['schema']})


import pandas as pd
DOCUMENTADO = pd.DataFrame([{'tabla': 'Procesos', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(20)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Código del proceso (sobusa). PK lógica.'}, {'tabla': 'Procesos', 'variable_original': 'nombre', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nombre del proceso (SOBUSA).'}, {'tabla': 'Procesos', 'variable_original': 'tipostrabajos', 'tipo_documentado': 'character varying(100)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tipos de trabajo habilitados (S/N o lista).'}, {'tabla': 'Procesos', 'variable_original': 'tiposrecursos', 'tipo_documentado': 'character varying(100)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tipos de recursos usados por el proceso.'}, {'tabla': 'Procesos', 'variable_original': 'municipios', 'tipo_documentado': 'character varying(100)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Municipios en los que opera el proceso.'}, {'tabla': 'Procesos', 'variable_original': 'contratistas', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Uso de contratistas en el proceso.'}, {'tabla': 'Procesos', 'variable_original': 'manejaturnos', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el proceso maneja turnos.'}, {'tabla': 'Procesos', 'variable_original': 'usacontratistas', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si usa contratistas (S/N).'}, {'tabla': 'Procesos', 'variable_original': 'solocentral', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si la operación es solo desde central.'}, {'tabla': 'Procesos', 'variable_original': 'manejalbs', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja LBS (localización base satelital).'}, {'tabla': 'Procesos', 'variable_original': 'programaciondefault', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de programación por defecto.'}, {'tabla': 'Procesos', 'variable_original': 'manejaflotas', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja flotas.'}, {'tabla': 'Procesos', 'variable_original': 'penaliza', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el proceso aplica penalizaciones.'}, {'tabla': 'Procesos', 'variable_original': 'multaatraso', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Valor de multa por atraso.'}, {'tabla': 'Procesos', 'variable_original': 'multaadelanto', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Valor de multa por adelanto.'}, {'tabla': 'Procesos', 'variable_original': 'multa', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Valor base de multa.'}, {'tabla': 'Procesos', 'variable_original': 'toleranciatiempo', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tolerancia de tiempo (minutos).'}, {'tabla': 'Procesos', 'variable_original': 'idcliente', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Cliente o empresa principal del proceso.'}, {'tabla': 'Procesos', 'variable_original': 'estado', 'tipo_documentado': 'character varying(2)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Estado del proceso (activo/inactivo).'}, {'tabla': 'Procesos', 'variable_original': 'manejatorniquete', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja torniquete de conteo.'}, {'tabla': 'Procesos', 'variable_original': 'torniqueteobligatorio', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el torniquete es obligatorio.'}, {'tabla': 'Procesos', 'variable_original': 'manejaibutton', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja i-button (identificación de conductor).'}, {'tabla': 'Procesos', 'variable_original': 'transporteespecial', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si es transporte especial.'}, {'tabla': 'Procesos', 'variable_original': 'manejainterno', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja servicio interno.'}, {'tabla': 'Procesos', 'variable_original': 'programacionautomatica', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si la programación es automática.'}, {'tabla': 'Procesos', 'variable_original': 'manejapesv', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja PESV (Plan Estratégico de Seguridad Vial).'}, {'tabla': 'Procesos', 'variable_original': 'manejasensorpasajeros', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja sensor de pasajeros.'}, {'tabla': 'Procesos', 'variable_original': 'exportaviajes', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si exporta viajes a otros sistemas.'}, {'tabla': 'Procesos', 'variable_original': 'estadisticas_powerbi', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si alimenta estadísticas a Power BI.'}, {'tabla': 'Procesos', 'variable_original': 'manejapowerbi', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja integración con Power BI.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_despacho_cond_rec', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bloquea despacho si el conductor no tiene recibos.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_despacho_cond_lic', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bloquea despacho si el conductor no tiene licencia.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_despacho_veh_rec', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bloquea despacho si el vehículo no tiene recibo.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_despacho_veh_soat', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bloquea despacho si el vehículo no tiene SOAT.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_bloq_desp', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el bloqueo de despacho está activo.'}, {'tabla': 'Procesos', 'variable_original': 'historial_conductor', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si se administra historial del conductor.'}, {'tabla': 'Procesos', 'variable_original': 'descarga_rutas_track', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si descarga rutas desde módulo Track.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_despacho_vehiculo_gps', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloquea despacho si el vehículo no tiene GPS.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_despacho_vehiculo_sensor_pasajeros', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloquea despacho si el vehículo no tiene sensor de pasajeros.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_capac_veh', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja capacidad de vehículo.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_buzer', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja dispositivo buzzer.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_relay', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja relay (relevador).'}, {'tabla': 'Procesos', 'variable_original': 'maneja_logistica', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja módulo de logística.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_logistica_mensajeria', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja logística de mensajería.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_logistica_paqueteria', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja logística de paquetería.'}, {'tabla': 'Procesos', 'variable_original': 'envia_interfaz_despacho', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si envía despachos por interfaz.'}, {'tabla': 'Procesos', 'variable_original': 'url_interfaz_despacho', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'URL de la interfaz de despacho.'}, {'tabla': 'Procesos', 'variable_original': 'usuario_interfaz_despacho', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Usuario de la interfaz de despacho.'}, {'tabla': 'Procesos', 'variable_original': 'clave_interfaz_despacho', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Clave de la interfaz de despacho.'}, {'tabla': 'Procesos', 'variable_original': 'tiene_credenciales_interfaz_despacho', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si tiene credenciales configuradas para la interfaz.'}, {'tabla': 'Procesos', 'variable_original': 'isactive_control_t', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si está activo control en tiempo real.'}, {'tabla': 'Procesos', 'variable_original': 'integraciones_especiales_logisticas', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si tiene integraciones logísticas especiales.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_proveedor_propio', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja proveedor propio.'}, {'tabla': 'Procesos', 'variable_original': 'id_proveedor_sms', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Proveedor de SMS configurado.'}, {'tabla': 'Procesos', 'variable_original': 'tolerancia_recaudo', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tolerancia en el recaudo.'}, {'tabla': 'Procesos', 'variable_original': 'tolerancia_inicial_recaudo', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tolerancia inicial antes del recaudo.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_modulo_paraderos', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja módulo de paraderos.'}, {'tabla': 'Procesos', 'variable_original': 'top_sensor', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tope de sensor (capacidad de conteo).'}, {'tabla': 'Procesos', 'variable_original': 'recaudo_comision_conductor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Comisión al conductor por recaudo.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_recaudo_electronico', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja recaudo electrónico.'}, {'tabla': 'Procesos', 'variable_original': 'transporte_individual', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si es transporte individual.'}, {'tabla': 'Procesos', 'variable_original': 'fecha_inicio_liquidacion', 'tipo_documentado': 'date', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de inicio del período de liquidación.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_performance_conductor', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja performance del conductor.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_calificacion_conductor', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja calificación del conductor.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_estadistica_consolidado', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja estadística de consolidado.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_estadistica_eventos', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja estadística de eventos.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_estadistica_detallado', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja estadística detallada.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_envio_correo', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja envío de correos.'}, {'tabla': 'Procesos', 'variable_original': 'manejatracking', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja tracking/GPS.'}, {'tabla': 'Procesos', 'variable_original': 'manejamodemprepago', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja modo prepago.'}, {'tabla': 'Procesos', 'variable_original': 'estadistica_perfil_profesional', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja estadística de perfil profesional.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_recuperar_punto', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si permite recuperar punto.'}, {'tabla': 'Procesos', 'variable_original': 'revision_automatica_viajes', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si realiza revisión automática de viajes.'}, {'tabla': 'Procesos', 'variable_original': 'manejan_coordenadas', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja coordenadas.'}, {'tabla': 'Procesos', 'variable_original': 'latitud', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Latitud de la sede del proceso.'}, {'tabla': 'Procesos', 'variable_original': 'longitud', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Longitud de la sede del proceso.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_estadisticas_historicas', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja estadísticas históricas.'}, {'tabla': 'Procesos', 'variable_original': 'validar_alarmas_historicas', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si valida alarmas históricas.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_hora_pico', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja módulo de hora pico.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_valle', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si hay bloqueo en horas valle.'}, {'tabla': 'Procesos', 'variable_original': 'bloqueo_pico', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si hay bloqueo en horas pico.'}, {'tabla': 'Procesos', 'variable_original': 'operador_relacional_bloqueo_pico', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Operador relacional aplicado al bloqueo en pico.'}, {'tabla': 'Procesos', 'variable_original': 'valor_operador_relacional_bloqueo_pico', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Valor umbral usado con el operador de bloqueo pico.'}, {'tabla': 'Procesos', 'variable_original': 'maneja_tiempo_minimo_viaje', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja tiempo mínimo de viaje.'}, {'tabla': 'Procesos', 'variable_original': 'valor_tiempo_minimo_viaje', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tiempo mínimo de viaje (minutos).'}, {'tabla': 'Procesos', 'variable_original': 'id_estacionalidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estacionalidad vigente del proceso.'}, {'tabla': 'Procesos', 'variable_original': 'ultima_estacionalidad', 'tipo_documentado': 'date', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de la última estacionalidad aplicada.'}, {'tabla': 'Procesos', 'variable_original': 'id_diasemana', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Día de la semana por defecto.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'id', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'PK del evento de conteo.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'fecha', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha/hora del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'idvehiculo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Vehículo del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_subida', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de subidas del torniquete.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_subida', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_bajada', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bajadas.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_bajada', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_bloqueo', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bloqueos.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_bloqueo', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueos del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'latitud', 'tipo_documentado': 'double precision', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Latitud del vehículo.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'longitud', 'tipo_documentado': 'double precision', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Longitud del vehículo.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'idprogramacion', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Programación/viaje (FK progvehiculos).'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'idpunto', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Punto/parada del conteo.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_bajada_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bajadas por puerta 1.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_bloqueo_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bloqueos por puerta 1.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_subida_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas por puerta 1 del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_bajada_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas por puerta 1 del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_bloqueo_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueos por puerta 1 del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_subida_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de subidas por puerta 2.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_bajada_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bajadas por puerta 2.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_bloqueo_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bloqueos por puerta 2.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_subida_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas por puerta 2 del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_bajada_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas por puerta 2 del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'cantidad_bloqueo_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueos por puerta 2 del evento.'}, {'tabla': 'conteo_pasajeros_destino', 'variable_original': 'acumulada_subida_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de subidas por puerta 1.'}, {'tabla': 'dias_semana', 'variable_original': 'id_dia_semana', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK 1=Lunes ... 7=Domingo.'}, {'tabla': 'dias_semana', 'variable_original': 'dia_semana', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nombre del día.'}, {'tabla': 'estacionalidades', 'variable_original': 'id_estacionalidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de estacionalidad.'}, {'tabla': 'estacionalidades', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'estacionalidades', 'variable_original': 'estacionalidad', 'tipo_documentado': 'character varying(255)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tipo: FESTIVO_NACIONAL, FIESTA_ATLANTICO, TEMPORADA_NACIONAL.'}, {'tabla': 'estacionalidades', 'variable_original': 'nombre', 'tipo_documentado': 'character varying(255)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nombre (Navidad, Carnaval, etc.).'}, {'tabla': 'estacionalidades', 'variable_original': 'descripcion', 'tipo_documentado': 'character varying(500)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Descripción de la estacionalidad.'}, {'tabla': 'estacionalidades_anio', 'variable_original': 'id_estacionalidad_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de vigencia anual.'}, {'tabla': 'estacionalidades_anio', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso.'}, {'tabla': 'estacionalidades_anio', 'variable_original': 'id_estacionalidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a estacionalidades.'}, {'tabla': 'estacionalidades_anio', 'variable_original': 'ano', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Año de vigencia.'}, {'tabla': 'estacionalidades_anio', 'variable_original': 'fecha_inicio', 'tipo_documentado': 'date', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio de vigencia.'}, {'tabla': 'estacionalidades_anio', 'variable_original': 'fecha_fin', 'tipo_documentado': 'date', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin de vigencia.'}, {'tabla': 'novedades_viales', 'variable_original': 'id_novedad_vial', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK catálogo de novedades.'}, {'tabla': 'novedades_viales', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'novedades_viales', 'variable_original': 'novedad', 'tipo_documentado': 'character varying(255)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Código/tipo de novedad.'}, {'tabla': 'novedades_viales', 'variable_original': 'nombre', 'tipo_documentado': 'character varying(255)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nombre de la novedad.'}, {'tabla': 'novedades_viales', 'variable_original': 'descripcion', 'tipo_documentado': 'text', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Descripción (TEXT).'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'id_novedad_vial_fecha', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de evento de novedad.'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'id_novedad_vial', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a novedades_viales.'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'fecha', 'tipo_documentado': 'date', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fecha del evento.'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'hora_inicio', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Hora inicio de afectación.'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'hora_fin', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Hora fin de afectación.'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'porcentaje_impacto', 'tipo_documentado': 'numeric(5)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Impacto esperado en demanda (%)'}, {'tabla': 'novedades_viales_fecha', 'variable_original': 'potencial_efecto', 'tipo_documentado': 'USER-DEFINED', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Efecto potencial (enum USER-DEFINED).'}, {'tabla': 'novedades_viales_ruta', 'variable_original': 'id_novedad_vial_ruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK del cruce novedad-ruta.'}, {'tabla': 'novedades_viales_ruta', 'variable_original': 'id_novedad_vial_fecha', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a novedades_viales_fecha.'}, {'tabla': 'novedades_viales_ruta', 'variable_original': 'id_ruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'progvehiculos', 'variable_original': 'idprogramacion', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de la programación/viaje (referenciado por id_viaje).'}, {'tabla': 'progvehiculos', 'variable_original': 'idvehiculo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Vehículo asignado (placa/interno).'}, {'tabla': 'progvehiculos', 'variable_original': 'idconductor', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Conductor asignado.'}, {'tabla': 'progvehiculos', 'variable_original': 'idruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Ruta del viaje (FK rutas.idruta).'}, {'tabla': 'progvehiculos', 'variable_original': 'fechasalida', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fecha/hora programada de salida.'}, {'tabla': 'progvehiculos', 'variable_original': 'fecharegistro', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fecha/hora de registro de la programación.'}, {'tabla': 'progvehiculos', 'variable_original': 'fechainicio', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Inicio efectivo del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'fechafin', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fin efectivo del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'finalizado', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Estado: viaje finalizado (S/N).'}, {'tabla': 'progvehiculos', 'variable_original': 'proceso', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'progvehiculos', 'variable_original': 'cancelada', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el viaje fue cancelado (S/N).'}, {'tabla': 'progvehiculos', 'variable_original': 'observacion', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Observación del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'puntoini', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Punto de inicio (parada).'}, {'tabla': 'progvehiculos', 'variable_original': 'ordenpunto', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Orden del punto inicial.'}, {'tabla': 'progvehiculos', 'variable_original': 'liquidado', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el viaje está liquidado (S/N).'}, {'tabla': 'progvehiculos', 'variable_original': 'fechaliq', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de liquidación.'}, {'tabla': 'progvehiculos', 'variable_original': 'grupo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Grupo operativo de la programación.'}, {'tabla': 'progvehiculos', 'variable_original': 'activa', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si la programación está activa (S/N).'}, {'tabla': 'progvehiculos', 'variable_original': 'geocercaact', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Geocerca activa.'}, {'tabla': 'progvehiculos', 'variable_original': 'gtransid', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Identificador de la transacción GPS.'}, {'tabla': 'progvehiculos', 'variable_original': 'usuario', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Usuario que programó el viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'excedeplazosalida', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si excede el plazo de salida.'}, {'tabla': 'progvehiculos', 'variable_original': 'iddespachador', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Despachador responsable.'}, {'tabla': 'progvehiculos', 'variable_original': 'puntofin', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Punto final (parada).'}, {'tabla': 'progvehiculos', 'variable_original': 'ordenpuntofin', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Orden del punto final.'}, {'tabla': 'progvehiculos', 'variable_original': 'distancia_recorrida', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Distancia recorrida en el viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'torniqueteinicial', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Conteo del torniquete al inicio.'}, {'tabla': 'progvehiculos', 'variable_original': 'torniquetefinal', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Conteo del torniquete al final.'}, {'tabla': 'progvehiculos', 'variable_original': 'numeropasajeros', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Total de pasajeros transportados (subidas).'}, {'tabla': 'progvehiculos', 'variable_original': 'excepcion', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de excepción del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'tipo', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de programación/viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'idempresa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Empresa del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'data_log', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Log de datos del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'numerobajadas', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Total de bajadas del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'numerobloqueos', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Total de bloqueos de puerta del viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_subida', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de subidas (torniquete).'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_bajada', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bajadas.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_bloqueo', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bloqueos.'}, {'tabla': 'progvehiculos', 'variable_original': 'exportado', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si el viaje fue exportado (S/N).'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_subida_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de subidas por puerta 1.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_bajada_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bajadas por puerta 1.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_bloqueo_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bloqueos por puerta 1.'}, {'tabla': 'progvehiculos', 'variable_original': 'cantidad_subida_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas por puerta 1 del evento.'}, {'tabla': 'progvehiculos', 'variable_original': 'cantidad_bajada_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas por puerta 1 del evento.'}, {'tabla': 'progvehiculos', 'variable_original': 'cantidad_bloqueo_puerta1', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueos por puerta 1 del evento.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_subida_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de subidas por puerta 2.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_bajada_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bajadas por puerta 2.'}, {'tabla': 'progvehiculos', 'variable_original': 'acumulada_bloqueo_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Acumulado de bloqueos por puerta 2.'}, {'tabla': 'progvehiculos', 'variable_original': 'cantidad_subida_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas por puerta 2 del evento.'}, {'tabla': 'progvehiculos', 'variable_original': 'cantidad_bajada_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas por puerta 2 del evento.'}, {'tabla': 'progvehiculos', 'variable_original': 'cantidad_bloqueo_puerta2', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueos por puerta 2 del evento.'}, {'tabla': 'progvehiculos', 'variable_original': 'bloqueo_cond', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si hubo bloqueo por condición del conductor.'}, {'tabla': 'progvehiculos', 'variable_original': 'bloqueo_veh', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si hubo bloqueo por condición del vehículo.'}, {'tabla': 'progvehiculos', 'variable_original': 'bloqueo_despacho_cond_rec', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueado por recibo del conductor.'}, {'tabla': 'progvehiculos', 'variable_original': 'bloqueo_despacho_cond_lic', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueado por licencia del conductor.'}, {'tabla': 'progvehiculos', 'variable_original': 'bloqueo_despacho_veh_rec', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueado por recibo del vehículo.'}, {'tabla': 'progvehiculos', 'variable_original': 'bloqueo_despacho_veh_soat', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bloqueado por SOAT del vehículo.'}, {'tabla': 'progvehiculos', 'variable_original': 'observacion_bloq', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Observación del bloqueo.'}, {'tabla': 'progvehiculos', 'variable_original': 'recaudo_pasajeros', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Recaudo declarado por pasajeros.'}, {'tabla': 'progvehiculos', 'variable_original': 'viaje_culminado', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si el viaje culminó (BOOLEAN).'}, {'tabla': 'progvehiculos', 'variable_original': 'recaudado', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si el viaje fue recaudado (BOOLEAN).'}, {'tabla': 'progvehiculos', 'variable_original': 'estado_amb', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estado ambiental.'}, {'tabla': 'progvehiculos', 'variable_original': 'idtarifa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tarifa aplicada al viaje.'}, {'tabla': 'progvehiculos', 'variable_original': 'valor_tarifa', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Valor de la tarifa.'}, {'tabla': 'progvehiculos', 'variable_original': 'modificado_manual', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si el viaje fue modificado manualmente.'}, {'tabla': 'progvehiculos', 'variable_original': 'timbradas_manuales', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Cantidad de timbradas manuales.'}, {'tabla': 'progvehiculos', 'variable_original': 'modificacion_observacion', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Observación de la modificación.'}, {'tabla': 'progvehiculos', 'variable_original': 'recaudo_manual', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Recaudo manual ingresado.'}, {'tabla': 'progvehiculos', 'variable_original': 'fecha_final_manual', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de fin manual.'}, {'tabla': 'progvehiculos', 'variable_original': 'id_pendiente', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Identificador de pendiente.'}, {'tabla': 'progvehiculos', 'variable_original': 'fecha_despacho_almacenado_movil', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha del despacho almacenado en móvil.'}, {'tabla': 'progvehiculos', 'variable_original': 'apto_estadisticas_historicas', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el viaje es apto para estadísticas históricas.'}, {'tabla': 'progvehiculos', 'variable_original': 'fecha_fin_liquidacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de fin del período de liquidación.'}, {'tabla': 'rutas', 'variable_original': 'idruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de la ruta (2=SILENCIO, 16=CRA 54, 21=CL 72).'}, {'tabla': 'rutas', 'variable_original': 'nombre', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nombre de la ruta.'}, {'tabla': 'rutas', 'variable_original': 'codigo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Código operativo de la ruta.'}, {'tabla': 'rutas', 'variable_original': 'proceso', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'rutas', 'variable_original': 'geom', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Geometría de la ruta (cadena de coordenadas).'}, {'tabla': 'rutas', 'variable_original': 'tolerancia', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tolerancia de la ruta.'}, {'tabla': 'rutas', 'variable_original': 'penaliza', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si la ruta penaliza incumplimientos.'}, {'tabla': 'rutas', 'variable_original': 'manejahp', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si maneja hora pico en la ruta.'}, {'tabla': 'rutas', 'variable_original': 'tolsalida', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tolerancia de salida (min).'}, {'tabla': 'rutas', 'variable_original': 'tiempoplazosalida', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Plazo máximo de salida (min).'}, {'tabla': 'rutas', 'variable_original': 'distancia_global', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Distancia total de la ruta.'}, {'tabla': 'rutas', 'variable_original': 'tiempo_global', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tiempo total de ciclo de la ruta.'}, {'tabla': 'rutas', 'variable_original': 'tipo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tipo de ruta (3 = urbana).'}, {'tabla': 'rutas', 'variable_original': 'estado_ruta', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': "Estado ('A' = activa)."}, {'tabla': 'rutas', 'variable_original': 'idrutageo', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Identificador de la ruta geográfica.'}, {'tabla': 'rutas', 'variable_original': 'idrutaequivalente', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ruta equivalente (intercambio).'}, {'tabla': 'rutas', 'variable_original': 'idempresa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Empresa propietaria.'}, {'tabla': 'rutas', 'variable_original': 'metodo_prog', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Método de programación.'}, {'tabla': 'rutas', 'variable_original': 'grupos_prog', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Grupos de programación.'}, {'tabla': 'rutas', 'variable_original': 'cunas_prog', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Cunas horarias de programación.'}, {'tabla': 'rutas', 'variable_original': 'flotadisp_prog', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Flota disponible para programación.'}, {'tabla': 'rutas', 'variable_original': 'flotadisp_prog_metodo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Método de cálculo de flota disponible.'}, {'tabla': 'rutas', 'variable_original': 'veh_respaldo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Vehículos de respaldo.'}, {'tabla': 'rutas', 'variable_original': 'primerdespacho', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Hora del primer despacho.'}, {'tabla': 'rutas', 'variable_original': 'frecuenciadespachos', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Frecuencia vigente de despachos (min).'}, {'tabla': 'rutas', 'variable_original': 'tipo_asig_tiempo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tipo de asignación de tiempo.'}, {'tabla': 'rutas', 'variable_original': 'manejasensorpasajeros', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si la ruta maneja sensor de pasajeros.'}, {'tabla': 'rutas', 'variable_original': 'ruta_especial', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si es una ruta especial.'}, {'tabla': 'rutas', 'variable_original': 'maneja_sorteo_dom_festivos', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja sorteo en domingos/festivos.'}, {'tabla': 'rutas', 'variable_original': 'reagrupar', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si permite reagrupar vehículos.'}, {'tabla': 'rutas', 'variable_original': 'cede_vehiculos', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si cede vehículos a otras rutas.'}, {'tabla': 'rutas', 'variable_original': 'grupos_vehiculos_cedidos', 'tipo_documentado': 'smallint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Grupos de vehículos cedidos.'}, {'tabla': 'rutas', 'variable_original': 'hexcolor', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Color hexadecimal de la ruta en mapas.'}, {'tabla': 'rutas', 'variable_original': 'maneja_zonas', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si la ruta maneja zonas tarifarias.'}, {'tabla': 'rutas', 'variable_original': 'id_tipo_tarifa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de tarifa asociado.'}, {'tabla': 'rutas', 'variable_original': 'maneja_tarifa', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si la ruta maneja tarifa variable.'}, {'tabla': 'rutas', 'variable_original': 'colorrutapara', 'tipo_documentado': 'character varying(255)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Color de la ruta para mapas.'}, {'tabla': 'rutas', 'variable_original': 'codigo_amb', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Código ambiente alterno.'}, {'tabla': 'rutas', 'variable_original': 'tolerancia_inicial_recaudo', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tolerancia inicial de recaudo.'}, {'tabla': 'rutas', 'variable_original': 'tolerancia_recaudo', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tolerancia de recaudo.'}, {'tabla': 'rutas', 'variable_original': 'maneja_historio_puntos', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja historial de puntos.'}, {'tabla': 'rutas', 'variable_original': 'idruta_sibus', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'ID de la ruta en sistema SIBUS.'}, {'tabla': 'rutas', 'variable_original': 'idruta_itinerario', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'ID de la ruta en sistema de itinerarios.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'id', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK del segmento.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'id_config', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas_segmentos_config.id.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'idruta', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'idrutageo', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Ruta geográfica.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'nro_segmento', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Número de orden del segmento.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'activo', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Segmento activo.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'porciones_ids', 'tipo_documentado': 'text', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Lista de porciones que lo componen (TEXT).'}, {'tabla': 'rutas_segmentos', 'variable_original': 'longitud_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Longitud del segmento en metros.'}, {'tabla': 'rutas_segmentos', 'variable_original': 'geom_segmento', 'tipo_documentado': 'USER-DEFINED', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Geometría (PostGIS).'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'id', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de configuración.'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'idruta', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'idrutageo', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Ruta geográfica.'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'tipo_agrupacion', 'tipo_documentado': 'character varying(20)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tipo de agrupación (distancia/cantidad).'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'tamaño_segmento_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tamaño objetivo del segmento (m).'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'numero_segmentos', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Nº de segmentos resultantes.'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'fecha_creacion', 'tipo_documentado': 'timestamp with time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fecha de creación de la config.'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'tamaño_porcion_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tamaño de porción (m).'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'tamaño_minimo_porcion_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Tamaño mínimo de porción (m).'}, {'tabla': 'rutas_segmentos_config', 'variable_original': 'activo', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Configuración activa.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'id', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de la porción.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'idruta', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'idrutageo', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Ruta geográfica.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'nro_porcion', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Número de orden de la porción.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'inicio_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio en metros sobre la ruta.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'fin_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin en metros sobre la ruta.'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'longitud_porcion_m', 'tipo_documentado': 'numeric(12)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Longitud de la porción (m).'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'punto_inicio', 'tipo_documentado': 'USER-DEFINED', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Punto de inicio (PostGIS).'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'punto_fin', 'tipo_documentado': 'USER-DEFINED', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Punto de fin (PostGIS).'}, {'tabla': 'rutas_segmentos_porcion', 'variable_original': 'geom_porcion', 'tipo_documentado': 'USER-DEFINED', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Geometría de la porción (PostGIS).'}, {'tabla': 'spatial_ref_sys', 'variable_original': 'srid', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Identificador espacial de referencia (EPSG).'}, {'tabla': 'spatial_ref_sys', 'variable_original': 'auth_name', 'tipo_documentado': 'character varying(256)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Autoridad del SRID.'}, {'tabla': 'spatial_ref_sys', 'variable_original': 'auth_srid', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'SRID según la autoridad.'}, {'tabla': 'spatial_ref_sys', 'variable_original': 'srtext', 'tipo_documentado': 'character varying(2048)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Texto Well-Known Text del sistema.'}, {'tabla': 'spatial_ref_sys', 'variable_original': 'proj4text', 'tipo_documentado': 'character varying(2048)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Cadena de proyección Proj4.'}, {'tabla': 'timbradas_anio', 'variable_original': 'id_timbrada_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK del consolidado anual.'}, {'tabla': 'timbradas_anio', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso operacional (sobusa).'}, {'tabla': 'timbradas_anio', 'variable_original': 'ano', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Año del consolidado (2023-2026).'}, {'tabla': 'timbradas_anio', 'variable_original': 'id_ruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'timbradas_anio', 'variable_original': 'id_dia_semana', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a dias_semana.'}, {'tabla': 'timbradas_anio', 'variable_original': 'id_estacionalidad_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a estacionalidades_anio (ojo: 92 valores, sin filas de sobusa).'}, {'tabla': 'timbradas_anio', 'variable_original': 'hora', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Hora inicio de franja (redondeada).'}, {'tabla': 'timbradas_anio', 'variable_original': 'hora_inicio', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio de la franja de 15 min.'}, {'tabla': 'timbradas_anio', 'variable_original': 'hora_fin', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin de la franja de 15 min.'}, {'tabla': 'timbradas_anio', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de subidas en la franja.'}, {'tabla': 'timbradas_anio', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de bajadas en la franja.'}, {'tabla': 'timbradas_anio', 'variable_original': 'nro_viajes', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nº de viajes del promedio.'}, {'tabla': 'timbradas_anio', 'variable_original': 'promedio_subidas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio histórico de subidas por viaje.'}, {'tabla': 'timbradas_anio', 'variable_original': 'promedio_bajadas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio histórico de bajadas por viaje.'}, {'tabla': 'timbradas_anio', 'variable_original': 'ultima_actualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua para lectura incremental.'}, {'tabla': 'timbradas_anio_trazabilidad', 'variable_original': 'id_timbrada_anio_trazabilidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK trazabilidad.'}, {'tabla': 'timbradas_anio_trazabilidad', 'variable_original': 'id_timbrada_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'FK al agregado anual.'}, {'tabla': 'timbradas_anio_trazabilidad', 'variable_original': 'idprogramacion', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Programación que compone el agregado.'}, {'tabla': 'timbradas_anio_trazabilidad', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas de esa programación.'}, {'tabla': 'timbradas_anio_trazabilidad', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas de esa programación.'}, {'tabla': 'timbradas_anio_trazabilidad', 'variable_original': 'vigente', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si participa en el agregado vigente.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'id_timbrada_anio_zona', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de la desagregación por zona.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'id_timbrada_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a timbradas_anio.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'id_zona_tarifa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Zona tarifaria.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Subidas en la zona.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bajadas en la zona.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'nro_viajes', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Viajes en la zona.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'promedio_subidas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de subidas en la zona.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'promedio_bajadas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de bajadas en la zona.'}, {'tabla': 'timbradas_anio_zonas', 'variable_original': 'ultima_actualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua.'}, {'tabla': 'timbradas_anio_zonas_trazabilidad', 'variable_original': 'id_timbrada_anio_trazabilidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de trazabilidad por zona.'}, {'tabla': 'timbradas_anio_zonas_trazabilidad', 'variable_original': 'id_timbrada_anio_zona', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'FK a timbradas_anio_zonas.'}, {'tabla': 'timbradas_anio_zonas_trazabilidad', 'variable_original': 'idprogramacion', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Programación componente.'}, {'tabla': 'timbradas_anio_zonas_trazabilidad', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Subidas de la programación.'}, {'tabla': 'timbradas_anio_zonas_trazabilidad', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bajadas de la programación.'}, {'tabla': 'timbradas_anio_zonas_trazabilidad', 'variable_original': 'vigente', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Participa en el agregado vigente.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'id_timbrada_consolidado', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK del consolidado principal.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso operacional (sobusa).'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'id_ruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'id_dia_semana', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a dias_semana.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'id_estacionalidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a estacionalidades.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'hora', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Hora inicio de franja (redondeada).'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'hora_inicio', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio de la franja de 15 min.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'hora_fin', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin de la franja de 15 min.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de subidas en la franja.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de bajadas en la franja.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'nro_viajes', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nº de viajes del promedio.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'promedio_subidas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio histórico de subidas.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'promedio_bajadas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio histórico de bajadas.'}, {'tabla': 'timbradas_consolidado', 'variable_original': 'ultima_actualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'id_timbrada_consolidado_zona', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK desagregación por zona.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'id_timbrada_consolidado', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK al consolidado.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'id_zona_tarifa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Zona tarifaria.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Subidas en la zona.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bajadas en la zona.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'nro_viajes', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Viajes en la zona.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'promedio_subidas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio subidas en la zona.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'promedio_bajadas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio bajadas en la zona.'}, {'tabla': 'timbradas_consolidado_zonas', 'variable_original': 'ultima_actualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'id_timbrada_segmento', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK de detalle por segmento y viaje.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'id_segmento', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas_segmentos.id.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'id_viaje', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a progvehiculos.idprogramacion.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'fecha', 'tipo_documentado': 'date', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fecha del viaje.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'hora_inicio', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio de la franja en el segmento.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'hora_fin', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin de la franja en el segmento.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Subidas en el segmento para ese viaje.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bajadas en el segmento para ese viaje.'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'pasajeros', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Carga/pasajeros a bordo (ojo: suma negativa → revisar semántica).'}, {'tabla': 'timbradas_segmentos', 'variable_original': 'ultimaactualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'id_timbrada_segmento_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK agregado anual por segmento.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'id_segmento', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas_segmentos.id.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'ano', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Año del agregado.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'id_ruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'id_dia_semana', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a dias_semana.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'id_estacionalidad_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a estacionalidades_anio.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'hora', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Hora de inicio de la franja.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'hora_inicio', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio de franja de 15 min.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'hora_fin', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin de franja de 15 min.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de subidas.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de bajadas.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'pasajeros', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Pasajeros (carga).'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'nro_viajes', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nº de viajes del agregado.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'promedio_subidas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de subidas.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'promedio_bajadas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de bajadas.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'promedio_pasajeros', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de pasajeros.'}, {'tabla': 'timbradas_segmentos_anio', 'variable_original': 'ultima_actualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'id_trazabilidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK trazabilidad.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'id_timbrada_segmento_anio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK al agregado de segmento.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'idprogramacion', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Programación componente del agregado.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Subidas de la programación.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Bajadas de la programación.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'pasajeros', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Pasajeros de la programación.'}, {'tabla': 'timbradas_segmentos_anio_trazabilidad', 'variable_original': 'vigente', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Participa en el agregado vigente.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'id_timbrada_segmentos_consolidado', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'PK consolidado por segmento.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'id_segmento', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas_segmentos.id.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'proceso', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Proceso (sobusa).'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'id_ruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a rutas.idruta.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'id_dia_semana', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a dias_semana.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'id_estacionalidad', 'tipo_documentado': 'bigint', 'nulo_documentado': 'NO', 'descripcion_documentada': 'FK a estacionalidades.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'hora', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Hora de inicio de la franja.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'hora_inicio', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Inicio de franja de 15 min.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'hora_fin', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Fin de franja de 15 min.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'subidas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de subidas.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'bajadas', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Suma de bajadas.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'pasajeros', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Pasajeros (carga).'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'nro_viajes', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Nº de viajes del consolidado.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'promedio_subidas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de subidas.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'promedio_bajadas', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de bajadas.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'promedio_pasajeros', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Promedio de pasajeros.'}, {'tabla': 'timbradas_segmentos_consolidado', 'variable_original': 'ultima_actualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca de agua.'}, {'tabla': 'vehiculos', 'variable_original': 'placa', 'tipo_documentado': 'character varying(15)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Placa del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'idvehiculo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Identificador interno del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'nromovil', 'tipo_documentado': 'character varying(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Número de móvil asignado.'}, {'tabla': 'vehiculos', 'variable_original': 'idconductor', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Conductor asignado al vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'modelo', 'tipo_documentado': 'character varying(4)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Modelo del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'marca', 'tipo_documentado': 'character varying(15)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'fechaingreso', 'tipo_documentado': 'date', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de ingreso a flota.'}, {'tabla': 'vehiculos', 'variable_original': 'estado', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estado del vehículo (activo/inactivo).'}, {'tabla': 'vehiculos', 'variable_original': 'xpos', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Coordenada X de posición.'}, {'tabla': 'vehiculos', 'variable_original': 'ypos', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Coordenada Y de posición.'}, {'tabla': 'vehiculos', 'variable_original': 'latitud', 'tipo_documentado': 'character varying(25)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Latitud de la última posición.'}, {'tabla': 'vehiculos', 'variable_original': 'longitud', 'tipo_documentado': 'character varying(25)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Longitud de la última posición.'}, {'tabla': 'vehiculos', 'variable_original': 'ultimaactualizacion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Última actualización de posición.'}, {'tabla': 'vehiculos', 'variable_original': 'velocidad', 'tipo_documentado': 'character varying(100)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Velocidad del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'direccion', 'tipo_documentado': 'character varying(100)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Dirección (rumbo) actual.'}, {'tabla': 'vehiculos', 'variable_original': 'alarma', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Alarma activa.'}, {'tabla': 'vehiculos', 'variable_original': 'idgps', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'ID del dispositivo GPS.'}, {'tabla': 'vehiculos', 'variable_original': 'idsim', 'tipo_documentado': 'numeric(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'ID de la SIM.'}, {'tabla': 'vehiculos', 'variable_original': 'zona', 'tipo_documentado': 'smallint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Zona operativa.'}, {'tabla': 'vehiculos', 'variable_original': 'idempresa', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Empresa propietaria.'}, {'tabla': 'vehiculos', 'variable_original': 'color', 'tipo_documentado': 'character varying(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Color del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'tipo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'idmodem', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'ID del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'idpropietario', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Propietario.'}, {'tabla': 'vehiculos', 'variable_original': 'municipio', 'tipo_documentado': 'character varying(30)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Municipio de operación.'}, {'tabla': 'vehiculos', 'variable_original': 'departamento', 'tipo_documentado': 'character varying(30)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Departamento de operación.'}, {'tabla': 'vehiculos', 'variable_original': 'ultimoevento', 'tipo_documentado': 'character varying(5)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Último evento reportado.'}, {'tabla': 'vehiculos', 'variable_original': 'tipo_modem', 'tipo_documentado': 'character varying(25)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_imei', 'tipo_documentado': 'character varying(20)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'IMEI del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_min', 'tipo_documentado': 'character varying(20)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'MIN del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_puk', 'tipo_documentado': 'character varying(20)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'PUK del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_dirip', 'tipo_documentado': 'character varying(15)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Dirección IP del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_estado', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estado del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_puerto', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Puerto del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'recurso', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Recurso asociado.'}, {'tabla': 'vehiculos', 'variable_original': 'contratista', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Contratista del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'idcontacto', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Contacto del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'asegurado', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si está asegurado.'}, {'tabla': 'vehiculos', 'variable_original': 'aseguradora', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Aseguradora.'}, {'tabla': 'vehiculos', 'variable_original': 'tel_emergencia', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Teléfono de emergencia.'}, {'tabla': 'vehiculos', 'variable_original': 'vigencia_poliza', 'tipo_documentado': 'character varying(11)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Vigencia de la póliza.'}, {'tabla': 'vehiculos', 'variable_original': 'poliza', 'tipo_documentado': 'character varying(30)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Número de póliza.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_poliza', 'tipo_documentado': 'character varying(11)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de la póliza.'}, {'tabla': 'vehiculos', 'variable_original': 'tipo_carroceria', 'tipo_documentado': 'character varying(30)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de carrocería.'}, {'tabla': 'vehiculos', 'variable_original': 'color_carroceria', 'tipo_documentado': 'character varying(30)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Color de carrocería.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_user', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Usuario del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_passwd', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Contraseña del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'programacion', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Nombre de programación.'}, {'tabla': 'vehiculos', 'variable_original': 'capacidad', 'tipo_documentado': 'double precision', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Capacidad nominal de pasajeros.'}, {'tabla': 'vehiculos', 'variable_original': 'ultperiodo', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Último período reportado.'}, {'tabla': 'vehiculos', 'variable_original': 'numinterno', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Número interno del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'idprogactual', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Programación actual.'}, {'tabla': 'vehiculos', 'variable_original': 'estadoruta', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estado en ruta.'}, {'tabla': 'vehiculos', 'variable_original': 'rumbo', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Rumbo.'}, {'tabla': 'vehiculos', 'variable_original': 'idruta', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ruta actual (FK rutas.idruta).'}, {'tabla': 'vehiculos', 'variable_original': 'enc_apa', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estado encendido/apagado.'}, {'tabla': 'vehiculos', 'variable_original': 'vel_prom_dia', 'tipo_documentado': 'double precision', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Velocidad promedio diaria.'}, {'tabla': 'vehiculos', 'variable_original': 'vel_max_dia', 'tipo_documentado': 'double precision', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Velocidad máxima diaria.'}, {'tabla': 'vehiculos', 'variable_original': 'indexgeoc', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Índice de geocerca.'}, {'tabla': 'vehiculos', 'variable_original': 'idcliente', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Cliente del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'tipocombustible', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de combustible.'}, {'tabla': 'vehiculos', 'variable_original': 'consumocombustible', 'tipo_documentado': 'double precision', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Consumo de combustible.'}, {'tabla': 'vehiculos', 'variable_original': 'indexevento', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Índice del último evento.'}, {'tabla': 'vehiculos', 'variable_original': 'ult_distancia_recorrida', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Última distancia recorrida.'}, {'tabla': 'vehiculos', 'variable_original': 'encendido', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si el vehículo está encendido.'}, {'tabla': 'vehiculos', 'variable_original': 'estadosenal', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Estado de la señal.'}, {'tabla': 'vehiculos', 'variable_original': 'serie', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Serial general.'}, {'tabla': 'vehiculos', 'variable_original': 'serie_chasis', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Serial del chasis.'}, {'tabla': 'vehiculos', 'variable_original': 'serie_motor', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Serial del motor.'}, {'tabla': 'vehiculos', 'variable_original': 'sim_serie', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Serial de la SIM.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_serie', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Serial del módem.'}, {'tabla': 'vehiculos', 'variable_original': 'operador_satelital', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Operador satelital.'}, {'tabla': 'vehiculos', 'variable_original': 'idmarca', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Marca (ID).'}, {'tabla': 'vehiculos', 'variable_original': 'idlinea', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Línea (ID).'}, {'tabla': 'vehiculos', 'variable_original': 'tipomaquinaria', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de maquinaria.'}, {'tabla': 'vehiculos', 'variable_original': 'tipovehiculo', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de vehículo (ID).'}, {'tabla': 'vehiculos', 'variable_original': 'manejaibutton', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si maneja i-button.'}, {'tabla': 'vehiculos', 'variable_original': 'idconductor_actual', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Conductor actual.'}, {'tabla': 'vehiculos', 'variable_original': 'capacidad_tanque', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Capacidad del tanque.'}, {'tabla': 'vehiculos', 'variable_original': 'idruta_actual', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ruta actual (ID).'}, {'tabla': 'vehiculos', 'variable_original': 'enviadomaquinaria', 'tipo_documentado': 'character varying(1)', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si envió maquinaria.'}, {'tabla': 'vehiculos', 'variable_original': 'runt', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'RUNT del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_instalacion', 'tipo_documentado': 'date', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de instalación de dispositivo.'}, {'tabla': 'vehiculos', 'variable_original': 'documento_instalacion', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Documento de instalación.'}, {'tabla': 'vehiculos', 'variable_original': 'rumbo_linea_tiempo', 'tipo_documentado': 'integer', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Rumbo en línea de tiempo.'}, {'tabla': 'vehiculos', 'variable_original': 'grupo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Grupo del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'rutaunica', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Ruta única.'}, {'tabla': 'vehiculos', 'variable_original': 'ultimo_odometro', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha del último odómetro.'}, {'tabla': 'vehiculos', 'variable_original': 'numero_vin', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Número VIN.'}, {'tabla': 'vehiculos', 'variable_original': 'tarjeta_operacion', 'tipo_documentado': 'numeric(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tarjeta de operación.'}, {'tabla': 'vehiculos', 'variable_original': 'despacho_habilitado', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si el despacho está habilitado.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_inspeccion', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de inspección.'}, {'tabla': 'vehiculos', 'variable_original': 'resultado_inspeccion', 'tipo_documentado': 'character varying(2)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Resultado de la inspección.'}, {'tabla': 'vehiculos', 'variable_original': 'idprogramacion_especial', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Programación especial.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_ultimo_conteo', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha del último conteo de pasajeros.'}, {'tabla': 'vehiculos', 'variable_original': 'sim_operador', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Operador de la SIM.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_solicitud', 'tipo_documentado': 'timestamp without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fecha de solicitud.'}, {'tabla': 'vehiculos', 'variable_original': 'estado_envio', 'tipo_documentado': 'character varying', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Estado de envío de configuración.'}, {'tabla': 'vehiculos', 'variable_original': 'cambio_frecuencia', 'tipo_documentado': 'boolean', 'nulo_documentado': 'NO', 'descripcion_documentada': 'Si hay cambio de frecuencia.'}, {'tabla': 'vehiculos', 'variable_original': 'frecuencia_rap_tiempo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Frecuencia rápida en tiempo.'}, {'tabla': 'vehiculos', 'variable_original': 'frecuencia_reg_tiempo', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Frecuencia regular en tiempo.'}, {'tabla': 'vehiculos', 'variable_original': 'frecuencia_rap_distancia', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Frecuencia rápida en distancia.'}, {'tabla': 'vehiculos', 'variable_original': 'frecuencia_reg_distancia', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Frecuencia regular en distancia.'}, {'tabla': 'vehiculos', 'variable_original': 'medio_envio_apagado', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Medio de envío de apagado.'}, {'tabla': 'vehiculos', 'variable_original': 'proceso_envio_apag', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Proceso de envío de apagado.'}, {'tabla': 'vehiculos', 'variable_original': 'usuario_envio_apag', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Usuario de envío de apagado.'}, {'tabla': 'vehiculos', 'variable_original': 'idinterfaz_envio', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Interfaz de envío.'}, {'tabla': 'vehiculos', 'variable_original': 'unidad_estrategica_negocio', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Unidad estratégica de negocio.'}, {'tabla': 'vehiculos', 'variable_original': 'peso_maximo_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Peso máximo del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'tara_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tara del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'largo_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Largo del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'ancho_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ancho del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'alto_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Alto del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'ancho_techo_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ancho del techo del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'alto_techo_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Alto del techo del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'ancho_apertura_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ancho de apertura del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'alto_apertura_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Alto de apertura del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'temperatura_minima_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Temperatura mínima del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'temperatura_maxima_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Temperatura máxima del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'voltaje_contenedor', 'tipo_documentado': 'numeric', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Voltaje del contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'id_territorio', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Territorio.'}, {'tabla': 'vehiculos', 'variable_original': 'id_localidad', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Localidad.'}, {'tabla': 'vehiculos', 'variable_original': 'id_tipo_contenedor', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de contenedor.'}, {'tabla': 'vehiculos', 'variable_original': 'hora_inicio_viaje', 'tipo_documentado': 'time without time zone', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Hora de inicio del viaje.'}, {'tabla': 'vehiculos', 'variable_original': 'modifica_kilometraje_movil', 'tipo_documentado': 'boolean', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Si modifica kilometraje desde móvil.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_vencimiento_tecnomecanica', 'tipo_documentado': 'date', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Vencimiento de la tecnomecánica.'}, {'tabla': 'vehiculos', 'variable_original': 'fecha_vencimiento_soap', 'tipo_documentado': 'date', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Vencimiento del SOAP.'}, {'tabla': 'vehiculos', 'variable_original': 'id_tipo_recurso', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de recurso.'}, {'tabla': 'vehiculos', 'variable_original': 'alias', 'tipo_documentado': 'character varying(50)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Alias del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'id_bodega', 'tipo_documentado': 'bigint', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Bodega.'}, {'tabla': 'vehiculos', 'variable_original': 'ciudad', 'tipo_documentado': 'character varying(10)', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Ciudad.'}, {'tabla': 'vehiculos', 'variable_original': 'imagen', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Imagen del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'idprogespecialactual', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Programación especial actual.'}, {'tabla': 'vehiculos', 'variable_original': 'tipo_sensor', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Tipo de sensor de pasajeros.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_min_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'MIN del módem secundario.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_dirip_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'IP del módem secundario.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_user_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Usuario del módem secundario.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_passwd_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Contraseña del módem secundario.'}, {'tabla': 'vehiculos', 'variable_original': 'modem_puk_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'PUK del módem secundario.'}, {'tabla': 'vehiculos', 'variable_original': 'sim_operador_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Operador SIM secundaria.'}, {'tabla': 'vehiculos', 'variable_original': 'sim_serie_2', 'tipo_documentado': 'character varying', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Serial SIM secundaria.'}, {'tabla': 'vehiculos', 'variable_original': 'fase', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Fase del vehículo.'}, {'tabla': 'vehiculos', 'variable_original': 'n_sentados', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Capacidad de asientos (sentados).'}, {'tabla': 'vehiculos', 'variable_original': 'n_depie', 'tipo_documentado': 'integer', 'nulo_documentado': 'YES', 'descripcion_documentada': 'Capacidad de pasajeros de pie.'}])


from pathlib import Path
from html import escape
from IPython.display import HTML, display
import pandas as pd
import pyarrow.parquet as pq
import sqlite3
import tempfile
import pickle
from contextlib import contextmanager

# Criterios analíticos editables; no son propiedades calculadas de PostgreSQL.
CATALOGO_TABLAS = {
    'rutas': ('Catálogo de rutas', 'Contexto de la operación', 'Registro de ruta', 'Identificar y contextualizar SILENCIO', 'Catálogo actual: no garantiza vigencia histórica.'),
    'progvehiculos': ('Programación y registros de operación', 'Fuente principal de viajes', 'Registro de programación; unicidad pendiente', 'Construir viajes válidos y series temporales', 'Programación no equivale automáticamente a viaje realizado.'),
    'vehiculos': ('Catálogo de vehículos', 'Contexto de la flota', 'Registro de vehículo', 'Vincular atributos y revisar capacidades', 'La ruta actual del vehículo no prueba su ruta histórica; capacidad requiere validación.'),
    'rutas_segmentos': ('Segmentos de rutas', 'Referencia espacial', 'Registro de segmento', 'Relacionar observaciones con segmentos', 'Revisar configuración, actividad y vigencia.'),
    'rutas_segmentos_config': ('Configuración de segmentos', 'Contexto de segmentación', 'Registro de configuración', 'Interpretar la división espacial', 'No asumir una configuración constante en todo el período.'),
    'timbradas_segmentos': ('Timbradas por segmento', 'Observación de operación por segmento', 'Viaje–segmento–franja según documentación; validar', 'Explorar distribución espacial y temporal', 'El campo pasajeros no está validado como carga a bordo.'),
    'timbradas_anio': ('Agregados anuales de timbradas', 'Perfil histórico', 'Año–ruta–día–estacionalidad–franja; validar claves', 'Contrastar perfiles históricos', 'Revisar cómo se construyó el agregado para evitar fuga temporal.'),
    'timbradas_consolidado': ('Timbradas consolidadas', 'Perfil agregado', 'Ruta–día–estacionalidad–franja; validar claves', 'Contrastar patrones agregados', 'El rango temporal del consolidado debe confirmarse.'),
    'dias_semana': ('Días de la semana', 'Catálogo temporal', 'Registro de día', 'Interpretar códigos de día', 'Confirmar correspondencia de códigos antes de unir.'),
    'estacionalidades': ('Catálogo de estacionalidades', 'Contexto de calendario', 'Registro de estacionalidad', 'Interpretar períodos especiales', 'Catálogo compartido: revisar proceso y pertinencia.'),
    'estacionalidades_anio': ('Calendario anual de estacionalidades', 'Referencia temporal', 'Registro de período anual', 'Relacionar fechas con períodos especiales', 'Validar proceso, intervalos y solapamientos.'),
    'novedades_viales': ('Catálogo de novedades viales', 'Contexto de eventos', 'Registro de tipo de novedad', 'Interpretar eventos documentados', 'Un catálogo no demuestra la ocurrencia de eventos.'),
    'novedades_viales_fecha': ('Novedades viales por fecha', 'Eventos temporales', 'Registro de novedad fechada', 'Evaluar contexto de incidentes', 'Ausencia con el filtro no equivale a ausencia de incidentes.'),
    'novedades_viales_ruta': ('Relación de novedades y rutas', 'Vínculo entre evento y ruta', 'Registro de asociación', 'Atribuir eventos a rutas', 'Revisar cobertura de asociaciones.'),
    'Procesos': ('Configuración de procesos', 'Contexto del sistema', 'Registro de proceso', 'Interpretar funcionalidades habilitadas', 'Configuración actual; no garantiza funcionamiento histórico.'),
    'conteo_pasajeros_destino': ('Registros del sensor de pasajeros', 'Fuente de eventos del dispositivo', 'Registro de evento; unicidad pendiente', 'Validar conteos y evaluar carga a bordo', 'Extracción masiva diferida; consultar con filtros o agregaciones documentadas. No sumar acumulados como incrementos.'),
}
# Sobrescribir por (tabla, columna) cuando se confirme con SOBUSA.
# Campos admitidos: descripcion, tipo_analitico, unidad, papel, nota, estado_definicion.
DEFINICIONES_VARIABLES = {
 ('progvehiculos','numeropasajeros'): dict(tipo_analitico='Cuantitativa discreta según definición documental', unidad='Pasajeros/abordajes; confirmar', papel='Resultado de operación', nota='No equivale automáticamente a ocupación ni demanda no atendida.'),
 ('progvehiculos','fechasalida'): dict(tipo_analitico='Temporal', unidad='Fecha y hora', papel='Salida programada', nota='Zona horaria y momento de disponibilidad por confirmar.'),
 ('progvehiculos','fechainicio'): dict(tipo_analitico='Temporal', unidad='Fecha y hora', papel='Inicio registrado', nota='No imputar desde la salida programada.'),
 ('progvehiculos','fechafin'): dict(tipo_analitico='Temporal', unidad='Fecha y hora', papel='Fin registrado', nota='Resultado posterior al viaje; revisar fuga de información.'),
 ('timbradas_segmentos','pasajeros'): dict(unidad='Pendiente de confirmar', papel='Conteo registrado', nota='No habilitado para calcular ocupación; revisar negativos.'),
 ('vehiculos','capacidad'): dict(unidad='Plazas según documentación', papel='Capacidad nominal', nota='Confirmar vigencia histórica; cero no sirve como denominador.'),
}


@contextmanager
def _almacen_unicos():
    """Valores distintos en disco; caché SQLite limitada a unos 8 MiB."""
    with tempfile.TemporaryDirectory(prefix="sobusa_unicos_") as carpeta:
        con = sqlite3.connect(str(Path(carpeta)/"unicos.sqlite"))
        try:
            con.execute("PRAGMA cache_size=-8192")
            con.execute("PRAGMA temp_store=FILE")
            con.execute("CREATE TABLE valores (col TEXT, valor BLOB, PRIMARY KEY(col,valor)) WITHOUT ROWID")
            yield con
        finally:
            con.close()

def _clave_unica(valor):
    # Conserva estructura y tipos; no publica los valores en el diccionario.
    if isinstance(valor, dict):
        valor = ("dict", tuple(sorted((k, _clave_unica(v)) for k,v in valor.items())))
    elif isinstance(valor, (list, tuple)):
        valor = ("lista", tuple(_clave_unica(v) for v in valor))
    elif hasattr(valor, "tolist"):
        valor = valor.tolist()
        return _clave_unica(valor)
    return pickle.dumps(valor, protocol=4)

DESCRIPCIONES_AMPLIADAS = {
 ('estacionalidades','id'): 'Identificador de la estacionalidad en el catálogo. Permite vincularla con otras tablas; la unicidad se comprueba por separado.',
 ('estacionalidades','proceso'): 'Código del proceso operacional al que pertenece la estacionalidad. Permite verificar su pertinencia para SOBUSA antes de integrar el calendario.',
 ('estacionalidades','tipo'): 'Categoría general de la estacionalidad. La documentación menciona FESTIVO_NACIONAL, FIESTA_ATLANTICO y TEMPORADA_NACIONAL; comprobar los códigos observados.',
 ('estacionalidades','nombre'): 'Nombre específico de la estacionalidad, como Navidad o Carnaval según la documentación. Facilita interpretar el contexto del calendario.',
 ('estacionalidades','descripcion'): 'Texto descriptivo con información adicional de la estacionalidad. No sustituye las fechas de vigencia del calendario anual.',
 ('progvehiculos','idprogramacion'): 'Identificador del registro de programación. Se utiliza para relacionar el viaje programado con fuentes vinculadas; comprobar unicidad y cobertura de las relaciones.',
 ('progvehiculos','idruta'): 'Identificador de la ruta asociada al registro de programación. Fundamenta la selección de SILENCIO y su vinculación con el catálogo de rutas.',
 ('progvehiculos','idvehiculo'): 'Identificador del vehículo asociado al registro. Permite integrar atributos de la flota; la asignación registrada en el viaje debe distinguirse de la ruta actual del vehículo.',
 ('progvehiculos','idconductor'): 'Identificador del conductor asociado al registro. Es una categoría nominal, aunque se almacene como número; no representa una magnitud.',
 ('progvehiculos','fechasalida'): 'Fecha y hora de salida programada. Permite analizar la planificación y definir el alcance temporal aplicado a esta extracción; no prueba la salida efectiva.',
 ('progvehiculos','fechainicio'): 'Fecha y hora de inicio registrada. Permite contrastar la operación con la programación, previa validación de su significado y zona horaria.',
 ('progvehiculos','fechafin'): 'Fecha y hora de finalización registrada. Se conoce después del viaje y requiere cautela al construir predictores disponibles antes de su ejecución.',
 ('progvehiculos','numeropasajeros'): 'Cantidad de pasajeros registrada para la programación según el diccionario. Debe confirmarse si representa abordajes u otra medida; no equivale por sí sola a ocupación ni demanda no atendida.',
 ('vehiculos','capacidad'): 'Capacidad nominal del vehículo según la documentación. Debe confirmarse su unidad y vigencia; los valores cero o nulos no permiten calcular una razón de ocupación válida.'
}

def _descripcion_ampliada(tabla, columna, base):
    especifica = DESCRIPCIONES_AMPLIADAS.get((tabla,columna))
    if especifica:
        return especifica
    if pd.isna(base):
        return 'Sin definición documental disponible. Su significado debe confirmarse antes de utilizarla en el análisis.'
    # No atribuir unidades, relaciones ni códigos que no estén documentados.
    return str(base)

# Conteos generales sin filtro de ruta ni periodo, reutilizados dentro de la ejecución.
# Cada COUNT usa la transacción de solo lectura y el timeout de consulta_pequena.
# No se carga SELECT * en memoria para obtener la dimensión.
if '_CONTEOS_GENERALES_FICHAS' not in globals():
    _CONTEOS_GENERALES_FICHAS = {}

def _dimension_general_ficha(tabla, columnas):
    formato = lambda n: f'{int(n):,}'.replace(',', '.')
    clave = (id(engine), str(globals().get('RUN_ID', BASE)), CFG['schema'], tabla)
    if clave not in _CONTEOS_GENERALES_FICHAS:
        if tabla not in set(ESQUEMA['tabla']):
            raise ValueError(f'Tabla no encontrada en ESQUEMA: {tabla}')
        # Comillas para nombres con mayúsculas o caracteres especiales.
        citar = lambda valor: '"' + str(valor).replace('"', '""') + '"'
        sql = f'SELECT COUNT(*) AS filas FROM {citar(CFG["schema"])}.{citar(tabla)}'
        try:
            print(f'{tabla}: contando filas generales sin filtros...', flush=True)
            conteo = consulta_pequena('conteo_general_' + tabla.encode('utf-8').hex(), sql)
            _CONTEOS_GENERALES_FICHAS[clave] = (int(conteo['filas'].iloc[0]), 'exacto')
        except Exception as error:
            # consulta_pequena cierra y revierte su transacción si falla.
            # Un timeout o un error de permisos no significa cero filas.
            estimadas = None
            inventario = globals().get('INVENTARIO')
            if isinstance(inventario, pd.DataFrame) and {'tabla', 'filas_aproximadas'}.issubset(inventario.columns):
                valores = inventario.loc[inventario.tabla.eq(tabla), 'filas_aproximadas']
                if not valores.empty and pd.notna(valores.iloc[0]) and valores.iloc[0] > 0:
                    estimadas = int(valores.iloc[0])
            if estimadas is None:
                valores = OBJETOS_BD.loc[OBJETOS_BD.tabla.eq(tabla), 'filas_estimadas_catalogo']
                if not valores.empty and pd.notna(valores.iloc[0]) and valores.iloc[0] > 0:
                    estimadas = int(valores.iloc[0])
            _CONTEOS_GENERALES_FICHAS[clave] = (estimadas, 'estimado' if estimadas is not None else 'pendiente')
            print(f'{tabla}: conteo general no completado ({type(error).__name__}); '
                  + ('se muestra una estimación.' if estimadas is not None else 'total pendiente de verificar.'), flush=True)
    filas, metodo = _CONTEOS_GENERALES_FICHAS[clave]
    if metodo == 'exacto':
        return f'{formato(filas)} filas × {columnas} columnas'
    if metodo == 'estimado':
        return f'≈ {formato(filas)} filas × {columnas} columnas (estimación; conteo exacto no completado)'
    return f'Total de filas no calculado × {columnas} columnas'


def _ficha_tabla_base(tabla, *, perfil=True, mostrar=True, guardar=False):
    """Usa ESQUEMA, RAW, REGISTRO y DOCUMENTADO. Cuenta filas generales en SQL sin descargar la tabla.
    perfil=True recorre todos los archivos locales en lotes de 10.000 filas.
    Devuelve ficha (dict), diccionario (DataFrame), muestra (DataFrame) y HTML.
    """
    requeridas = {'tabla','columna','data_type','is_nullable','ordinal_position'}
    if not requeridas.issubset(ESQUEMA.columns):
        raise ValueError(f'Faltan campos del esquema: {sorted(requeridas-set(ESQUEMA.columns))}')
    esquema = ESQUEMA.loc[ESQUEMA.tabla.eq(tabla)].sort_values('ordinal_position').copy()
    if esquema.empty:
        raise ValueError(f'Tabla no encontrada en ESQUEMA: {tabla}')
    docs = DOCUMENTADO.rename(columns={'variable_original':'columna'})
    docs = docs.loc[docs.tabla.eq(tabla), ['tabla','columna','descripcion_documentada']]
    dic = esquema.merge(docs, on=['tabla','columna'], how='left', validate='one_to_one')
    diferida = tabla in globals().get("TABLAS_DIFERIDAS", {"conteo_pasajeros_destino"})
    perfil = perfil and not diferida
    archivos = sorted(Path(RAW[tabla]).glob('parte-*.parquet')) if tabla in RAW and not diferida else []
    registros = [r for r in REGISTRO if r.get('fuente') == tabla and r.get('estado') != 'metadatos']
    registro = registros[-1] if registros else {}
    estado = 'Extracción diferida; consulta bajo demanda' if diferida else registro.get('estado','sin registro de extracción')
    filas = sum(pq.ParquetFile(p).metadata.num_rows for p in archivos) if archivos else (0 if estado=='vacia_en_filtro' else None)
    presentes = set()
    for p in archivos:
        presentes.update(pq.ParquetFile(p).schema_arrow.names)
    fechas = set(dic.loc[dic.data_type.str.contains('date|timestamp', case=False, regex=True), 'columna'])
    faltantes = {c:0 for c in presentes}
    observadas = {c:0 for c in presentes}
    rangos = {}
    muestra_df = pd.DataFrame()
    if archivos:
        primer_lote = next(pq.ParquetFile(archivos[0]).iter_batches(batch_size=5), None)
        if primer_lote is not None:
            muestra_df = primer_lote.to_pandas(types_mapper=pd.ArrowDtype)
    unicos = {}
    if perfil and archivos:
        with _almacen_unicos() as con:
            for numero,p in enumerate(archivos,1):
                for batch in pq.ParquetFile(p).iter_batches(batch_size=10000):
                    lote = batch.to_pandas(types_mapper=pd.ArrowDtype)
                    for c in lote.columns:
                        faltantes[c] += int(lote[c].isna().sum())
                        observadas[c] += len(lote)
                        valores = lote[c].dropna()
                        # Deduplicación por lote antes de insertar en disco.
                        claves = {_clave_unica(v) for v in valores.tolist()}
                        con.executemany("INSERT OR IGNORE INTO valores VALUES (?,?)", ((c,v) for v in claves))
                    con.commit()
                if numero % 20 == 0 or numero == len(archivos):
                    print(f"{tabla}: perfil {numero}/{len(archivos)} archivos", flush=True)
            unicos = dict(con.execute("SELECT col,COUNT(*) FROM valores GROUP BY col"))
            for c in presentes:
                unicos.setdefault(c,0)
    def tipo_propuesto(r):
        t = str(r.data_type).lower()
        if r.columna == 'id' or r.columna.startswith('id') or r.columna in {'idruta','idvehiculo','idprogramacion','idrutageo','idrutaequivalente'}:
            return 'Identificador nominal propuesto; validar'
        if 'date' in t or 'time' in t: return 'Temporal según tipo SQL'
        if t == 'boolean': return 'Cualitativa binaria según tipo SQL'
        if t in {'smallint','integer','bigint'}: return 'Numérica entera; uso analítico pendiente'
        if t in {'numeric','decimal','real','double precision'}: return 'Numérica; escala analítica pendiente'
        return 'Texto/otra estructura; clasificación pendiente'
    dic['tipo_analitico'] = dic.apply(tipo_propuesto,axis=1)
    dic['descripcion'] = [_descripcion_ampliada(tabla,r.columna,r.descripcion_documentada) for r in dic.itertuples()]
    dic['estado_definicion'] = dic.descripcion_documentada.map(lambda x:'Documentada; validación empresarial pendiente' if pd.notna(x) else 'Pendiente de documentar')
    dic['extraida'] = dic.columna.map(lambda c:'Sí' if c in presentes else 'No verificable: fuente vacía' if estado=='vacia_en_filtro' else 'No')
    dic['unidad'] = 'Pendiente de documentar'
    dic['papel'] = 'Pendiente de documentar'
    dic['nota'] = 'Conservar faltantes; definir reglas antes de limpiar.'
    for i,r in dic.iterrows():
        for campo,valor in DEFINICIONES_VARIABLES.get((tabla,r.columna),{}).items():
            dic.at[i,campo] = valor
    dic['filas_observadas'] = dic.columna.map(observadas) if perfil else pd.NA
    dic['faltantes'] = dic.columna.map(faltantes) if perfil else pd.NA
    dic['faltantes_pct'] = [100*f/n if pd.notna(n) and n>0 else pd.NA for f,n in zip(dic.faltantes,dic.filas_observadas)]
    dic['valores_unicos'] = dic.columna.map(unicos) if perfil and archivos else pd.NA
    dic['min_fecha'] = dic.columna.map(lambda c: str(rangos[c][0]) if c in rangos else '')
    dic['max_fecha'] = dic.columna.map(lambda c: str(rangos[c][1]) if c in rangos else '')
    formato = lambda n: f'{int(n):,}'.replace(',','.')
    dimension_general = _dimension_general_ficha(tabla, len(dic))
    dimension_extraida = (f'{formato(filas)} filas × {len(presentes)} columnas' if filas is not None
                          else 'No extraída en esta fase')
    titulo,papel,unidad,uso,nota = CATALOGO_TABLAS.get(tabla,('Descripción pendiente','Pendiente','Pendiente','Pendiente','Completar documentación.'))
    ficha = dict(tabla=tabla, descripcion=titulo, papel=papel, unidad_observacion=unidad,
                 granularidad=unidad, uso_mvp_propuesto=uso, utilidad=uso, nota=nota,
                 estado=estado, dimension_general=dimension_general, dimension_extraida=dimension_extraida, filas_extraidas=filas, columnas_esquema=len(dic),
                 columnas_en_archivos=len(presentes), campos_fecha=', '.join(sorted(fechas)) or 'Sin columnas DATE/TIMESTAMP en el esquema',
                 perfil='Completo sobre archivos locales' if perfil and archivos else 'No calculado: sin filas' if not archivos else 'Desactivado',
                 parametros_extraccion=registro.get('parametros',{}))
    e=lambda v:escape(str(v))
    columnas=['columna','descripcion','data_type','tipo_analitico','is_nullable','valores_unicos','faltantes','faltantes_pct']
    extraidas = dic.copy() if diferida else dic.loc[dic.columna.isin(presentes)].copy()
    no_extraidas = dic.loc[~dic.columna.isin(presentes)].copy()
    vista=extraidas[columnas].rename(columns={'columna':'Variable','descripcion':'Descripción documental','data_type':'Tipo SQL','tipo_analitico':'Tipo analítico','is_nullable':'Permite NULL en BD','valores_unicos':'Valores únicos (sin nulos)','faltantes':'Faltantes','faltantes_pct':'Faltantes (%)'})
    vista['Permite NULL en BD']=vista['Permite NULL en BD'].map({'YES':'Sí','NO':'No'}).fillna('Sin información')
    vista['Valores únicos (sin nulos)']=vista['Valores únicos (sin nulos)'].map(lambda v:formato(v) if pd.notna(v) else 'No calculado')
    vista['Faltantes']=vista['Faltantes'].map(lambda v:f'{int(v):,}'.replace(',','.') if pd.notna(v) else 'No calculado')
    vista['Faltantes (%)']=vista['Faltantes (%)'].map(lambda v:f'{v:.2f}' if pd.notna(v) else 'No calculado')
    excluidas=no_extraidas[['columna','descripcion','data_type']].rename(columns={'columna':'Variable','descripcion':'Descripción documental','data_type':'Tipo SQL'})
    ficha['columnas_fuera_archivos']=len(no_extraidas)
    campos_visibles=['dimension_general','dimension_extraida','descripcion','papel','unidad_observacion','uso_mvp_propuesto','nota','estado','filas_extraidas','columnas_esquema','columnas_en_archivos','columnas_fuera_archivos','perfil','parametros_extraccion']
    resumen=''.join(f'<div><b>{e(k.replace("_"," ").capitalize())}</b><span>{e(ficha[k])}</span></div>' for k in campos_visibles)
    particulares=extraidas.loc[extraidas.nota.ne('Conservar faltantes; definir reglas antes de limpiar.'),['columna','nota']]
    consideraciones=''.join(f'<li><strong>{e(r.columna)}</strong>: {e(r.nota)}</li>' for r in particulares.itertuples())
    consideraciones='<h3>Consideraciones específicas</h3><ul>'+consideraciones+'</ul>' if consideraciones else ''
    css='''<style>.sobusa-ficha{font-family:Arial,sans-serif;color:#173d27;line-height:1.5;border:1px solid #cce4d2;border-radius:14px;padding:24px;background:#fff;margin:16px 0}.sobusa-ficha h2{color:#166534;margin:0;border-bottom:4px solid #238b45;padding-bottom:12px}.sobusa-ficha h3{color:#166534}.sobusa-ficha .resumen{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:12px;margin:20px 0}.sobusa-ficha .resumen div{background:#f0f8f2;border-radius:8px;padding:12px}.sobusa-ficha b,.sobusa-ficha span{display:block}.sobusa-ficha .scroll{overflow:auto;max-height:650px}.sobusa-ficha table{border-collapse:collapse;font-size:13px;width:100%;text-align:left}.sobusa-ficha th{background:#166534;color:white;padding:10px;position:sticky;top:0;text-align:left;white-space:nowrap}.sobusa-ficha td{padding:10px;border-bottom:1px solid #dcebdd;min-width:110px;max-width:330px;vertical-align:top;text-align:left}.sobusa-ficha tr:nth-child(even){background:#f0f8f2}.sobusa-ficha .nota{padding:12px;border-left:4px solid #238b45;background:#f0f8f2}@media print{.sobusa-ficha .scroll{max-height:none;overflow:visible}.sobusa-ficha th{position:static}}</style>'''
    principal= vista.to_html(index=False,escape=True,border=0) if not vista.empty else '<p>Sin columnas verificables en archivos locales. Fuente vacía o sin extracción; no se calculan faltantes.</p>'
    texto_exclusion='Estas variables existen en el esquema, pero no están disponibles en los archivos locales. Para una fuente extraída, revisar la selección CAMPOS antes de decidir si se necesitan. Para una fuente vacía, no se puede comprobar su proyección desde los archivos.'
    auditoria='<details><summary>Variables del esquema no disponibles en archivos ('+str(len(excluidas))+')</summary><p>'+texto_exclusion+'</p><div class="scroll">'+excluidas.to_html(index=False,escape=True,border=0)+'</div></details>' if not excluidas.empty and not diferida else ''
    html=css+f'<section class="sobusa-ficha"><h2>{e(tabla)}</h2><p>Ruta SILENCIO</p><div class="resumen">{resumen}</div><p class="nota">Dimensión general: COUNT(*) sin filtros de ruta ni periodo, en el momento de la consulta; si no se completa, se identifica la estimación o el total pendiente. Un conteo exacto cero sí confirma ausencia de filas visibles en esa consulta. Dimensión extraída, faltantes y valores únicos: archivos locales del alcance seleccionado. Valores únicos excluyen nulos y se cuentan sobre todos los archivos, no sobre el head.</p><h3>Diccionario de variables</h3><div class="scroll">'+principal+'</div>'+consideraciones+auditoria+'<h3>Head — primeros cinco registros disponibles</h3><p>Primer archivo; sin muestreo aleatorio ni orden cronológico garantizado.</p><div class="scroll">'+muestra_df.to_html(index=False,escape=True,border=0)+'</div></section>'
    if guardar:
        destino=Path(BASE)/'reportes'/'fichas_tablas'
        destino.mkdir(parents=True,exist_ok=True)
        nombre=tabla.encode('utf-8').hex() # Nombre seguro incluso para identificadores con separadores.
        (destino/f'{nombre}.html').write_text('<!doctype html><meta charset="utf-8">'+html,encoding='utf-8')
        dic.to_csv(destino/f'{nombre}_variables.csv',index=False)
    if mostrar: display(HTML(html))
    return {'ficha':ficha,'diccionario':dic,'no_extraidas':no_extraidas,'muestra':muestra_df,'html':html}


# resultado = ficha_tabla('progvehiculos', perfil=True, guardar=True)
CATALOGO_TABLAS.update({
'rutas_segmentos_porcion': ('Detalle de porciones de segmentos de ruta', 'Contexto de segmentación', 'Registro de porción', 'Contextualizar patrones de operación', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_anio_trazabilidad': ('Trazabilidad del agregado anual', 'Auditoría del agregado', 'Registro de trazabilidad', 'Examinar procedencia de los registros', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_anio_zonas': ('Detalle anual por zona', 'Perfil agregado de operación', 'Registro de agregado por zona', 'Contextualizar patrones de operación', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_anio_zonas_trazabilidad': ('Trazabilidad del detalle anual por zona', 'Auditoría del agregado', 'Registro de trazabilidad', 'Examinar procedencia de los registros', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_consolidado_zonas': ('Detalle consolidado por zona', 'Perfil agregado de operación', 'Registro de consolidado por zona', 'Contextualizar patrones de operación', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_segmentos_anio': ('Agregado anual por segmento', 'Perfil agregado de operación', 'Registro de agregado por segmento', 'Contextualizar patrones de operación', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_segmentos_anio_trazabilidad': ('Trazabilidad del agregado anual por segmento', 'Auditoría del agregado', 'Registro de trazabilidad', 'Examinar procedencia de los registros', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
'timbradas_segmentos_consolidado': ('Consolidado por segmento', 'Perfil agregado de operación', 'Registro de consolidado por segmento', 'Contextualizar patrones de operación', 'Las claves declaradas describen las relaciones; la cardinalidad observada se audita antes de unir.'),
})
# Ejecutar debajo de la definición vigente de ficha_tabla.
from html import escape
from IPython.display import HTML, display
from pathlib import Path
import pandas as pd

OBJETOS_TECNICOS = {
    'geometry_columns': 'Vista técnica: inventario de columnas geometry.',
    'geography_columns': 'Vista técnica: inventario de columnas geography.',
    'raster_columns': 'Vista técnica: inventario de columnas ráster.',
    'raster_overviews': 'Vista técnica: inventario de versiones de menor resolución de rásteres.',
    'spatial_ref_sys': 'Catálogo técnico: sistemas de referencia espacial y códigos SRID.'
}
TABLAS_DIFERIDAS = {'conteo_pasajeros_destino'}

# Definiciones específicas: reglas analíticas, no CHECK impuestos por la BD.
REGLAS_VARIABLES = {
 ('progvehiculos','numeropasajeros'): dict(
    operacional='Valor original de numeropasajeros por programación; sin sumar sensores ni imputar faltantes.',
    unidad='Pasajeros registrados', papel='Resultado; posible objetivo',
    validez='Propuesta: número entero no negativo. Marcar negativos; conservar cero y NULL por separado.',
    nulos_ceros='NULL: conteo ausente. Cero: valor registrado; no prueba por sí solo ausencia de demanda.',
    disponibilidad='Después de registrar el conteo del viaje.',
    fuga='Alta como predictor del mismo viaje; admisible como objetivo o rezago disponible.'),
 ('progvehiculos','fechasalida'): dict(
    operacional='Fecha y hora programadas almacenadas; no sustituir por fechainicio.',
    unidad='Fecha y hora', papel='Contexto temporal; posible predictor',
    validez='Propuesta: fecha interpretable; aplicar zona horaria común y alcance de extracción.',
    disponibilidad='Cuando se registra la programación; requiere versión histórica disponible al corte.',
    fuga='Condicional: una programación modificada después del corte puede introducir fuga.'),
 ('progvehiculos','fechainicio'): dict(
    operacional='Fecha y hora de inicio registradas, conservando NULL sin copiar fechasalida.',
    unidad='Fecha y hora', papel='Resultado de operación',
    disponibilidad='Al registrar el inicio efectivo.',
    fuga='Alta para predicción antes del inicio del mismo viaje.'),
 ('progvehiculos','fechafin'): dict(
    operacional='Fecha y hora de finalización registradas; conservar fecha completa al cruzar medianoche.',
    unidad='Fecha y hora', papel='Resultado; indicador de calidad',
    validez='Propuesta: fechafin >= fechainicio cuando ambas existen; marcar discrepancias sin corregir automáticamente.',
    disponibilidad='Al registrar la finalización.',
    fuga='Alta para predicción anterior al fin del mismo viaje.'),
 ('vehiculos','capacidad'): dict(
    operacional='Capacidad nominal del catálogo; no estimar a partir del total de pasajeros transportados.',
    unidad='Plazas nominales según documentación', papel='Contexto de flota',
    validez='Propuesta: capacidad > 0 para utilizarla como denominador.',
    nulos_ceros='NULL o cero: capacidad no utilizable como denominador; conservar el dato original.',
    disponibilidad='Versión del catálogo vigente y disponible al corte.',
    fuga='Condicional: el catálogo actual puede incorporar cambios posteriores al periodo analizado.'),
 ('timbradas_segmentos','pasajeros'): dict(
    operacional='Valor original por registro de segmento; sin convertirlo automáticamente en carga a bordo.',
    unidad='Conteo registrado', papel='Resultado de operación',
    validez='Propuesta: registrar negativos como anomalías; no descartarlos sin aclarar el cálculo del campo.',
    disponibilidad='Después de registrar la observación del segmento.',
    fuga='Alta si corresponde al segmento o intervalo que se pretende predecir.')
}

def _llaves_ficha(tabla):
    origen = globals().get('LLAVES', pd.DataFrame())
    if origen.empty or not {'tabla','constraint_name','constraint_type'}.issubset(origen.columns):
        return pd.DataFrame(columns=['Restricción','Tipo','Columnas','Definición'])
    filas = []
    seleccion = origen.loc[origen.tabla.eq(tabla)].copy()
    for (nombre,tipo), grupo in seleccion.groupby(['constraint_name','constraint_type'],dropna=False):
        if 'ordinal_position' in grupo:
            grupo = grupo.sort_values('ordinal_position')
        col = 'columna' if 'columna' in grupo else 'column_name'
        columnas = ', '.join(grupo[col].dropna().astype(str)) if col in grupo else ''
        definicion = grupo['definicion'].iloc[0] if 'definicion' in grupo else 'Detalle no incluido en los metadatos cargados.'
        filas.append([nombre,tipo,columnas,definicion])
    return pd.DataFrame(filas,columns=['Restricción','Tipo','Columnas','Definición'])

def _operacionalizar(tabla,dic,granularidad,llaves):
    filas = []
    for r in dic.to_dict('records'):
        c,t = r['columna'], str(r['data_type']).lower()
        rol_llave = []
        for llave in llaves.to_dict('records'):
            if c in [v.strip() for v in llave['Columnas'].split(',')]:
                rol_llave.append(f"{llave['Tipo']} ({llave['Restricción']})")
        identificador = bool(rol_llave) or c == 'id' or c.startswith('id_') or c in {'idruta','idvehiculo','idconductor','idprogramacion','idestacionalidad','iddia'}
        temporal = any(x in t for x in ['date','time'])
        descripcion = r.get('descripcion','')
        base = dict(
            origen=f'{tabla}.{c}', conceptual=descripcion,
            operacional=f'Lectura directa de {tabla}.{c}, sin transformación ni imputación.',
            formula='No aplica: variable original.', granularidad=granularidad,
            unidad='Sin unidad física: código' if identificador else 'Fecha/hora según tipo SQL' if temporal else 'Indicador lógico' if t=='boolean' else 'Unidad no especificada en la documentación' if t in {'numeric','integer','bigint','smallint','real','double precision'} else 'Texto o estructura según tipo SQL',
            papel='Identificador / vínculo' if identificador else 'Contexto; uso predictivo sujeto a disponibilidad',
            codificacion='TRUE / FALSE; NULL si el esquema lo permite' if t=='boolean' else 'Código de identificación; no interpretar como magnitud' if identificador else 'Representación temporal según tipo SQL' if temporal else 'Representación original; sin recodificación',
            permitidos='Dominio SQL '+str(r['data_type'])+'; no equivale a un dominio empresarial validado.',
            validez='Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.',
            nulos_ceros='NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.',
            disponibilidad='Solo usar la versión registrada y accesible antes del corte de predicción.',
            fuga='Condicional: depende del momento de registro y de la versión utilizada.',
            llave='; '.join(rol_llave) or 'Sin rol de llave declarado en los metadatos disponibles',
            evidencia='Descripción documental' if pd.notna(r.get('descripcion_documentada')) else 'Esquema SQL; significado empresarial no documentado')
        if t=='boolean':
            base['permitidos']='TRUE, FALSE; NULL únicamente si is_nullable=YES.'
        if temporal:
            base['validez']='Propuesta: valor temporal interpretable; conservar fecha completa y aplicar la zona horaria acordada.'
        if tabla.startswith('timbradas_') and tabla!='timbradas_segmentos':
            base['disponibilidad']='Después del cierre y cálculo del agregado del periodo correspondiente.'
            base['fuga']='Alta si el agregado incluye fechas posteriores al corte; usar solo historia ya disponible.'
        if tabla in {'rutas','vehiculos','Procesos','rutas_segmentos','rutas_segmentos_config','rutas_segmentos_porcion'}:
            base['fuga']='Condicional: catálogo actual sin garantía de versión histórica.'
        if identificador:
            base['papel']='Identificador / vínculo'
        base.update(REGLAS_VARIABLES.get((tabla,c),{}))
        r.update(base); filas.append(r)
    return pd.DataFrame(filas)

def ficha_tabla(tabla, *, perfil=True, mostrar=True, guardar=False):
    if tabla in OBJETOS_TECNICOS:
        raise ValueError('Objeto técnico; se documenta en el inventario técnico, no como tabla operacional.')
    resultado = _ficha_tabla_base(tabla,perfil=perfil and tabla not in TABLAS_DIFERIDAS,mostrar=False,guardar=False)
    ficha = resultado['ficha']; llaves = _llaves_ficha(tabla)
    dic = _operacionalizar(tabla,resultado['diccionario'],ficha['unidad_observacion'],llaves)
    # El diccionario siempre muestra el esquema completo, aunque no haya filas.
    campos = {'columna':'Variable','origen':'Origen','conceptual':'Significado conceptual',
              'operacional':'Definición operacional','data_type':'Tipo SQL','tipo_analitico':'Tipo analítico',
              'unidad':'Unidad','granularidad':'Granularidad','papel':'Papel analítico',
              'formula':'Fórmula','codificacion':'Codificación','permitidos':'Valores permitidos',
              'validez':'Reglas de validez','nulos_ceros':'Nulos, ceros y especiales',
              'is_nullable':'Permite NULL en BD','llave':'Rol de llave',
              'disponibilidad':'Disponibilidad para predecir','fuga':'Riesgo de data leakage',
              'evidencia':'Base de la definición','extraida':'En extracción',
              'valores_unicos':'Valores únicos sin nulos','faltantes':'Faltantes','faltantes_pct':'Faltantes (%)'}
    vista=dic[list(campos)].rename(columns=campos).copy()
    vista['Permite NULL en BD']=vista['Permite NULL en BD'].map({'YES':'Sí','NO':'No'})
    for col in ['Valores únicos sin nulos','Faltantes']:
        vista[col]=vista[col].map(lambda v:f'{int(v):,}'.replace(',','.') if pd.notna(v) else 'No calculado')
    vista['Faltantes (%)']=vista['Faltantes (%)'].map(lambda v:f'{float(v):.2f}' if pd.notna(v) else 'No calculado')
    if tabla in TABLAS_DIFERIDAS:
        vista['En extracción']='Diferida; consulta bajo demanda'
    # Una extracción vacía no tiene cero variables: tiene el esquema previsto.
    if ficha['filas_extraidas']==0:
        ficha['dimension_extraida']=f"0 filas × {len(dic)} columnas previstas; sin archivos de datos"
    ficha['dimension_general']=ficha['dimension_general'].replace('Filas no verificadas','Total de filas no calculado')
    e=lambda v:escape(str(v))
    tarjetas=['dimension_general','dimension_extraida','descripcion','papel','unidad_observacion','uso_mvp_propuesto','estado']
    resumen=''.join(f'<div><b>{e(k.replace("_"," ").capitalize())}</b><span>{e(ficha[k])}</span></div>' for k in tarjetas)
    seccion_llaves=(llaves.to_html(index=False,escape=True,border=0) if not llaves.empty else '<p>No hay restricciones declaradas visibles en los metadatos cargados. Esto no demuestra ausencia de claves lógicas ni unicidad de los datos.</p>')
    muestra=resultado['muestra'].copy()
    for c in muestra:
        if any(token in c.lower() for token in ['clave','password','token','secret']):
            muestra[c]='[oculto]'
    css=resultado['html'].split('</style>',1)[0]+'</style>'
    html=css+f'<section class="sobusa-ficha"><h2>{e(tabla)}</h2><div class="resumen">{resumen}</div></p><h3>Llaves y restricciones declaradas</h3><div class="scroll">{seccion_llaves}</div><h3>Diccionario y operacionalización</h3><div class="scroll">{vista.to_html(index=False,escape=True,border=0)}</div><h3>Head</h3><p>Orden del primer archivo; no garantiza orden cronológico ni representatividad.</p><div class="scroll">{muestra.to_html(index=False,escape=True,border=0) if not muestra.empty else "Sin registros locales disponibles."}</div></section>'
    resultado.update(ficha=ficha,diccionario=dic,llaves=llaves,muestra=muestra,html=html)
    if guardar:
        destino=Path(BASE)/'reportes'/'fichas_tablas';destino.mkdir(parents=True,exist_ok=True)
        nombre=tabla.encode().hex()
        (destino/f'{nombre}.html').write_text('<!doctype html><meta charset="utf-8">'+html,encoding='utf-8')
        dic.to_csv(destino/f'{nombre}_variables.csv',index=False)
        llaves.to_csv(destino/f'{nombre}_llaves.csv',index=False)
    if mostrar: display(HTML(html))
    return resultado

TECNICOS = pd.DataFrame([{'Objeto':t,'Descripción':v} for t,v in OBJETOS_TECNICOS.items() if t in set(ESQUEMA.tabla)])
if not TECNICOS.empty:
    TECNICOS=TECNICOS.merge(OBJETOS_BD.rename(columns={'tabla':'Objeto','tipo_objeto':'Tipo SQL'}),on='Objeto',how='left')
    display(TECNICOS)
TABLAS_FICHAS=sorted(set(ESQUEMA.tabla)-set(OBJETOS_TECNICOS))
for tabla in TABLAS_FICHAS:
    ficha_tabla(tabla,perfil=tabla not in TABLAS_DIFERIDAS,mostrar=True,guardar=True)


,Objeto,Descripción,filas_estimadas_catalogo,Tipo SQL
0,geometry_columns,Vista técnica: inventario de columnas geometry.,-1.0,Vista
1,geography_columns,Vista técnica: inventario de columnas geography.,-1.0,Vista
2,raster_columns,Vista técnica: inventario de columnas ráster.,-1.0,Vista
3,raster_overviews,Vista técnica: inventario de versiones de meno...,-1.0,Vista
4,spatial_ref_sys,Catálogo técnico: sistemas de referencia espac...,8500.0,Tabla


Procesos: perfil 1/1 archivos


Variable,Origen,Significado conceptual,Definición operacional,Tipo SQL,Tipo analítico,Unidad,Granularidad,Papel analítico,Fórmula,Codificación,Valores permitidos,Reglas de validez,"Nulos, ceros y especiales",Permite NULL en BD,Rol de llave,Disponibilidad para predecir,Riesgo de data leakage,Base de la definición,En extracción,Valores únicos sin nulos,Faltantes,Faltantes (%)
id,conteo_pasajeros_destino.id,PK del evento de conteo.,"Lectura directa de conteo_pasajeros_destino.id, sin transformación ni imputación.",bigint,Identificador nominal propuesto; validar,Sin unidad física: código,Registro de evento; unicidad pendiente,Identificador / vínculo,No aplica: variable original.,Código de identificación; no interpretar como magnitud,Dominio SQL bigint; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.,Sí,Sin rol de llave declarado en los metadatos disponibles,Solo usar la versión registrada y accesible antes del corte de predicción.,Condicional: depende del momento de registro y de la versión utilizada.,Descripción documental,Diferida; consulta bajo demanda,No calculado,No calculado,No calculado
fecha,conteo_pasajeros_destino.fecha,Fecha/hora del evento.,"Lectura directa de conteo_pasajeros_destino.fecha, sin transformación ni imputación.",timestamp without time zone,Temporal según tipo SQL,Fecha/hora según tipo SQL,Registro de evento; unicidad pendiente,Contexto; uso predictivo sujeto a disponibilidad,No aplica: variable original.,Representación temporal según tipo SQL,Dominio SQL timestamp without time zone; no equivale a un dominio empresarial validado.,Propuesta: valor temporal interpretable; conservar fecha completa y aplicar la zona horaria acordada.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.,Sí,Sin rol de llave declarado en los metadatos disponibles,Solo usar la versión registrada y accesible antes del corte de predicción.,Condicional: depende del momento de registro y de la versión utilizada.,Descripción documental,Diferida; consulta bajo demanda,No calculado,No calculado,No calculado
idvehiculo,conteo_pasajeros_destino.idvehiculo,Vehículo del evento.,"Lectura directa de conteo_pasajeros_destino.idvehiculo, sin transformación ni imputación.",character varying,Identificador nominal propuesto; validar,Sin unidad física: código,Registro de evento; unicidad pendiente,Identificador / vínculo,No aplica: variable original.,Código de identificación; no interpretar como magnitud,Dominio SQL character varying; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.,Sí,Sin rol de llave declarado en los metadatos disponibles,Solo usar la versión registrada y accesible antes del corte de predicción.,Condicional: depende del momento de registro y de la versión utilizada.,Descripción documental,Diferida; consulta bajo demanda,No calculado,No calculado,No calculado
acumulada_subida,conteo_pasajeros_destino.acumulada_subida,Acumulado de subidas del torniquete.,"Lectura directa de conteo_pasajeros_destino.acumulada_subida, sin transformación ni imputación.",bigint,Numérica entera; uso analítico pendiente,Unidad no especificada en la documentación,Registro de evento; unicidad pendiente,Contexto; uso predictivo sujeto a disponibilidad,No aplica: variable original.,Representación original; sin recodificación,Dominio SQL bigint; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticament

dias_semana: perfil 1/1 archivos


estacionalidades: perfil 1/1 archivos


...

timbradas_anio: perfil 2/2 archivos


Variable,Origen,Significado conceptual,Definición operacional,Tipo SQL,Tipo analítico,Unidad,Granularidad,Papel analítico,Fórmula,Codificación,Valores permitidos,Reglas de validez,"Nulos, ceros y especiales",Permite NULL en BD,Rol de llave,Disponibilidad para predecir,Riesgo de data leakage,Base de la definición,En extracción,Valores únicos sin nulos,Faltantes,Faltantes (%)
id_timbrada_anio_trazabilidad,timbradas_anio_trazabilidad.id_timbrada_anio_trazabilidad,PK trazabilidad.,"Lectura directa de timbradas_anio_trazabilidad.id_timbrada_anio_trazabilidad, sin transformación ni imputación.",bigint,Identificador nominal propuesto; validar,Sin unidad física: código,Registro de trazabilidad,Identificador / vínculo,No aplica: variable original.,Código de identificación; no interpretar como magnitud,Dominio SQL bigint; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.,No,Sin rol de llave declarado en los metadatos disponibles,Después del cierre y cálculo del agregado del periodo correspondiente.,Alta si el agregado incluye fechas posteriores al corte; usar solo historia ya disponible.,Descripción documental,No verificable: fuente vacía,No calculado,No calculado,No calculado
id_timbrada_anio,timbradas_anio_trazabilidad.id_timbrada_anio,FK al agregado anual.,"Lectura directa de timbradas_anio_trazabilidad.id_timbrada_anio, sin transformación ni imputación.",bigint,Identificador nominal propuesto; validar,Sin unidad física: código,Registro de trazabilidad,Identificador / vínculo,No aplica: variable original.,Código de identificación; no interpretar como magnitud,Dominio SQL bigint; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.,Sí,Sin rol de llave declarado en los metadatos disponibles,Después del cierre y cálculo del agregado del periodo correspondiente.,Alta si el agregado incluye fechas posteriores al corte; usar solo historia ya disponible.,Descripción documental,No verificable: fuente vacía,No calculado,No calculado,No calculado
idprogramacion,timbradas_anio_trazabilidad.idprogramacion,Programación que compone el agregado.,"Lectura directa de timbradas_anio_trazabilidad.idprogramacion, sin transformación ni imputación.",bigint,Identificador nominal propuesto; validar,Sin unidad física: código,Registro de trazabilidad,Identificador / vínculo,No aplica: variable original.,Código de identificación; no interpretar como magnitud,Dominio SQL bigint; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos especiales conservan su valor; no se reemplazan automáticamente.,Sí,Sin rol de llave declarado en los metadatos disponibles,Después del cierre y cálculo del agregado del periodo correspondiente.,Alta si el agregado incluye fechas posteriores al corte; usar solo historia ya disponible.,Descripción documental,No verificable: fuente vacía,No calculado,No calculado,No calculado
subidas,timbradas_anio_trazabilidad.subidas,Subidas de esa programación.,"Lectura directa de timbradas_anio_trazabilidad.subidas, sin transformación ni imputación.",integer,Numérica entera; uso analítico pendiente,Unidad no especificada en la documentación,Registro de trazabilidad,Contexto; uso predictivo sujeto a disponibilidad,No aplica: variable original.,Representación original; sin recodificación,Dominio SQL integer; no equivale a un dominio empresarial validado.,Respetar tipo SQL y nulabilidad. Las restricciones declaradas aparecen en la sección de llaves y restricciones.,NULL: valor ausente. Cero y códigos esp

timbradas_consolidado: perfil 1/1 archivos


timbradas_segmentos: perfil 20/80 archivos
timbradas_segmentos: perfil 40/80 archivos
timbradas_segmentos: perfil 60/80 archivos
timbradas_segmentos: perfil 80/80 archivos


vehiculos: perfil 1/1 archivos


## **5. Auditoría de calidad inicial**

En esta sección verificamos si los archivos publicados corresponden al alcance de extracción, documentar su cobertura temporal y separar problemas técnicos de fuentes vacías dentro del filtro o diferidas intencionalmente. Queremos comprobar qué información tenemos y si la extracción puede utilizarse antes de evaluar tipos, nulos, duplicados y coherencia operativa.


Se comparan filas registradas por el extractor, filas físicas de los Parquet y filas en PostgreSQL con exactamente el mismo filtro. 

Nota: La consulta a la base usa una instantánea nueva por tabla. Una diferencia puede deberse a cambios posteriores a la extracción; no certifica retrospectivamente aquella instantánea. Coincidir en conteos tampoco demuestra identidad de registros ni ausencia de duplicados.

**Consideraciones**
- Mantener filtros y parámetros originales. No excluir cancelaciones, ceros, faltantes ni atípicos.
- Para progvehiculos, fechasalida representa programación, no salida real ni demanda por franja.
- Para timbradas_segmentos, fecha representa cobertura de sus registros; no asumir equivalencia con viajes.
- Describir las demás columnas SQL de tipo fecha/timestamp, sin convertir fechas de actualización en fechas operativas.
- Conservar los años declarados de timbradas_anio sin recortarlos al periodo de viajes.
- No consultar los sensores diferidos.
- Los días sin registros no implican demanda cero, cancelación ni falla de sensor confirmada.

**Productos.** Conciliación por tabla; inventario de partes; perfil de fechas; cobertura mensual comparada con la fuente; calendario diario; tramos sin registros; años declarados; incidencias y reporte HTML verde. Las consultas exactas tienen el timeout configurado: si no terminan, quedan pendientes.


### **5.1. Verificación de la extracción**
#### **5.1.1. Preparación**

Usamos las carpetas RAW, el registro del extractor y el plan ya definidos. Creamos una carpeta independiente para esta auditoría. Las funciones auxiliares solo organizan la presentación y guardan el detalle; no alteran los datos.

In [ ]:
from collections import Counter
from html import escape
from sqlalchemy import text
from IPython.display import display, HTML

necesarios = ["CFG", "BASE", "RAW", "REGISTRO", "PLAN_EXTRACCION",
              "ESQUEMA", "CAMPOS", "seleccion", "qtable", "lectura"]
faltan = [nombre for nombre in necesarios if nombre not in globals()]
if faltan:
    raise RuntimeError("Ejecuta primero extracción y fichas: " + ", ".join(faltan))
if any(not isinstance(carpeta, (str, Path)) for carpeta in RAW.values()):
    raise TypeError("RAW debe contener carpetas Parquet.")
if PLAN_EXTRACCION["tabla"].duplicated().any():
    raise ValueError("El plan tiene tablas repetidas.")

AUD_ID = datetime.now(ZoneInfo(CFG["zona"])).strftime("%Y%m%dT%H%M%S_%f")
AUD_DIR = Path(BASE) / "reportes" / ("auditoria_cobertura_" + AUD_ID)
AUD_DIR.mkdir(parents=True, exist_ok=False)
(AUD_DIR / "sql").mkdir()
plan_aud = PLAN_EXTRACCION.set_index("tabla")
diferidas_aud = set(globals().get("TABLAS_DIFERIDAS", {"conteo_pasajeros_destino"}))
reg_aud = {}
for entrada in REGISTRO:
    if entrada.get("estado") in {"extraida", "vacia_en_filtro", "fallida", "no_disponible"}:
        reg_aud[entrada["fuente"]] = entrada

def guardar_aud(nombre, df):
    df.to_csv(AUD_DIR / (nombre + ".csv"), index=False, encoding="utf-8-sig")

def mensaje_aud(texto):
    display(HTML("<div style='padding:12px;border-left:4px solid #238B45;"
                 "background:#eef7ef;color:#166534;margin:10px 0'>" +
                 escape(texto) + "</div>"))

def mostrar_aud(df, n=None):
    vista = df if n is None else df.head(n)
    estilo = "<style>.tabla-aud th{background:#166534;color:white;padding:8px}" + \
             ".tabla-aud td{padding:7px;border-bottom:1px solid #d7e8da}" + \
             ".tabla-aud{border-collapse:collapse}</style>"
    display(HTML(estilo + vista.to_html(index=False, escape=True, classes="tabla-aud", na_rep="Pendiente")))

print("Preparación lista. Los archivos RAW se conservan.")

Preparación lista. Los archivos RAW se conservan.


#### **5.1.2. ¿Los archivos coinciden con el registro de extracción?**

Verificamos todas las fuentes previstas: existencia de carpetas, número de partes, filas físicas y columnas. Mostramos cinco filas como vista previa, pero el diagnóstico se calcula sobre **todas las fuentes. Cualquier diferencia aparece aparte.

**Notas:** Una fuente diferida no es un error. Una fuente vacía debe tener cero filas y cero partes conforme al registro. Esta revisión usa metadatos Parquet.

In [76]:
control, detalle_partes = [], []
for tabla in sorted(set(plan_aud.index) | set(RAW)):
    reg = reg_aud.get(tabla, {})
    fila = {"Tabla": tabla, "Filas registradas": reg.get("filas"),
            "Filas en archivos": None, "Estado": "Pendiente", "Detalle": ""}
    if tabla not in RAW:
        fila["Estado"] = "Diferida" if tabla in diferidas_aud else "Revisar"
        fila["Detalle"] = "Descarga aplazada" if tabla in diferidas_aud else "Fuente no extraída"
        control.append(fila)
        continue
    carpeta = Path(RAW[tabla])
    if not carpeta.is_dir():
        fila.update(Estado="Revisar", Detalle="Carpeta inexistente")
        control.append(fila)
        continue
    partes = sorted(carpeta.glob("parte-*.parquet"))
    total, errores = 0, []
    if any(p not in partes for p in carpeta.iterdir()):
        errores.append("Archivos adicionales en la carpeta")
    for archivo in partes:
        try:
            pf = pq.ParquetFile(archivo)
            n = pf.metadata.num_rows
            total += n
            esquema_ok = set(pf.schema_arrow.names) == set(CAMPOS.get(tabla, []))
            if not esquema_ok:
                errores.append("Columnas distintas: " + archivo.name)
            detalle_partes.append({"Tabla": tabla, "Archivo": archivo.name, "Filas": n,
                                   "Columnas coinciden": esquema_ok})
        except Exception as exc:
            errores.append(archivo.name + ": " + type(exc).__name__)
    if reg.get("filas") is None or reg.get("partes") is None:
        errores.append("Sin registro de extracción completo")
    else:
        if total != int(reg["filas"]):
            errores.append("El número de filas no coincide")
        if len(partes) != int(reg["partes"]):
            errores.append("El número de partes no coincide")
    fila.update({"Filas en archivos": total, "Estado": "Revisar" if errores else "Verificada",
                 "Detalle": "; ".join(errores)})
    control.append(fila)

CONTROL_EXTRACCION = pd.DataFrame(control)
guardar_aud("control_extraccion", CONTROL_EXTRACCION)
guardar_aud("detalle_partes", pd.DataFrame(detalle_partes))
resumen = CONTROL_EXTRACCION["Estado"].value_counts()
mostrar_aud(CONTROL_EXTRACCION[["Tabla", "Filas registradas", "Filas en archivos", "Estado"]], n=29)
pendientes_locales = CONTROL_EXTRACCION.query("Estado == 'Revisar'")
if not pendientes_locales.empty:
    mostrar_aud(pendientes_locales)

Tabla,Filas registradas,Filas en archivos,Estado
Procesos,1.0,1.0,Verificada
conteo_pasajeros_destino,Pendiente,Pendiente,Diferida
dias_semana,7.0,7.0,Verificada
estacionalidades,24.0,24.0,Verificada
estacionalidades_anio,0.0,0.0,Verificada
geography_columns,0.0,0.0,Verificada
geometry_columns,4.0,4.0,Verificada
novedades_viales,59.0,59.0,Verificada
novedades_viales_fecha,0.0,0.0,Verificada
novedades_viales_ruta,0.0,0.0,Verificada


#### **5.1.3. ¿Coinciden las filas con la fuente consultada?**

El conteo usa el mismo filtro y los parámetros originales. Primero comprobamos que el plan reproduce el SQL guardado por el extractor. 

* Se cuentan solo las fuentes extraídas y verificadas localmente. Los sensores diferidos no se consultan.

In [79]:
COMPARAR_FUENTE = True  # False permite dejar esta comprobación pendiente.
conciliacion, consultas_aud = [], []
for fila in control:
    tabla = fila["Tabla"]
    salida = {"Tabla": tabla, "Filas en archivos": fila["Filas en archivos"],
              "Filas en fuente actual": None, "Diferencia": None, "Resultado": "Pendiente"}
    if fila["Estado"] == "Diferida":
        salida["Resultado"] = "Diferida"
    elif fila["Estado"] != "Verificada":
        salida["Resultado"] = "Resolver revisión local"
    elif not COMPARAR_FUENTE:
        salida["Resultado"] = "Consulta desactivada"
    else:
        reg = reg_aud[tabla]
        try:
            if tabla not in plan_aud.index or reg.get("parametros") is None:
                raise ValueError("Faltan plan o parámetros originales")
            filtro = plan_aud.loc[tabla, "filtro_sql"]
            sql_original = (Path(BASE) / "sql" / (tabla + ".sql")).read_text(encoding="utf-8")
            reproducido = f"SELECT {seleccion(tabla, CAMPOS[tabla])} FROM {qtable(tabla)} WHERE {filtro}"
            if sql_original != reproducido or hashlib.sha256(sql_original.encode()).hexdigest() != reg.get("sql_sha256"):
                raise ValueError("El plan no reproduce el SQL original")
            sql = f"SELECT COUNT(*) FROM {qtable(tabla)} WHERE {filtro}"
            nombre_sql = hashlib.sha256(tabla.encode()).hexdigest()[:12] + ".sql"
            (AUD_DIR / "sql" / nombre_sql).write_text(sql, encoding="utf-8")
            consultas_aud.append({"tabla": tabla, "sql": nombre_sql, "parametros": reg["parametros"]})
            print("Comparando fuente:", tabla, flush=True)
            with lectura() as conn:
                n = int(conn.execute(text(sql), reg["parametros"]).scalar_one())
            diferencia = int(fila["Filas en archivos"]) - n
            salida.update({"Filas en fuente actual": n, "Diferencia": diferencia,
                           "Resultado": "Coincide la cantidad" if diferencia == 0 else "Revisar diferencia"})
        except Exception as exc:
            salida["Resultado"] = "Consulta pendiente"
            consultas_aud.append({"tabla": tabla, "error_tipo": type(exc).__name__,
                                 "motivo": str(exc) if isinstance(exc, ValueError) else
                                           "Revisar error en el entorno local"})
    conciliacion.append(salida)

CONCILIACION = pd.DataFrame(conciliacion)
guardar_aud("conciliacion_fuente", CONCILIACION)
(AUD_DIR / "consultas_y_parametros.json").write_text(
    json.dumps(consultas_aud, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
n_coinciden = int(CONCILIACION["Resultado"].eq("Coincide la cantidad").sum())
mensaje_aud(f"{n_coinciden} fuentes coinciden en cantidad con la fuente actual. "
            "La comparación detallada quedó guardada.")
mostrar_aud(CONCILIACION, n=29)
revisar_fuente = CONCILIACION.loc[~CONCILIACION["Resultado"].isin(["Coincide la cantidad", "Diferida"])]
if not revisar_fuente.empty:
    mostrar_aud(revisar_fuente)

Comparando fuente: Procesos
Comparando fuente: dias_semana
Comparando fuente: estacionalidades
Comparando fuente: estacionalidades_anio
Comparando fuente: geography_columns
Comparando fuente: geometry_columns
Comparando fuente: novedades_viales
Comparando fuente: novedades_viales_fecha
Comparando fuente: novedades_viales_ruta
Comparando fuente: progvehiculos
Comparando fuente: raster_columns
Comparando fuente: raster_overviews
Comparando fuente: rutas
Comparando fuente: rutas_segmentos
Comparando fuente: rutas_segmentos_config
Comparando fuente: rutas_segmentos_porcion
Comparando fuente: spatial_ref_sys
Comparando fuente: timbradas_anio
Comparando fuente: timbradas_anio_trazabilidad
Comparando fuente: timbradas_anio_zonas
Comparando fuente: timbradas_anio_zonas_trazabilidad
Comparando fuente: timbradas_consolidado
Comparando fuente: timbradas_consolidado_zonas
Comparando fuente: timbradas_segmentos
Comparando fuente: timbradas_segmentos_anio
Comparando fuente: timbradas_segmentos_anio_

Tabla,Filas en archivos,Filas en fuente actual,Diferencia,Resultado
Procesos,1.0,1.0,0.0,Coincide la cantidad
conteo_pasajeros_destino,Pendiente,Pendiente,Pendiente,Diferida
dias_semana,7.0,7.0,0.0,Coincide la cantidad
estacionalidades,24.0,24.0,0.0,Coincide la cantidad
estacionalidades_anio,0.0,0.0,0.0,Coincide la cantidad
geography_columns,0.0,0.0,0.0,Coincide la cantidad
geometry_columns,4.0,4.0,0.0,Coincide la cantidad
novedades_viales,59.0,59.0,0.0,Coincide la cantidad
novedades_viales_fecha,0.0,0.0,0.0,Coincide la cantidad
novedades_viales_ruta,0.0,0.0,0.0,Coincide la cantidad


### **5.2. Cobertura temporal de las fuentes operativas**
#### **5.2.1. ¿Qué fechas tienen registros?**


Ahora revisamos las fechas útiles para describir disponibilidad operativa. No mezclamos fechas de actualización o trazabilidad.

| Fuente | Campo | Qué describe |
|---|---|---|
| progvehiculos | fechasalida | Fecha de salida **programada** de los viajes extraídos. |
| timbradas_segmentos | fecha | Fecha de los registros de segmentos extraídos. |

Para cada fuente mostramos inicio, fin, días con registros y días sin registros en el periodo solicitado. Los días con registros no garantizan cobertura completa de todas las horas. No se interpreta ninguna cantidad como pasajeros por día.

In [80]:
FECHAS_COBERTURA = {"progvehiculos": "fechasalida", "timbradas_segmentos": "fecha"}
inicio = pd.Timestamp(CFG["inicio"])
fin = pd.Timestamp(CFG["fin_exclusivo"])
if inicio != inicio.normalize() or fin != fin.normalize() or fin <= inicio:
    raise ValueError("Este resumen diario requiere límites a medianoche y un periodo válido.")
calendario = pd.date_range(inicio, fin, inclusive="left", freq="D")
estado_local = CONTROL_EXTRACCION.set_index("Tabla")["Estado"].to_dict()
conteos_diarios, cobertura, incidencias_fechas = {}, [], []

for tabla, columna in FECHAS_COBERTURA.items():
    fila = {"Fuente": tabla, "Fecha utilizada": columna, "Primera fecha": None, "Última fecha": None,
            "Días con registros": None, "Días sin registros": None, "Resultado": "Pendiente"}
    if estado_local.get(tabla) != "Verificada":
        fila["Resultado"] = "Fuente no disponible o pendiente de revisión"
        cobertura.append(fila)
        continue
    meta = ESQUEMA.loc[ESQUEMA["tabla"].eq(tabla) & ESQUEMA["columna"].eq(columna)]
    if meta.empty or meta.iloc[0]["udt_name"] not in {"date", "timestamp", "timestamptz"}:
        fila["Resultado"] = "Revisar campo temporal en el esquema"
        cobertura.append(fila)
        continue
    tipo = meta.iloc[0]["udt_name"]
    dias = Counter()
    n_leidas, n_nulas, n_invalidas, n_fuera = 0, 0, 0, 0
    primera, ultima = None, None
    try:
        for archivo in sorted(Path(RAW[tabla]).glob("parte-*.parquet")):
            for batch in pq.ParquetFile(archivo).iter_batches(
                    batch_size=CFG.get("lote", 10000), columns=[columna]):
                s_original = batch.to_pandas()[columna]
                if tipo == "timestamptz":
                    s = pd.to_datetime(s_original, errors="coerce", utc=True).dt.tz_convert(
                        CFG["zona"]).dt.tz_localize(None)
                else:
                    s = pd.to_datetime(s_original, errors="coerce")
                n_leidas += len(s)
                n_nulas += int(s_original.isna().sum())
                n_invalidas += int((s_original.notna() & s.isna()).sum())
                validas = s.dropna()
                n_fuera += int(((validas < inicio) | (validas >= fin)).sum())
                if not validas.empty:
                    primera = validas.min() if primera is None else min(primera, validas.min())
                    ultima = validas.max() if ultima is None else max(ultima, validas.max())
                dentro = validas[(validas >= inicio) & (validas < fin)]
                dias.update(dentro.dt.strftime("%Y-%m-%d").value_counts().to_dict())
        esperadas = int(CONTROL_EXTRACCION.set_index("Tabla").loc[tabla, "Filas en archivos"])
        if n_leidas != esperadas:
            raise ValueError("La lectura temporal no cubrió todas las filas")
        incidencias_fechas.append({"Fuente": tabla, "Filas leídas": n_leidas, "Fechas nulas": n_nulas,
                                  "Fechas no interpretables": n_invalidas, "Fuera del periodo": n_fuera})
        conteos_diarios[tabla] = pd.Series(
            [dias.get(d.strftime("%Y-%m-%d"), 0) for d in calendario], index=calendario, dtype="int64")
        fila.update({"Primera fecha": primera, "Última fecha": ultima,
                     "Días con registros": int((conteos_diarios[tabla] > 0).sum()),
                     "Días sin registros": int((conteos_diarios[tabla] == 0).sum()),
                     "Resultado": "Revisar fechas" if n_nulas+n_invalidas+n_fuera else
                                  "Sin registros en el filtro" if n_leidas == 0 else "Cobertura calculada"})
    except Exception as exc:
        fila["Resultado"] = "Lectura pendiente: " + type(exc).__name__
    cobertura.append(fila)

COBERTURA_OPERATIVA = pd.DataFrame(cobertura)
guardar_aud("cobertura_operativa", COBERTURA_OPERATIVA)
guardar_aud("control_fechas_operativas", pd.DataFrame(incidencias_fechas))
mostrar_aud(COBERTURA_OPERATIVA)
mensaje_aud("Los días sin registros son ausencia de observaciones dentro del periodo solicitado. "
            "Su causa debe contrastarse con la operación y la disponibilidad de cada fuente.")
fechas_a_revisar = [r for r in incidencias_fechas
                   if r["Fechas nulas"]+r["Fechas no interpretables"]+r["Fuera del periodo"] > 0]
if fechas_a_revisar:
    mostrar_aud(pd.DataFrame(fechas_a_revisar))

Fuente,Fecha utilizada,Primera fecha,Última fecha,Días con registros,Días sin registros,Resultado
progvehiculos,fechasalida,2023-01-01 05:00:00,2025-12-31 19:00:00,1096,0,Cobertura calculada
timbradas_segmentos,fecha,2023-01-01 00:00:00,2023-02-27 00:00:00,56,1040,Cobertura calculada


#### **5.2.2 Cobertura diaria de programaciones y registros de segmentos | ¿Cuáles son los períodos sin registros?**

Distinguimos periodos anteriores al primer registro, interrupciones entre registros y periodos posteriores al último. Esto permite reconocer una fuente de cobertura corta sin confundir todos sus días ausentes con fallas internas.

Mostramos hasta cinco tramos por fuente, priorizando los más largos. Los CSV contienen todos los tramos, el calendario diario y un resumen mensual de cobertura. No se imputan valores.

In [81]:
tramos, detalle_diario, cobertura_mensual = [], [], []
for tabla, serie in conteos_diarios.items():
    for fecha, n in serie.items():
        detalle_diario.append({"Fuente": tabla, "Fecha": fecha.date().isoformat(), "Registros": int(n)})
    for mes, grupo in serie.groupby(serie.index.strftime("%Y-%m")):
        cobertura_mensual.append({"Fuente": tabla, "Mes": mes, "Días evaluados": len(grupo),
                                  "Días con registros": int(grupo.gt(0).sum()),
                                  "Días sin registros": int(grupo.eq(0).sum())})
    con_datos = serie[serie.gt(0)]
    grupos = serie.eq(0).ne(serie.eq(0).shift()).cumsum()
    for _, grupo in serie.groupby(grupos):
        if not grupo.eq(0).all():
            continue
        desde, hasta = grupo.index.min(), grupo.index.max()
        ubicacion = ("Sin registros en todo el periodo" if con_datos.empty else
                     "Antes del primer registro" if hasta < con_datos.index.min() else
                     "Después del último registro" if desde > con_datos.index.max() else
                     "Entre fechas con registros")
        tramos.append({"Fuente": tabla, "Desde": desde.date().isoformat(), "Hasta": hasta.date().isoformat(),
                       "Días": len(grupo), "Ubicación": ubicacion})

TRAMOS_SIN_REGISTROS = pd.DataFrame(tramos, columns=["Fuente", "Desde", "Hasta", "Días", "Ubicación"])
guardar_aud("periodos_sin_registros", TRAMOS_SIN_REGISTROS)
guardar_aud("calendario_diario", pd.DataFrame(detalle_diario, columns=["Fuente", "Fecha", "Registros"]))
guardar_aud("cobertura_mensual", pd.DataFrame(cobertura_mensual))
if TRAMOS_SIN_REGISTROS.empty:
    mensaje_aud("No se encontraron tramos sin registros en las fuentes cuya cobertura pudo calcularse.")
else:
    for tabla, grupo in TRAMOS_SIN_REGISTROS.groupby("Fuente"):
        mensaje_aud(f"{tabla}: {len(grupo)} tramos sin registros. Se muestran hasta cinco, "
                    "ordenados por duración; el detalle completo está en el CSV.")
        mostrar_aud(grupo.sort_values(["Días", "Desde"], ascending=[False, True]), n=5)

Fuente,Desde,Hasta,Días,Ubicación
timbradas_segmentos,2023-02-28,2025-12-31,1038,Después del último registro
timbradas_segmentos,2023-01-31,2023-02-01,2,Entre fechas con registros


### **5.3. Diferencias de cobertura entre fuentes**
#### **5.3.1. Años presentes en los agregados históricos**

Revisamos el campo anio de timbradas_anio tal como fue extraído, sin limitarlo a 2023–2025. Mostramos los años declarados y si pertenecen a los años incluidos en el periodo solicitado.

Un año presente en esta tabla no certifica que tenga todos los meses o días. Su granularidad no permite comparar directamente sus filas con viajes ni con registros de segmentos.

In [83]:
anio_tabla = "timbradas_anio"
ANIOS_AGREGADOS = pd.DataFrame(columns=["Fuente", "Año declarado", "Filas", "Relación con el periodo"])
anio_estado = "Pendiente"
if estado_local.get(anio_tabla) == "Verificada" and "ano" in CAMPOS.get(anio_tabla, []):
    try:
        cuentas = Counter()
        for archivo in sorted(Path(RAW[anio_tabla]).glob("parte-*.parquet")):
            for batch in pq.ParquetFile(archivo).iter_batches(batch_size=CFG.get("lote", 10000), columns=["ano"]):
                cuentas.update(batch.to_pandas()["ano"].astype("string").fillna("SIN_ANIO").value_counts().to_dict())
        anios_periodo = set(calendario.year)
        filas_anio = []
        for valor, n in sorted(cuentas.items()):
            try:
                numero = float(valor)
                if not numero.is_integer():
                    raise ValueError()
                relacion = "Dentro de los años del periodo" if int(numero) in anios_periodo else "Fuera de los años del periodo"
            except (ValueError, OverflowError):
                relacion = "Revisar año declarado"
            filas_anio.append({"Fuente": anio_tabla, "Año declarado": valor, "Filas": int(n),
                               "Relación con el periodo": relacion})
        ANIOS_AGREGADOS = pd.DataFrame(filas_anio, columns=ANIOS_AGREGADOS.columns)
        anio_estado = "Calculado"
    except Exception as exc:
        anio_estado = "Lectura pendiente: " + type(exc).__name__
guardar_aud("anios_agregados", ANIOS_AGREGADOS)
if not ANIOS_AGREGADOS.empty:
    mostrar_aud(ANIOS_AGREGADOS)
else:
    mensaje_aud("Años de timbradas_anio: " + ("sin registros" if anio_estado == "Calculado"
                                            else "pendiente; revisar disponibilidad y campo anio") + ".")

Fuente,Año declarado,Filas,Relación con el periodo
timbradas_anio,2023,2502,Dentro de los años del periodo
timbradas_anio,2024,2517,Dentro de los años del periodo
timbradas_anio,2025,2666,Dentro de los años del periodo
timbradas_anio,2026,2689,Fuera de los años del periodo


#### **5.3.2. ¿Qué días tienen registros en ambas fuentes operativas?**

La coincidencia temporal se calcula con los días observados, no solo con mínimos y máximos. Si viajes y segmentos comparten un día, eso únicamente significa que ambas fuentes tienen algún registro ese día.

No demuestra cobertura horaria completa, identidad de viajes, correspondencia por segmento ni equivalencia entre conteos. Es una primera condición para plantear una comparación posterior.

In [85]:
dias_comunes = pd.DataFrame(columns=["Fuentes", "Días con registros en ambas", "Primer día común", "Último día común"])
fuentes = ["progvehiculos", "timbradas_segmentos"]
if all(t in conteos_diarios for t in fuentes):
    comunes = conteos_diarios[fuentes[0]].gt(0) & conteos_diarios[fuentes[1]].gt(0)
    fechas_comunes = comunes.index[comunes]
    dias_comunes = pd.DataFrame([{
        "Fuentes": "Viajes programados y registros de segmentos",
        "Días con registros en ambas": len(fechas_comunes),
        "Primer día común": fechas_comunes.min().date().isoformat() if len(fechas_comunes) else None,
        "Último día común": fechas_comunes.max().date().isoformat() if len(fechas_comunes) else None}])
    mostrar_aud(dias_comunes)
    mensaje_aud("Nota: Compartir fechas no vuelve equivalentes sus registros.")
else:
    mensaje_aud("Comparación temporal pendiente: falta cobertura verificable de una fuente operativa.")
guardar_aud("dias_comunes_fuentes", dias_comunes)

Fuentes,Días con registros en ambas,Primer día común,Último día común
Viajes programados y registros de segmentos,56,2023-01-01,2023-02-27


La verificación mostró coincidencia entre las filas registradas por el extractor, las almacenadas en los archivos y los conteos actuales de PostgreSQL bajo los mismos filtros. Esto respalda la consistencia de las cantidades extraídas, aunque no demuestra identidad de registros, ausencia de duplicados ni calidad de sus valores. Las fuentes vacías coinciden igualmente con los conteos del alcance consultado; por tanto, su ausencia no debe interpretarse como inexistencia de eventos o demanda cero. La fuente de sensores permanece diferida. Además, las salidas aún incluyen vistas técnicas de PostGIS, que deben excluirse del inventario analítico antes del cierre definitivo.

La cobertura temporal presenta diferencias relevantes: las 162.678 programaciones de SILENCIO tienen registros en los 1.096 días de 2023–2025, mientras que los 799.030 registros de segmentos se concentran en 56 días entre el 1 de enero y el 27 de febrero de 2023, con una interrupción el 31 de enero y el 1 de febrero. Estos 56 días constituyen la cobertura diaria compartida, sin garantizar correspondencia entre registros ni cobertura horaria completa. Por su parte, los agregados anuales contienen 10.374 filas con años declarados de 2023 a 2026; estas cantidades no representan pasajeros ni certifican años completos. En consecuencia, se conservará el histórico de programaciones y se limitarán las comparaciones con segmentos a su cobertura disponible, manteniendo los agregados separados hasta validar su granularidad y procedencia. La siguiente etapa comprobará tipos, faltantes, duplicados, llaves y relaciones antes de definir limpieza o integración.

## **6. Calidad estructural**

Esta sección revisa columnas, tipos de almacenamiento, valores ausentes, textos vacíos, filas repetidas y llaves. No modifica RAW, elimina filas, imputa ni corrige valores. Los resultados se calculan al ejecutar; no se anticipa que haya nulos o duplicados.


In [87]:
from datetime import datetime, date, time, timedelta
from decimal import Decimal
import math
import sqlite3
import pyarrow as pa
import pyarrow.compute as pc

necesarios = ["CFG", "BASE", "RAW", "REGISTRO", "ESQUEMA", "CAMPOS", "LLAVES"]
faltan = [n for n in necesarios if n not in globals()]
if faltan:
    raise RuntimeError("Ejecuta extracción y fichas: " + ", ".join(faltan))
if any(not isinstance(p, (str, Path)) for p in RAW.values()):
    raise TypeError("RAW debe contener carpetas Parquet; no ejecutar después de sustituirlo por DataFrames.")

TECNICOS_EST = {"raster_columns", "raster_overviews", "geometry_columns", "geography_columns"}
DIFERIDOS_EST = set(globals().get("TABLAS_DIFERIDAS", {"conteo_pasajeros_destino"}))
RAW_EST = {t: Path(p) for t, p in RAW.items() if t not in TECNICOS_EST}
# No modificar ESQUEMA, CAMPOS, RAW ni REGISTRO.
META_EST = ESQUEMA.loc[~ESQUEMA["tabla"].isin(TECNICOS_EST)].copy()
LLAVES_EST = LLAVES.loc[~LLAVES["tabla"].isin(TECNICOS_EST)].copy()
REG_EST = {}
for r in REGISTRO:
    if r.get("estado") in {"extraida", "vacia_en_filtro", "fallida", "no_disponible"}:
        REG_EST[r["fuente"]] = r
TABLAS_EST = sorted((set(CAMPOS) | set(RAW_EST)) - TECNICOS_EST)
EST_ID = datetime.now(ZoneInfo(CFG["zona"])).strftime("%Y%m%dT%H%M%S_%f")
EST_DIR = Path(BASE) / "reportes" / ("calidad_estructural_" + EST_ID)
EST_DIR.mkdir(parents=True, exist_ok=False)
(EST_DIR / "auxiliares").mkdir()
LOTE_EST = min(int(CFG.get("lote", 10000)), 10000)
# Ejecuta duplicados exactos y llaves con comparación completa en disco.
# Si se desactiva por presupuesto, el cierre indicará explícitamente el pendiente.
EVALUAR_REPETICIONES = True

def guardar_est(nombre, df):
    df.to_csv(EST_DIR / (nombre + ".csv"), index=False, encoding="utf-8-sig")

def nota_est(s):
    display(HTML("<div style='padding:12px;margin:10px 0;background:#eef7ef;"
                 "color:#166534;border-left:4px solid #238B45'>" + escape(s) + "</div>"))

def mostrar_est(df, n=None):
    vista = df if n is None else df.head(n)
    display(HTML("<style>.est-tabla th{background:#166534;color:white;padding:8px}"
                 ".est-tabla td{padding:7px;border-bottom:1px solid #d7e8da}"
                 ".est-tabla{border-collapse:collapse}</style>" +
                 vista.to_html(index=False, classes="est-tabla", escape=True, na_rep="No evaluado")))

nota_est("Preparación lista. Los cuatro objetos técnicos se excluyen; sensores diferidos y RAW intacto.")

### **6.1. Verificación del tipo de columna**

Comparamos las columnas de cada parte Parquet con el esquema SQL observado. Verificamos también si el tipo de archivo es compatible con la familia de tipo SQL y si una columna cambia de representación entre partes.

**Nota:** SQL integer y Arrow int64 pueden ser compatibles. Las geometrías/geografías fueron exportadas deliberadamente como texto; no se interpretan como pérdida de estructura. Un tipo Arrow null puede aparecer en una parte donde todos los valores son nulos: no lo calificamos por sí solo como error de tipo.

In [88]:
def compatibilidad_est(tipo_sql, tipo_arrow):
    t = tipo_arrow
    if pa.types.is_dictionary(t):
        t = t.value_type
    if pa.types.is_null(t):
        return "Parte sin valores", "Sin evidencia para verificar el tipo"
    texto = pa.types.is_string(t) or pa.types.is_large_string(t)
    if tipo_sql in {"geometry", "geography"}:
        return ("Compatible", "Exportación deliberada como texto") if texto else ("Revisar", "Representación espacial inesperada")
    reglas = {
        "int2": pa.types.is_integer(t), "int4": pa.types.is_integer(t), "int8": pa.types.is_integer(t),
        "float4": pa.types.is_floating(t), "float8": pa.types.is_floating(t),
        "numeric": pa.types.is_decimal(t) or pa.types.is_integer(t),
        "bool": pa.types.is_boolean(t),
        "date": pa.types.is_date(t),
        "timestamp": pa.types.is_timestamp(t) and t.tz is None,
        "timestamptz": pa.types.is_timestamp(t) and t.tz is not None,
        "time": pa.types.is_time(t), "timetz": False,
        "interval": pa.types.is_duration(t) or pa.types.is_interval(t),
        "text": texto, "varchar": texto, "bpchar": texto,
        "uuid": texto or pa.types.is_binary(t) or pa.types.is_fixed_size_binary(t),
        "bytea": pa.types.is_binary(t) or pa.types.is_large_binary(t),
        "json": texto or pa.types.is_struct(t) or pa.types.is_list(t),
        "jsonb": texto or pa.types.is_struct(t) or pa.types.is_list(t),
    }
    if tipo_sql in reglas:
        return ("Compatible", "Familia de tipo compatible") if reglas[tipo_sql] else ("Revisar", "Familia de tipo distinta")
    if str(tipo_sql).startswith("_"):
        return ("Compatible", "Tipo SQL de arreglo") if (
            pa.types.is_list(t) or pa.types.is_large_list(t) or pa.types.is_fixed_size_list(t)) else (
            "Revisar", "Arreglo con representación distinta")
    return "Revisión manual", "Tipo SQL no cubierto por la comparación automática"

estructura, tipos_est, incidencias_est = [], [], []
for tabla in TABLAS_EST:
    reg = REG_EST.get(tabla, {})
    fila = {"Tabla": tabla, "Filas": None, "Partes": None, "Estado": "Pendiente", "Detalle": ""}
    if tabla not in RAW_EST:
        fila.update(Estado="Diferida" if tabla in DIFERIDOS_EST else "No extraída",
                    Detalle="Sin lectura de contenido")
        estructura.append(fila)
        continue
    carpeta = RAW_EST[tabla]
    if not carpeta.is_dir():
        fila.update(Estado="Revisar estructura", Detalle="Carpeta inexistente")
        estructura.append(fila)
        continue
    archivos = sorted(carpeta.glob("parte-*.parquet"))
    esperadas = set(CAMPOS.get(tabla, []))
    meta = META_EST.loc[META_EST["tabla"].eq(tabla)].set_index("columna")
    total, errores, variaciones = 0, [], {}
    if not esperadas or meta.empty:
        errores.append("Sin esquema de columnas validable")
    if any(p not in archivos for p in carpeta.iterdir()):
        errores.append("Archivos no previstos")
    for archivo in archivos:
        try:
            pf = pq.ParquetFile(archivo)
            total += pf.metadata.num_rows
            reales = set(pf.schema_arrow.names)
            if reales != esperadas:
                errores.append("Columnas distintas en " + archivo.name)
                incidencias_est.append({"Tabla": tabla, "Comprobación": "Columnas",
                    "Detalle": json.dumps({"archivo": archivo.name, "faltan": sorted(esperadas-reales),
                                          "sobran": sorted(reales-esperadas)}, ensure_ascii=False)})
            for campo in pf.schema_arrow:
                if campo.name not in meta.index:
                    continue
                sql_tipo = str(meta.loc[campo.name, "udt_name"])
                estado, detalle = compatibilidad_est(sql_tipo, campo.type)
                tipos_est.append({"Tabla": tabla, "Columna": campo.name, "Archivo": archivo.name,
                                  "Tipo SQL": sql_tipo, "Tipo Parquet": str(campo.type),
                                  "Estado": estado, "Detalle": detalle})
                if not pa.types.is_null(campo.type):
                    variaciones.setdefault(campo.name, set()).add(str(campo.type))
        except Exception as exc:
            errores.append(archivo.name + ": " + type(exc).__name__)
    if reg.get("filas") is None or reg.get("partes") is None:
        errores.append("Sin registro completo")
    else:
        if total != int(reg["filas"]) or len(archivos) != int(reg["partes"]):
            errores.append("Filas o partes no coinciden con el registro")
    for columna, tipos in variaciones.items():
        if len(tipos) > 1:
            errores.append("Tipo cambia entre partes: " + columna)
    fila.update(Filas=total, Partes=len(archivos),
                Estado="Revisar estructura" if errores else "Vacía en el filtro" if total == 0 else "Lista para perfilado",
                Detalle="; ".join(errores))
    estructura.append(fila)

ESTRUCTURA = pd.DataFrame(estructura)
TIPOS_ESTRUCTURALES = pd.DataFrame(tipos_est, columns=[
    "Tabla", "Columna", "Archivo", "Tipo SQL", "Tipo Parquet", "Estado", "Detalle"])
guardar_est("estructura_archivos", ESTRUCTURA)
guardar_est("tipos_por_parte", TIPOS_ESTRUCTURALES)
mostrar_est(ESTRUCTURA[["Tabla", "Filas", "Estado"]])
estructura_revisar = ESTRUCTURA.query("Estado == 'Revisar estructura'")
if not estructura_revisar.empty:
    mostrar_est(estructura_revisar)
tipos_revisar = TIPOS_ESTRUCTURALES.loc[TIPOS_ESTRUCTURALES["Estado"].isin(["Revisar", "Revisión manual"])]
if not tipos_revisar.empty:
    nota_est("Hay tipos que necesitan interpretación. Se muestran combinaciones únicas; el CSV identifica cada parte.")
    mostrar_est(tipos_revisar.drop(columns="Archivo").drop_duplicates(), n=15)
else:
    nota_est("No se detectaron incompatibilidades en los tipos evaluables. Esto no certifica validez de los valores.")

Tabla,Filas,Estado
Procesos,1.0,Lista para perfilado
conteo_pasajeros_destino,No evaluado,Diferida
dias_semana,7.0,Lista para perfilado
estacionalidades,24.0,Lista para perfilado
estacionalidades_anio,0.0,Vacía en el filtro
novedades_viales,59.0,Lista para perfilado
novedades_viales_fecha,0.0,Vacía en el filtro
novedades_viales_ruta,0.0,Vacía en el filtro
progvehiculos,162678.0,Lista para perfilado
rutas,1.0,Lista para perfilado


Tabla,Columna,Tipo SQL,Tipo Parquet,Estado,Detalle
Procesos,recaudo_comision_conductor,numeric,string,Revisar,Familia de tipo distinta
Procesos,fecha_inicio_liquidacion,date,string,Revisar,Familia de tipo distinta
Procesos,latitud,numeric,string,Revisar,Familia de tipo distinta
Procesos,longitud,numeric,string,Revisar,Familia de tipo distinta
Procesos,valor_operador_relacional_bloqueo_pico,numeric,string,Revisar,Familia de tipo distinta
Procesos,valor_tiempo_minimo_viaje,int4,string,Revisar,Familia de tipo distinta
Procesos,id_estacionalidad,int8,string,Revisar,Familia de tipo distinta
Procesos,ultima_estacionalidad,date,string,Revisar,Familia de tipo distinta
Procesos,id_diasemana,int4,string,Revisar,Familia de tipo distinta
progvehiculos,fechaliq,timestamp,string,Revisar,Familia de tipo distinta


### **6.2. Información faltante en los registros**

Para cada columna calculamos filas evaluadas, nulos SQL/Arrow, NaN numéricos, cadenas vacías, textos que contienen solo espacios y etiquetas literales de ausencia como NULL, N/A o SIN DATO. Estas últimas son señales candidatas: no se recodifican sin confirmar su significado. Las categorías son separadas y no se suman automáticamente como nulos.

In [90]:
ETIQUETAS_AUSENCIA = {"NULL", "NONE", "NAN", "N/A", "NA", "SIN DATO", "SIN DATOS"}
perfil_est, lecturas_est = [], []
tablas_perfilables = ESTRUCTURA.loc[ESTRUCTURA["Estado"].eq("Lista para perfilado"), "Tabla"].tolist()
for tabla in tablas_perfilables:
    print("Perfilando valores:", tabla, flush=True)
    columnas = list(CAMPOS[tabla])
    acumulado = {c: Counter(nulos=0, nan=0, vacias=0, espacios=0, etiquetas=0) for c in columnas}
    leidas = 0
    try:
        for archivo in sorted(RAW_EST[tabla].glob("parte-*.parquet")):
            for batch in pq.ParquetFile(archivo).iter_batches(batch_size=LOTE_EST, columns=columnas):
                leidas += batch.num_rows
                for c, array in zip(batch.schema.names, batch.columns):
                    if pa.types.is_dictionary(array.type):
                        array = pc.dictionary_decode(array)
                    a = acumulado[c]
                    a["nulos"] += array.null_count
                    if pa.types.is_floating(array.type):
                        a["nan"] += int(pc.sum(pc.fill_null(pc.is_nan(array), False)).as_py() or 0)
                    if pa.types.is_string(array.type) or pa.types.is_large_string(array.type):
                        s = pd.Series(array.to_pylist(), dtype="string")
                        a["vacias"] += int(s.eq("").fillna(False).sum())
                        a["espacios"] += int((s.notna() & s.ne("") & s.str.strip().eq("")).fillna(False).sum())
                        a["etiquetas"] += int(s.str.strip().str.upper().isin(ETIQUETAS_AUSENCIA).sum())
        n_esperadas = int(ESTRUCTURA.set_index("Tabla").loc[tabla, "Filas"])
        if leidas != n_esperadas:
            raise ValueError("Lectura incompleta frente a las filas físicas")
        meta = META_EST.loc[META_EST["tabla"].eq(tabla)].set_index("columna")
        for c, a in acumulado.items():
            not_null = c in meta.index and str(meta.loc[c, "is_nullable"]) == "NO"
            perfil_est.append({"Tabla": tabla, "Columna": c, "Filas evaluadas": leidas,
                "Nulos": a["nulos"], "% nulos": round(100*a["nulos"]/leidas, 2),
                "NaN numéricos": a["nan"], "Cadenas vacías": a["vacias"],
                "Solo espacios": a["espacios"], "Etiquetas candidatas": a["etiquetas"],
                "NOT NULL declarado": not_null, "Nulos incompatibles con NOT NULL": a["nulos"] if not_null else 0})
        lecturas_est.append({"Tabla": tabla, "Filas leídas": leidas, "Estado": "Perfil completo"})
    except Exception as exc:
        lecturas_est.append({"Tabla": tabla, "Filas leídas": leidas, "Estado": "Lectura pendiente"})
        incidencias_est.append({"Tabla": tabla, "Comprobación": "Perfilado",
                                "Detalle": type(exc).__name__ + ": " + str(exc)})

PERFIL_ESTRUCTURAL = pd.DataFrame(perfil_est, columns=[
    "Tabla", "Columna", "Filas evaluadas", "Nulos", "% nulos", "NaN numéricos",
    "Cadenas vacías", "Solo espacios", "Etiquetas candidatas", "NOT NULL declarado",
    "Nulos incompatibles con NOT NULL"])
LECTURAS_ESTRUCTURALES = pd.DataFrame(lecturas_est, columns=["Tabla", "Filas leídas", "Estado"])
guardar_est("perfil_columnas", PERFIL_ESTRUCTURAL)
guardar_est("lecturas_perfilado", LECTURAS_ESTRUCTURALES)
resumen_nulos = []
for tabla, grupo in PERFIL_ESTRUCTURAL.groupby("Tabla"):
    resumen_nulos.append({"Tabla": tabla, "Columnas evaluadas": len(grupo),
        "Columnas con nulos": int(grupo["Nulos"].gt(0).sum()),
        "Columnas totalmente nulas": int(grupo["Nulos"].eq(grupo["Filas evaluadas"]).sum()),
        "Columnas con otras señales de ausencia": int(grupo[
            ["NaN numéricos", "Cadenas vacías", "Solo espacios", "Etiquetas candidatas"]].sum(axis=1).gt(0).sum())})
mostrar_est(pd.DataFrame(resumen_nulos), n=30)
for tabla in sorted(
    PERFIL_ESTRUCTURAL.loc[
        PERFIL_ESTRUCTURAL["Nulos"].gt(0), "Tabla"
    ].unique()
):
    relevantes = PERFIL_ESTRUCTURAL.loc[PERFIL_ESTRUCTURAL["Tabla"].eq(tabla) &
                                      PERFIL_ESTRUCTURAL["Nulos"].gt(0)]
    if not relevantes.empty:
        nota_est(tabla + ": columnas con mayor porcentaje de nulos; detalle completo en CSV.")
        mostrar_est(relevantes.sort_values(["% nulos", "Columna"], ascending=[False, True])[
            ["Columna", "Filas evaluadas", "Nulos", "% nulos"]], n=10)
otras_senales = PERFIL_ESTRUCTURAL.loc[PERFIL_ESTRUCTURAL[
    ["NaN numéricos", "Cadenas vacías", "Solo espacios", "Etiquetas candidatas"]].sum(axis=1).gt(0)]
if not otras_senales.empty:
    nota_est("Señales adicionales de ausencia: revisar significado antes de tratarlas.")
    mostrar_est(otras_senales[["Tabla", "Columna", "NaN numéricos", "Cadenas vacías",
                              "Solo espacios", "Etiquetas candidatas"]], n=15)
violaciones_nulos = PERFIL_ESTRUCTURAL.loc[PERFIL_ESTRUCTURAL["Nulos incompatibles con NOT NULL"].gt(0)]
if not violaciones_nulos.empty:
    mostrar_est(violaciones_nulos[["Tabla", "Columna", "Nulos", "NOT NULL declarado"]])
if not LECTURAS_ESTRUCTURALES.empty:
    pendientes = LECTURAS_ESTRUCTURALES.query("Estado != 'Perfil completo'")
    if not pendientes.empty:
        mostrar_est(pendientes)

Perfilando valores: Procesos


Perfilando valores: dias_semana
Perfilando valores: estacionalidades
Perfilando valores: novedades_viales
Perfilando valores: progvehiculos
Perfilando valores: rutas
Perfilando valores: rutas_segmentos
Perfilando valores: rutas_segmentos_config
Perfilando valores: rutas_segmentos_porcion
Perfilando valores: spatial_ref_sys
Perfilando valores: timbradas_anio
Perfilando valores: timbradas_consolidado
Perfilando valores: timbradas_segmentos
Perfilando valores: vehiculos


Tabla,Columnas evaluadas,Columnas con nulos,Columnas totalmente nulas,Columnas con otras señales de ausencia
Procesos,88,14,14,1
dias_semana,2,0,0,0
estacionalidades,5,0,0,0
novedades_viales,5,0,0,0
progvehiculos,73,46,24,0
rutas,43,6,6,0
rutas_segmentos,9,0,0,0
rutas_segmentos_config,10,1,1,0
rutas_segmentos_porcion,10,0,0,0
spatial_ref_sys,5,0,0,2


Columna,Filas evaluadas,Nulos,% nulos
clave_interfaz_despacho,1,1,100.0
exportaviajes,1,1,100.0
fecha_inicio_liquidacion,1,1,100.0
id_diasemana,1,1,100.0
id_estacionalidad,1,1,100.0
latitud,1,1,100.0
longitud,1,1,100.0
operador_relacional_bloqueo_pico,1,1,100.0
programaciondefault,1,1,100.0
recaudo_comision_conductor,1,1,100.0


Columna,Filas evaluadas,Nulos,% nulos
bloqueo_cond,162678,162678,100.0
bloqueo_despacho_cond_lic,162678,162678,100.0
bloqueo_despacho_cond_rec,162678,162678,100.0
bloqueo_despacho_veh_rec,162678,162678,100.0
bloqueo_despacho_veh_soat,162678,162678,100.0
bloqueo_veh,162678,162678,100.0
data_log,162678,162678,100.0
fecha_despacho_almacenado_movil,162678,162678,100.0
fecha_fin_liquidacion,162678,162678,100.0
fecha_final_manual,162678,162678,100.0


Columna,Filas evaluadas,Nulos,% nulos
colorrutapara,1,1,100.0
geom,1,1,100.0
grupos_vehiculos_cedidos,1,1,100.0
hexcolor,1,1,100.0
id_tipo_tarifa,1,1,100.0
idempresa,1,1,100.0


Columna,Filas evaluadas,Nulos,% nulos
numero_segmentos,1,1,100.0


Columna,Filas evaluadas,Nulos,% nulos
alarma,293,293,100.0
alias,293,293,100.0
alto_apertura_contenedor,293,293,100.0
alto_contenedor,293,293,100.0
alto_techo_contenedor,293,293,100.0
ancho_apertura_contenedor,293,293,100.0
ancho_contenedor,293,293,100.0
ancho_techo_contenedor,293,293,100.0
aseguradora,293,293,100.0
color,293,293,100.0


Tabla,Columna,NaN numéricos,Cadenas vacías,Solo espacios,Etiquetas candidatas
Procesos,tipostrabajos,0,1,0,0
spatial_ref_sys,srtext,0,14,0,0
spatial_ref_sys,proj4text,0,2,0,0
vehiculos,modem_min,0,0,0,2
vehiculos,modem_puk,0,69,0,0
vehiculos,modem_user,0,19,0,0
vehiculos,modem_passwd,0,69,0,0
vehiculos,modem_serie,0,0,0,15
vehiculos,operador_satelital,0,0,0,34


#### **6.2.1. Campos prioritarios para el proyecto**

El ranking de faltantes puede estar dominado por columnas administrativas. Esta vista adicional muestra identificadores, fechas, estados, conteos y capacidades, incluso si no tienen nulos. Solo presenta campos que existen en los metadatos y cuyo perfil se completó; las capacidades cero se evaluarán después.

In [91]:
PRIORITARIOS_EST = {
    "progvehiculos": ["idprogramacion", "idruta", "idvehiculo", "fechasalida", "fechainicio",
                     "fechafin", "finalizado", "cancelada", "numeropasajeros", "numerobajadas",
                     "modificado_manual", "apto_estadisticas_historicas"],
    "vehiculos": ["idvehiculo", "capacidad", "n_sentados", "n_depie"],
    "timbradas_segmentos": ["id_timbrada_segmento", "id_segmento", "id_viaje", "fecha",
                           "hora_inicio", "hora_fin", "subidas", "bajadas", "pasajeros"],
    "timbradas_anio": ["id_timbrada_anio", "ano", "id_ruta", "proceso", "id_dia_semana",
                      "id_estacionalidad_anio", "hora_inicio", "hora_fin", "subidas", "bajadas", "nro_viajes"],
}
prioritarios = []
for tabla, columnas in PRIORITARIOS_EST.items():
    vista = PERFIL_ESTRUCTURAL.loc[PERFIL_ESTRUCTURAL["Tabla"].eq(tabla) &
                                   PERFIL_ESTRUCTURAL["Columna"].isin(columnas)]
    if not vista.empty:
        vista = vista.set_index("Columna").reindex([c for c in columnas if c in set(vista["Columna"])]).reset_index()
        prioritarios.append(vista)
        nota_est(tabla + ": disponibilidad de campos prioritarios.")
        mostrar_est(vista[["Columna", "Filas evaluadas", "Nulos", "% nulos", "NaN numéricos",
                           "Cadenas vacías", "Solo espacios", "Etiquetas candidatas"]])
PERFIL_PRIORITARIO_EST = pd.concat(prioritarios, ignore_index=True) if prioritarios else PERFIL_ESTRUCTURAL.iloc[:0].copy()
guardar_est("campos_prioritarios", PERFIL_PRIORITARIO_EST)

Columna,Filas evaluadas,Nulos,% nulos,NaN numéricos,Cadenas vacías,Solo espacios,Etiquetas candidatas
idprogramacion,162678,0,0.00,0,0,0,0
idruta,162678,0,0.00,0,0,0,0
idvehiculo,162678,0,0.00,0,0,0,0
fechasalida,162678,0,0.00,0,0,0,0
fechainicio,162678,1083,0.67,0,0,0,0
fechafin,162678,16262,10.00,0,0,0,0
finalizado,162678,0,0.00,0,0,0,0
cancelada,162678,0,0.00,0,0,0,0
numeropasajeros,162678,2584,1.59,0,0,0,0
numerobajadas,162678,2584,1.59,0,0,0,0


Columna,Filas evaluadas,Nulos,% nulos,NaN numéricos,Cadenas vacías,Solo espacios,Etiquetas candidatas
idvehiculo,293,0,0.0,0,0,0,0
capacidad,293,0,0.0,0,0,0,0
n_sentados,293,0,0.0,0,0,0,0
n_depie,293,0,0.0,0,0,0,0


Columna,Filas evaluadas,Nulos,% nulos,NaN numéricos,Cadenas vacías,Solo espacios,Etiquetas candidatas
id_timbrada_segmento,799030,0,0.0,0,0,0,0
id_segmento,799030,0,0.0,0,0,0,0
id_viaje,799030,0,0.0,0,0,0,0
fecha,799030,0,0.0,0,0,0,0
hora_inicio,799030,0,0.0,0,0,0,0
hora_fin,799030,0,0.0,0,0,0,0
subidas,799030,0,0.0,0,0,0,0
bajadas,799030,0,0.0,0,0,0,0
pasajeros,799030,0,0.0,0,0,0,0


Columna,Filas evaluadas,Nulos,% nulos,NaN numéricos,Cadenas vacías,Solo espacios,Etiquetas candidatas
id_timbrada_anio,10374,0,0.0,0,0,0,0
ano,10374,0,0.0,0,0,0,0
id_ruta,10374,0,0.0,0,0,0,0
proceso,10374,0,0.0,0,0,0,0
id_dia_semana,10374,0,0.0,0,0,0,0
id_estacionalidad_anio,10374,0,0.0,0,0,0,0
hora_inicio,10374,0,0.0,0,0,0,0
hora_fin,10374,0,0.0,0,0,0,0
subidas,10374,0,0.0,0,0,0,0
bajadas,10374,0,0.0,0,0,0,0


### **6.3. Identificadores de cada registro**


Se recuperan las llaves PRIMARY KEY y UNIQUE visibles en LLAVES, conservando todas las columnas de las llaves compuestas. No se convierte una llave foránea en identificador único.

Si no hay una llave visible, se informa "sin llave de unicidad visible", no que la tabla carezca definitivamente de ella. No se inventa una combinación de campos solo para obtener unicidad.


In [92]:
# Formato de una llave acordada:
# {"nombre_tabla": [{"nombre": "identificador_acordado", "columnas": ["campo_real_1", "campo_real_2"]}]}
# No rellenar con candidatos sin confirmar su significado.
LLAVES_NEGOCIO_EST = globals().get("LLAVES_NEGOCIO_EST", {})

requeridas_llaves = {"tabla", "constraint_name", "constraint_type", "column_name", "ordinal_position"}
if not requeridas_llaves.issubset(LLAVES_EST.columns):
    raise ValueError("LLAVES no tiene el formato de metadatos del notebook.")
contratos_llave, inventario_llaves = {}, []
for tabla in tablas_perfilables:
    contratos_llave[tabla] = []
    grupo_tabla = LLAVES_EST.loc[LLAVES_EST["tabla"].eq(tabla) &
                                LLAVES_EST["constraint_type"].isin(["PRIMARY KEY", "UNIQUE"])]
    for nombre, g in grupo_tabla.groupby("constraint_name", sort=True):
        cols = g.sort_values("ordinal_position")["column_name"].dropna().drop_duplicates().tolist()
        if not cols:
            continue
        tipo = g.iloc[0]["constraint_type"]
        definicion = str(g.iloc[0].get("definicion", ""))
        contrato = {"nombre": str(nombre), "tipo": tipo, "columnas": cols,
                    "nulos_no_distintos": "NULLS NOT DISTINCT" in definicion.upper()}
        contratos_llave[tabla].append(contrato)
    for negocio in LLAVES_NEGOCIO_EST.get(tabla, []):
        contratos_llave[tabla].append({"nombre": negocio["nombre"], "tipo": "NEGOCIO ACORDADA",
            "columnas": list(negocio["columnas"]), "nulos_no_distintos": False})
    for contrato in contratos_llave[tabla]:
        cols = contrato["columnas"]
        if not cols or len(cols) != len(set(cols)) or not set(cols).issubset(set(CAMPOS[tabla])):
            raise ValueError("Llave con columnas no válidas: " + tabla + "/" + contrato["nombre"])
        inventario_llaves.append({"Tabla": tabla, "Llave": contrato["nombre"],
                                 "Tipo": contrato["tipo"], "Columnas": ", ".join(cols)})
    if not contratos_llave[tabla]:
        inventario_llaves.append({"Tabla": tabla, "Llave": "Sin llave de unicidad visible",
                                 "Tipo": "Pendiente de definición", "Columnas": ""})

INVENTARIO_LLAVES_EST = pd.DataFrame(inventario_llaves, columns=["Tabla", "Llave", "Tipo", "Columnas"])
guardar_est("inventario_llaves", INVENTARIO_LLAVES_EST)
mostrar_est(INVENTARIO_LLAVES_EST)
nota_est("Las llaves declaradas definen qué comprobar. Su cumplimiento en los archivos se evalúa a continuación.")

Tabla,Llave,Tipo,Columnas
Procesos,Procesos_pkey,PRIMARY KEY,proceso
dias_semana,pk_dias_semana,PRIMARY KEY,id_dia_semana
dias_semana,uq_dias_semana_dia_semana,UNIQUE,dia_semana
estacionalidades,pk_estacionalidades,PRIMARY KEY,id_estacionalidad
novedades_viales,pk_novedades_viales,PRIMARY KEY,id_novedad_vial
progvehiculos,progvehiculos_pkey,PRIMARY KEY,idprogramacion
rutas,Sin llave de unicidad visible,Pendiente de definición,
rutas_segmentos,rutas_segmentos_id_config_nro_segmento_key,UNIQUE,"id_config, nro_segmento"
rutas_segmentos,rutas_segmentos_pkey,PRIMARY KEY,id
rutas_segmentos_config,rutas_segmentos_config_idruta_idrutageo_key,UNIQUE,"idruta, idrutageo"


### **6.4. Duplicados exactos y repeticiones de llaves**

Surgen los siguientes interrogantes:
1. ¿Hay filas con todos los valores iguales en la representación extraída?
2. ¿Hay filas que comparten una llave, aunque otros valores sean distintos?

* Una misma llave repetida puede corresponder a copias exactas o a versiones contradictorias; no se resuelve con un drop_duplicates automático. Varias filas del mismo vehículo o segmento tampoco son duplicados por sí mismas.
* La revisión es global por tabla, incluidas repeticiones entre partes y lotes. Una fila repetida dos veces tiene un grupo duplicado, dos filas involucradas y una repetición adicional. Reportamos las tres cantidades.
* Para evitar cargar tablas completas, se comparan representaciones completas de valores y llaves mediante una base auxiliar SQLite local. No se decide igualdad únicamente mediante un hash. El espacio.
* La base auxiliar se elimina al concluir y los informes se conservan.
* Los nulos se cuentan aparte en las llaves. En UNIQUE ordinaria no se exige unicidad a combinaciones que contienen NULL; si el metadato declara NULLS NOT DISTINCT, también se comparan esas combinaciones.
* Una llave PRIMARY KEY o de negocio acordada exige completar todas sus columnas. 
* La evaluación es estructural: no reproduce reglas especiales de colación, índices de expresión o igualdad semántica de geometrías/JSON.

In [94]:
# Serialización conservadora: conserva espacios, mayúsculas y tipos.
# No convierte None en cadena "None" ni la cadena "NULL" en un nulo.
def valor_est(v):
    if v is None:
        return ["null"]
    if isinstance(v, bool):
        return ["bool", v]
    if isinstance(v, int):
        return ["int", str(v)]
    if isinstance(v, float):
        if math.isnan(v):
            return ["float", "NaN"]
        return ["float", v.hex() if v else "0"]
    if isinstance(v, Decimal):
        return ["decimal", str(v)]  # Se conserva escala de la representación.
    if isinstance(v, str):
        return ["text", v]
    if isinstance(v, (bytes, bytearray)):
        return ["bytes", bytes(v).hex()]
    if isinstance(v, datetime):
        return ["datetime", v.isoformat()]
    if isinstance(v, date):
        return ["date", v.isoformat()]
    if isinstance(v, time):
        return ["time", v.isoformat()]
    if isinstance(v, timedelta):
        return ["timedelta", v.days, v.seconds, v.microseconds]
    if isinstance(v, list):
        return ["list", [valor_est(x) for x in v]]
    if isinstance(v, tuple):
        return ["tuple", [valor_est(x) for x in v]]
    if isinstance(v, dict):
        return ["dict", [[str(k), valor_est(x)] for k, x in sorted(v.items(), key=lambda z: str(z[0]))]]
    raise TypeError("Tipo de valor no soportado para igualdad exacta: " + type(v).__name__)

def firma_est(valores):
    return json.dumps([valor_est(v) for v in valores], ensure_ascii=False,
                      separators=(",", ":")).encode("utf-8")

def estadisticas_repeticion(conn, alcance):
    return conn.execute("""SELECT COUNT(*), COALESCE(SUM(n),0), COALESCE(SUM(n-1),0)
        FROM conteos WHERE alcance=? AND n>1""", (alcance,)).fetchone()

duplicados_est, resultados_llaves_est, muestras_est = [], [], []
for tabla in tablas_perfilables:
    if not EVALUAR_REPETICIONES:
        duplicados_est.append({"Tabla": tabla, "Filas evaluadas": None, "Grupos repetidos": None,
            "Filas involucradas": None, "Repeticiones adicionales": None, "Estado": "Pendiente por configuración"})
        continue
    columnas = list(CAMPOS[tabla])
    llaves = contratos_llave[tabla]
    nulos_llave = [0] * len(llaves)
    evaluadas_llave = [0] * len(llaves)
    ruta_aux = EST_DIR / "auxiliares" / (hashlib.sha256(tabla.encode()).hexdigest()[:16] + ".sqlite")
    conn = None
    n = 0
    try:
        conn = sqlite3.connect(ruta_aux)
        conn.execute("PRAGMA cache_size=-16384")
        conn.execute("CREATE TABLE conteos (alcance INTEGER, valor BLOB, n INTEGER, PRIMARY KEY(alcance,valor)) WITHOUT ROWID")
        insertar = """INSERT INTO conteos(alcance,valor,n) VALUES(?,?,1)
                      ON CONFLICT(alcance,valor) DO UPDATE SET n=n+1"""
        archivos = sorted(RAW_EST[tabla].glob("parte-*.parquet"))
        for num_parte, archivo in enumerate(archivos, start=1):
            print(f"Duplicados y llaves: {tabla}, parte {num_parte}/{len(archivos)}", flush=True)
            for batch in pq.ParquetFile(archivo).iter_batches(batch_size=LOTE_EST, columns=columnas):
                filas = batch.to_pylist()
                conn.executemany(insertar, ((0, firma_est([r[c] for c in columnas])) for r in filas))
                for i, llave in enumerate(llaves):
                    valores_validos = []
                    for r in filas:
                        valores = [r[c] for c in llave["columnas"]]
                        hay_nulo = any(v is None for v in valores)
                        nulos_llave[i] += int(hay_nulo)
                        if hay_nulo and not llave["nulos_no_distintos"]:
                            continue
                        valores_validos.append((i+1, firma_est(valores)))
                    evaluadas_llave[i] += len(valores_validos)
                    conn.executemany(insertar, valores_validos)
                conn.commit()
                n += len(filas)
        esperado = int(ESTRUCTURA.set_index("Tabla").loc[tabla, "Filas"])
        if n != esperado:
            raise ValueError("Lectura de duplicados incompleta")
        g, involucradas, adicionales = estadisticas_repeticion(conn, 0)
        duplicados_est.append({"Tabla": tabla, "Filas evaluadas": n, "Grupos repetidos": g,
            "Filas involucradas": involucradas, "Repeticiones adicionales": adicionales, "Estado": "Evaluado"})
        for i, llave in enumerate(llaves):
            g, involucradas, adicionales = estadisticas_repeticion(conn, i+1)
            nulos_problematicos = nulos_llave[i] if llave["tipo"] in {"PRIMARY KEY", "NEGOCIO ACORDADA"} else 0
            estado = "Revisar" if adicionales or nulos_problematicos else "Sin incidencias en lo evaluado"
            resultados_llaves_est.append({"Tabla": tabla, "Llave": llave["nombre"], "Tipo": llave["tipo"],
                "Columnas": ", ".join(llave["columnas"]), "Filas totales": n,
                "Filas con algún NULL en llave": nulos_llave[i],
                "Filas evaluadas para repetición": evaluadas_llave[i], "Grupos repetidos": g,
                "Filas involucradas": involucradas, "Repeticiones adicionales": adicionales, "Estado": estado})
            # Solo muestras de valores de llave; no exportar todas las filas completas.
            for payload, cantidad in conn.execute(
                    "SELECT valor,n FROM conteos WHERE alcance=? AND n>1 ORDER BY n DESC LIMIT 5", (i+1,)):
                muestras_est.append({"Tabla": tabla, "Llave": llave["nombre"],
                    "Valores tipados": payload.decode("utf-8"), "Apariciones": cantidad})
    except Exception as exc:
        duplicados_est.append({"Tabla": tabla, "Filas evaluadas": n, "Grupos repetidos": None,
            "Filas involucradas": None, "Repeticiones adicionales": None, "Estado": "Lectura pendiente"})
        incidencias_est.append({"Tabla": tabla, "Comprobación": "Duplicados y llaves",
                                "Detalle": type(exc).__name__ + ": " + str(exc)})
    finally:
        if conn is not None:
            conn.close()
        if ruta_aux.exists():
            ruta_aux.unlink()

DUPLICADOS_EXACTOS_EST = pd.DataFrame(duplicados_est, columns=[
    "Tabla", "Filas evaluadas", "Grupos repetidos", "Filas involucradas", "Repeticiones adicionales", "Estado"])
RESULTADOS_LLAVES_EST = pd.DataFrame(resultados_llaves_est, columns=[
    "Tabla", "Llave", "Tipo", "Columnas", "Filas totales", "Filas con algún NULL en llave",
    "Filas evaluadas para repetición", "Grupos repetidos", "Filas involucradas", "Repeticiones adicionales", "Estado"])
guardar_est("duplicados_exactos", DUPLICADOS_EXACTOS_EST)
guardar_est("resultados_llaves", RESULTADOS_LLAVES_EST)
guardar_est("muestras_llaves_repetidas", pd.DataFrame(muestras_est, columns=[
    "Tabla", "Llave", "Valores tipados", "Apariciones"]))

nota_est("Duplicados exactos: evaluación de filas completas, sin modificar los originales.")
mostrar_est(DUPLICADOS_EXACTOS_EST)
nota_est("Llaves: NULL y repeticiones se muestran por separado. Los ejemplos completos de llaves están en CSV.")
mostrar_est(RESULTADOS_LLAVES_EST)
llaves_revisar = RESULTADOS_LLAVES_EST.query("Estado == 'Revisar'")
if not llaves_revisar.empty:
    mostrar_est(llaves_revisar)
sin_llave = INVENTARIO_LLAVES_EST.query("Tipo == 'Pendiente de definición'")
if not sin_llave.empty:
    nota_est("Estas fuentes necesitan confirmar una llave de negocio antes de exigir unicidad analítica.")
    mostrar_est(sin_llave)

Duplicados y llaves: Procesos, parte 1/1
Duplicados y llaves: dias_semana, parte 1/1
Duplicados y llaves: estacionalidades, parte 1/1
Duplicados y llaves: novedades_viales, parte 1/1
Duplicados y llaves: progvehiculos, parte 1/17
Duplicados y llaves: progvehiculos, parte 2/17
Duplicados y llaves: progvehiculos, parte 3/17
Duplicados y llaves: progvehiculos, parte 4/17
Duplicados y llaves: progvehiculos, parte 5/17
Duplicados y llaves: progvehiculos, parte 6/17
Duplicados y llaves: progvehiculos, parte 7/17
Duplicados y llaves: progvehiculos, parte 8/17
Duplicados y llaves: progvehiculos, parte 9/17
Duplicados y llaves: progvehiculos, parte 10/17
Duplicados y llaves: progvehiculos, parte 11/17
Duplicados y llaves: progvehiculos, parte 12/17
Duplicados y llaves: progvehiculos, parte 13/17
Duplicados y llaves: progvehiculos, parte 14/17
Duplicados y llaves: progvehiculos, parte 15/17
Duplicados y llaves: progvehiculos, parte 16/17
Duplicados y llaves: progvehiculos, parte 17/17
Duplicados

Tabla,Filas evaluadas,Grupos repetidos,Filas involucradas,Repeticiones adicionales,Estado
Procesos,1,0,0,0,Evaluado
dias_semana,7,0,0,0,Evaluado
estacionalidades,24,0,0,0,Evaluado
novedades_viales,59,0,0,0,Evaluado
progvehiculos,162678,0,0,0,Evaluado
rutas,1,0,0,0,Evaluado
rutas_segmentos,135,0,0,0,Evaluado
rutas_segmentos_config,1,0,0,0,Evaluado
rutas_segmentos_porcion,336089,0,0,0,Evaluado
spatial_ref_sys,8500,0,0,0,Evaluado


Tabla,Llave,Tipo,Columnas,Filas totales,Filas con algún NULL en llave,Filas evaluadas para repetición,Grupos repetidos,Filas involucradas,Repeticiones adicionales,Estado
Procesos,Procesos_pkey,PRIMARY KEY,proceso,1,0,1,0,0,0,Sin incidencias en lo evaluado
dias_semana,pk_dias_semana,PRIMARY KEY,id_dia_semana,7,0,7,0,0,0,Sin incidencias en lo evaluado
dias_semana,uq_dias_semana_dia_semana,UNIQUE,dia_semana,7,0,7,0,0,0,Sin incidencias en lo evaluado
estacionalidades,pk_estacionalidades,PRIMARY KEY,id_estacionalidad,24,0,24,0,0,0,Sin incidencias en lo evaluado
novedades_viales,pk_novedades_viales,PRIMARY KEY,id_novedad_vial,59,0,59,0,0,0,Sin incidencias en lo evaluado
progvehiculos,progvehiculos_pkey,PRIMARY KEY,idprogramacion,162678,0,162678,0,0,0,Sin incidencias en lo evaluado
rutas_segmentos,rutas_segmentos_id_config_nro_segmento_key,UNIQUE,"id_config, nro_segmento",135,0,135,0,0,0,Sin incidencias en lo evaluado
rutas_segmentos,rutas_segmentos_pkey,PRIMARY KEY,id,135,0,135,0,0,0,Sin incidencias en lo evaluado
rutas_segmentos_config,rutas_segmentos_config_idruta_idrutageo_key,UNIQUE,"idruta, idrutageo",1,0,1,0,0,0,Sin incidencias en lo evaluado
rutas_segmentos_config,rutas_segmentos_config_pkey,PRIMARY KEY,id,1,0,1,0,0,0,Sin incidencias en lo evaluado


Tabla,Llave,Tipo,Columnas
rutas,Sin llave de unicidad visible,Pendiente de definición,


In [95]:
cierre_est = []
for _, fila in ESTRUCTURA.iterrows():
    tabla = fila["Tabla"]
    estructura_ok = fila["Estado"] == "Lista para perfilado"
    perfil_t = PERFIL_ESTRUCTURAL.loc[PERFIL_ESTRUCTURAL["Tabla"].eq(tabla)]
    lectura_t = LECTURAS_ESTRUCTURALES.loc[LECTURAS_ESTRUCTURALES["Tabla"].eq(tabla)]
    dup_t = DUPLICADOS_EXACTOS_EST.loc[DUPLICADOS_EXACTOS_EST["Tabla"].eq(tabla)]
    llave_t = RESULTADOS_LLAVES_EST.loc[RESULTADOS_LLAVES_EST["Tabla"].eq(tabla)]
    tipo_t = TIPOS_ESTRUCTURALES.loc[TIPOS_ESTRUCTURALES["Tabla"].eq(tabla) &
                                   TIPOS_ESTRUCTURALES["Estado"].isin(["Revisar", "Revisión manual"])]
    alertas = []
    if estructura_ok:
        if lectura_t.empty or not lectura_t["Estado"].eq("Perfil completo").all():
            alertas.append("Perfilado pendiente")
        if not tipo_t.empty:
            alertas.append("Revisar tipos")
        if perfil_t["Nulos incompatibles con NOT NULL"].sum() > 0:
            alertas.append("Nulos en NOT NULL")
        if dup_t.empty or not dup_t["Estado"].eq("Evaluado").all():
            alertas.append("Duplicados/llaves pendientes")
        elif dup_t["Repeticiones adicionales"].sum() > 0:
            alertas.append("Filas completas repetidas")
        if not contratos_llave.get(tabla):
            alertas.append("Confirmar llave de negocio")
        elif len(llave_t) != len(contratos_llave[tabla]):
            alertas.append("Llaves pendientes")
        if not llave_t.empty and llave_t["Estado"].eq("Revisar").any():
            alertas.append("Incidencias de llave")
    estado = ("Revisar" if alertas else "Comprobaciones estructurales completadas") if estructura_ok else fila["Estado"]
    cierre_est.append({"Tabla": tabla, "Estado": estado, "Acción o pendiente": "; ".join(alertas) or fila["Detalle"]})

CIERRE_ESTRUCTURAL = pd.DataFrame(cierre_est)
guardar_est("cierre_estructural", CIERRE_ESTRUCTURAL)
INCIDENCIAS_ESTRUCTURALES = pd.DataFrame(incidencias_est, columns=["Tabla", "Comprobación", "Detalle"])
guardar_est("incidencias_ejecucion", INCIDENCIAS_ESTRUCTURALES)
(EST_DIR / "metodologia.json").write_text(json.dumps({
    "auditoria_id": EST_ID, "configuracion": CFG, "excluidos_tecnicos": sorted(TECNICOS_EST),
    "diferidos": sorted(DIFERIDOS_EST), "duplicados_activados": EVALUAR_REPETICIONES,
    "llaves_negocio_acordadas": LLAVES_NEGOCIO_EST, "etiquetas_candidatas": sorted(ETIQUETAS_AUSENCIA),
    "llaves_declaradas": inventario_llaves,
    "reglas": {"duplicado": "Igualdad de representación tipada completa entre todas las partes",
               "nulo": "NULL Arrow; NaN numérico reportado aparte",
               "tratamiento": "Ninguno; conserva RAW"},
    "limites": ["No evalúa aquí coherencia operativa ni integridad de referencias.",
               "No reproduce igualdad por colación o equivalencia espacial/JSON.",
               "No clasifica el mecanismo de faltantes.",
               "No garantiza identidad con una instantánea pasada de PostgreSQL."]
}, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
mostrar_est(CIERRE_ESTRUCTURAL)
if not INCIDENCIAS_ESTRUCTURALES.empty:
    nota_est("Hay errores o diferencias de ejecución que requieren revisión; no se consideran resultados completos.")
    mostrar_est(INCIDENCIAS_ESTRUCTURALES, n=15)
nota_est("RAW se conserva. Resolver los pendientes de la fuente o unión afectada antes de aplicar limpieza.")
print("Reportes:", EST_DIR.resolve())


Tabla,Estado,Acción o pendiente
Procesos,Revisar,Revisar tipos
conteo_pasajeros_destino,Diferida,Sin lectura de contenido
dias_semana,Comprobaciones estructurales completadas,
estacionalidades,Comprobaciones estructurales completadas,
estacionalidades_anio,Vacía en el filtro,
novedades_viales,Comprobaciones estructurales completadas,
novedades_viales_fecha,Vacía en el filtro,
novedades_viales_ruta,Vacía en el filtro,
progvehiculos,Revisar,Revisar tipos
rutas,Revisar,Revisar tipos; Confirmar llave de negocio


Reportes: /Users/leoporojano/Documents/Data Challenge/salidas_extraccion/ruta_2/20261010T182243_04b6bd0f/reportes/calidad_estructural_20261011T183320_033220


In [96]:
detalle_tipos = (
    TIPOS_ESTRUCTURALES.loc[
        TIPOS_ESTRUCTURALES["Estado"].isin([
            "Revisar", "Revisión manual"
        ]),
        [
            "Tabla", "Columna", "Tipo SQL",
            "Tipo Parquet", "Estado", "Detalle"
        ]
    ]
    .drop_duplicates()
    .merge(
        PERFIL_ESTRUCTURAL[
            ["Tabla", "Columna", "Filas evaluadas", "Nulos", "% nulos"]
        ],
        on=["Tabla", "Columna"],
        how="left",
        validate="many_to_one"
    )
)

mostrar_est(detalle_tipos)

Tabla,Columna,Tipo SQL,Tipo Parquet,Estado,Detalle,Filas evaluadas,Nulos,% nulos
Procesos,recaudo_comision_conductor,numeric,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,fecha_inicio_liquidacion,date,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,latitud,numeric,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,longitud,numeric,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,valor_operador_relacional_bloqueo_pico,numeric,string,Revisar,Familia de tipo distinta,1,0,0.00
Procesos,valor_tiempo_minimo_viaje,int4,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,id_estacionalidad,int8,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,ultima_estacionalidad,date,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,id_diasemana,int4,string,Revisar,Familia de tipo distinta,1,1,100.00
progvehiculos,fechaliq,timestamp,string,Revisar,Familia de tipo distinta,162678,162678,100.00


In [97]:

from pathlib import Path
from decimal import Decimal, InvalidOperation
from datetime import datetime, date, time
from zoneinfo import ZoneInfo
import json
import re
import math
import shutil
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

for nombre in ["RAW", "BASE", "CFG", "ESQUEMA"]:
    if nombre not in globals():
        raise RuntimeError("Falta ejecutar: " + nombre)
TECNICOS_TIPO = {"geometry_columns", "geography_columns", "raster_columns", "raster_overviews"}
ORIGINALES_TIPO = {t: Path(p) for t, p in RAW.items() if t not in TECNICOS_TIPO}
if any(not p.is_dir() for p in ORIGINALES_TIPO.values()):
    raise ValueError("RAW debe contener carpetas existentes.")
TIPADO_ID = datetime.now(ZoneInfo(CFG["zona"])).strftime("%Y%m%dT%H%M%S_%f")
TIPADO_DIR = Path(BASE) / ("tipado_" + TIPADO_ID)
TIPADO_DIR.mkdir(exist_ok=False)
(TIPADO_DIR / "reportes").mkdir()
LOTE_TIPO = min(int(CFG.get("lote", 10000)), 10000)
CONTRATOS_TIPO, RAW_TIPADO = {}, {}
RESULTADO_TIPADO, ERRORES_TIPADO, DETALLE_TIPADO = [], [], []

def decimal_estricto(v):
    if isinstance(v, bool) or isinstance(v, float):
        raise ValueError("Numeric llegó como bool/float; verificar precisión antes de convertir")
    d = v if isinstance(v, Decimal) else Decimal(str(v))
    if not d.is_finite():
        raise ValueError("Numeric no finito: requiere tratamiento explícito")
    return d

def entero_estricto(v, bits):
    d = Decimal(str(v))
    if isinstance(v, bool) or not d.is_finite() or d != d.to_integral_value():
        raise ValueError("Valor no entero")
    n = int(d)
    if n < -(2**(bits-1)) or n >= 2**(bits-1):
        raise ValueError("Fuera del rango entero SQL")
    return n

def validar_microsegundos(s):
    m = re.search(r"[.,](\d+)", s)
    if m and len(m.group(1)) > 6 and any(c != "0" for c in m.group(1)[6:]):
        raise ValueError("Precisión temporal superior a microsegundos")
    return s

def convertir_valor_tipo(v, sql):
    if v is None:
        return None
    if sql in {"int2", "int4", "int8"}:
        return entero_estricto(v, {"int2":16, "int4":32, "int8":64}[sql])
    if sql == "numeric":
        return decimal_estricto(v)
    if sql == "date":
        if isinstance(v, datetime):
            raise ValueError("Timestamp recibido donde se esperaba fecha")
        return v if isinstance(v, date) else date.fromisoformat(str(v))
    if sql in {"time", "timetz"}:
        x = v if isinstance(v, time) else time.fromisoformat(validar_microsegundos(str(v)))
        if sql == "timetz":
            raise ValueError("Hora con zona requiere contrato específico")
        if x.tzinfo is not None:
            raise ValueError("Zona inesperada en SQL time")
        return x
    if sql in {"timestamp", "timestamptz"}:
        x = v if isinstance(v, datetime) else datetime.fromisoformat(validar_microsegundos(str(v)))
        tiene_zona = x.tzinfo is not None and x.utcoffset() is not None
        if (sql == "timestamptz") != tiene_zona:
            raise ValueError("Zona temporal incompatible con el tipo SQL")
        return x
    return v

def dimensiones_decimal(d):
    digitos, exponente = len(d.as_tuple().digits), d.as_tuple().exponent
    return max(digitos+exponente, 0), max(-exponente, 0)

def construir_contrato_tipo(tabla):
    meta = ESQUEMA.loc[ESQUEMA["tabla"].eq(tabla)].set_index("columna")
    if meta.empty or meta.index.duplicated().any():
        raise ValueError("Metadatos ausentes o duplicados")
    archivos = sorted(ORIGINALES_TIPO[tabla].glob("parte-*.parquet"))
    if not archivos:
        return None, meta
    nombres = pq.ParquetFile(archivos[0]).schema_arrow.names
    if set(nombres) != set(meta.index):
        raise ValueError("Columnas de archivo diferentes del esquema SQL")
    tipos_originales = {}
    for archivo in archivos:
        esquema = pq.ParquetFile(archivo).schema_arrow
        if set(esquema.names) != set(nombres):
            raise ValueError("Columnas diferentes entre partes")
        for f in esquema:
            if not pa.types.is_null(f.type):
                tipos_originales.setdefault(f.name, set()).add(f.type)
    numericos = [c for c in nombres if meta.loc[c, "udt_name"] == "numeric"]
    dimensiones = {c:[0, 0] for c in numericos}
    # Primera pasada solo por numeric: determinar precisión/escala globales sin redondeo.
    if numericos:
        for archivo in archivos:
            for batch in pq.ParquetFile(archivo).iter_batches(batch_size=LOTE_TIPO, columns=numericos):
                for c, a in zip(batch.schema.names, batch.columns):
                    for v in a.to_pylist():
                        if v is None:
                            continue
                        enteros, escala = dimensiones_decimal(decimal_estricto(v))
                        dimensiones[c][0] = max(dimensiones[c][0], enteros)
                        dimensiones[c][1] = max(dimensiones[c][1], escala)
    campos = []
    for c in nombres:
        sql = str(meta.loc[c, "udt_name"])
        if sql in {"int2","int4","int8"}:
            tipo = {"int2":pa.int16(), "int4":pa.int32(), "int8":pa.int64()}[sql]
        elif sql == "numeric":
            enteros, escala = dimensiones[c]
            precision_meta = meta.loc[c].get("numeric_precision")
            escala_meta = meta.loc[c].get("numeric_scale")
            if pd.notna(precision_meta) and pd.notna(escala_meta):
                p, s = int(precision_meta), int(escala_meta)
                # Arrow requiere escala no negativa; incorporar dominio SQL declarado.
                escala = max(escala, s, 0)
                enteros = max(enteros, p-s, 0)
            precision = max(1, enteros+escala, escala)
            if precision > 76:
                raise ValueError("Numeric excede precisión Arrow de 76 dígitos: " + c)
            tipo = pa.decimal128(precision, escala) if precision <= 38 else pa.decimal256(precision, escala)
        elif sql == "date":
            tipo = pa.date32()
        elif sql == "time":
            tipo = pa.time64("us")
        elif sql == "timestamp":
            tipo = pa.timestamp("us")
        elif sql == "timestamptz":
            tipo = pa.timestamp("us", tz=CFG["zona"])
        elif sql in {"text","varchar","bpchar","geometry","geography"}:
            tipo = pa.string()
        else:
            originales = tipos_originales.get(c, set())
            if len(originales) > 1:
                raise ValueError("Tipo no cubierto cambia entre partes: " + c)
            tipo = next(iter(originales), pa.null())
        campos.append(pa.field(c, tipo, nullable=True))
    return pa.schema(campos), meta

def convertir_batch_tipo(batch, contrato, meta):
    arrays = []
    for f in contrato:
        a = batch.column(batch.schema.get_field_index(f.name))
        sql = str(meta.loc[f.name, "udt_name"])
        if sql in {"int2","int4","int8","numeric","date","time","timetz","timestamp","timestamptz"}:
            valores = []
            for indice, v in enumerate(a.to_pylist()):
                try:
                    valores.append(convertir_valor_tipo(v, sql))
                except Exception as exc:
                    raise ValueError(f"Columna {f.name}, fila de lote {indice}: {type(exc).__name__}: {exc}") from exc
            nuevo = pa.array(valores, type=f.type, safe=True)
        else:
            nuevo = a if a.type == f.type else a.cast(f.type, safe=True)
        if nuevo.null_count != a.null_count:
            raise ValueError("Cambio inesperado de nulos: " + f.name)
        arrays.append(nuevo)
    return pa.Table.from_arrays(arrays, schema=contrato)

def verificar_batch_tipo(original, nuevo, meta):
    if original.num_rows != nuevo.num_rows:
        raise ValueError("Cambio de cantidad de filas")
    for c in nuevo.schema.names:
        antes = original.column(original.schema.get_field_index(c)).to_pylist()
        despues = nuevo.column(c).to_pylist()
        sql = str(meta.loc[c, "udt_name"])
        for v, w in zip(antes, despues):
            if v is None or w is None:
                if v is not None or w is not None:
                    raise ValueError("Cambio de nulo: " + c)
                continue
            esperado = convertir_valor_tipo(v, sql)
            if isinstance(esperado, float) and isinstance(w, float) and math.isnan(esperado) and math.isnan(w):
                continue
            if esperado != w:
                raise ValueError("Cambio del valor interpretado: " + c)

for tabla, carpeta in sorted(ORIGINALES_TIPO.items()):
    print("Preparando copia tipada:", tabla, flush=True)
    temporal = TIPADO_DIR / (tabla + ".incompleta")
    destino = TIPADO_DIR / tabla
    original_filas, tipadas_filas, partes = 0, 0, 0
    try:
        contrato, meta = construir_contrato_tipo(tabla)
        temporal.mkdir()
        if contrato is not None:
            for archivo in sorted(carpeta.glob("parte-*.parquet")):
                print("  Parte:", archivo.name, flush=True)
                escritor = None
                try:
                    for batch in pq.ParquetFile(archivo).iter_batches(batch_size=LOTE_TIPO):
                        convertido = convertir_batch_tipo(batch, contrato, meta)
                        verificar_batch_tipo(batch, convertido, meta)
                        if escritor is None:
                            escritor = pq.ParquetWriter(temporal / archivo.name, contrato)
                        escritor.write_table(convertido)
                        original_filas += batch.num_rows
                finally:
                    if escritor is not None:
                        escritor.close()
                partes += 1
            # Verificar bytes publicados: lectura completa y comparación de valores por lote.
            for archivo in sorted(carpeta.glob("parte-*.parquet")):
                archivo_nuevo = temporal / archivo.name
                old_pf, new_pf = pq.ParquetFile(archivo), pq.ParquetFile(archivo_nuevo)
                if new_pf.schema_arrow != contrato or old_pf.metadata.num_rows != new_pf.metadata.num_rows:
                    raise ValueError("Esquema o filas no coinciden en archivo tipado")
                # Lectura de lotes del mismo tamaño; alineación verificada explícitamente.
                from itertools import zip_longest
                for old_batch, new_batch in zip_longest(
                    old_pf.iter_batches(batch_size=LOTE_TIPO),
                    new_pf.iter_batches(batch_size=LOTE_TIPO)):
                    if old_batch is None or new_batch is None or old_batch.num_rows != new_batch.num_rows:
                        raise ValueError("Lotes de lectura no alineados")
                    verificar_batch_tipo(old_batch, pa.Table.from_batches([new_batch]), meta)
                tipadas_filas += new_pf.metadata.num_rows
            if original_filas != tipadas_filas:
                raise ValueError("Conteo de lectura y escritura distinto")
        temporal.replace(destino)
        RAW_TIPADO[tabla] = destino
        if contrato is not None:
            CONTRATOS_TIPO[tabla] = contrato
            for f in contrato:
                DETALLE_TIPADO.append({"Tabla":tabla,"Columna":f.name,
                    "Tipo SQL":str(meta.loc[f.name,"udt_name"]),"Tipo destino":str(f.type)})
        RESULTADO_TIPADO.append({"Tabla":tabla,"Filas originales":original_filas,
            "Filas tipadas":tipadas_filas,"Partes":partes,
            "Estado":"Vacía en el filtro" if contrato is None else "Copia tipada verificada"})
    except Exception as exc:
        shutil.rmtree(temporal, ignore_errors=True)
        ERRORES_TIPADO.append({"Tabla":tabla,"Error":type(exc).__name__,"Detalle":str(exc)})
        RESULTADO_TIPADO.append({"Tabla":tabla,"Filas originales":None,
            "Filas tipadas":None,"Partes":None,"Estado":"Pendiente; original conservado"})

REPORTE_TIPADO = pd.DataFrame(RESULTADO_TIPADO)
pd.DataFrame(DETALLE_TIPADO).to_csv(TIPADO_DIR/"reportes"/"contratos_tipos.csv",index=False)
REPORTE_TIPADO.to_csv(TIPADO_DIR/"reportes"/"conversiones.csv",index=False)
ERRORES_CONVERSION = pd.DataFrame(ERRORES_TIPADO,columns=["Tabla","Error","Detalle"])
ERRORES_CONVERSION.to_csv(TIPADO_DIR/"reportes"/"pendientes.csv",index=False)
# Guardar contrato Arrow para reutilizarlo incluso tras reiniciar el kernel.
for tabla, contrato in CONTRATOS_TIPO.items():
    (TIPADO_DIR/"reportes"/(tabla+".arrow_schema")).write_bytes(contrato.serialize().to_pybytes())
(TIPADO_DIR/"procedencia.json").write_text(json.dumps({
    "origen":{t:str(p) for t,p in ORIGINALES_TIPO.items()},
    "destino":{t:str(p) for t,p in RAW_TIPADO.items()},
    "configuracion":CFG,
    "verificacion":"Filas, nulos y valores interpretados, incluida lectura de Parquet escrito",
    "sin_tratamientos":"No imputación, exclusión de filas ni corrección operativa",
},ensure_ascii=False,indent=2,default=str),encoding="utf-8")
display(REPORTE_TIPADO)
if not ERRORES_CONVERSION.empty:
    display(ERRORES_CONVERSION)
print("Originales conservados. Copias:",TIPADO_DIR.resolve())


Preparando copia tipada: Procesos
  Parte: parte-00000.parquet
Preparando copia tipada: dias_semana
  Parte: parte-00000.parquet
Preparando copia tipada: estacionalidades
  Parte: parte-00000.parquet
Preparando copia tipada: estacionalidades_anio
Preparando copia tipada: novedades_viales
  Parte: parte-00000.parquet
Preparando copia tipada: novedades_viales_fecha
Preparando copia tipada: novedades_viales_ruta
Preparando copia tipada: progvehiculos
  Parte: parte-00000.parquet
  Parte: parte-00001.parquet
  Parte: parte-00002.parquet
  Parte: parte-00003.parquet
  Parte: parte-00004.parquet
  Parte: parte-00005.parquet
  Parte: parte-00006.parquet
  Parte: parte-00007.parquet
  Parte: parte-00008.parquet
  Parte: parte-00009.parquet
  Parte: parte-00010.parquet
  Parte: parte-00011.parquet
  Parte: parte-00012.parquet
  Parte: parte-00013.parquet
  Parte: parte-00014.parquet
  Parte: parte-00015.parquet
  Parte: parte-00016.parquet
Preparando copia tipada: rutas
  Parte: parte-00000.par

,Tabla,Filas originales,Filas tipadas,Partes,Estado
0,Procesos,1,1,1,Copia tipada verificada
1,dias_semana,7,7,1,Copia tipada verificada
2,estacionalidades,24,24,1,Copia tipada verificada
3,estacionalidades_anio,0,0,0,Vacía en el filtro
4,novedades_viales,59,59,1,Copia tipada verificada
5,novedades_viales_fecha,0,0,0,Vacía en el filtro
6,novedades_viales_ruta,0,0,0,Vacía en el filtro
7,progvehiculos,162678,162678,17,Copia tipada verificada
8,rutas,1,1,1,Copia tipada verificada
9,rutas_segmentos,135,135,1,Copia tipada verificada


Originales conservados. Copias: /Users/leoporojano/Documents/Data Challenge/salidas_extraccion/ruta_2/20261010T182243_04b6bd0f/tipado_20261011T231118_417889


In [98]:
detalle_tipos = (
    TIPOS_ESTRUCTURALES.loc[
        TIPOS_ESTRUCTURALES["Estado"].isin([
            "Revisar", "Revisión manual"
        ]),
        [
            "Tabla", "Columna", "Tipo SQL",
            "Tipo Parquet", "Estado", "Detalle"
        ]
    ]
    .drop_duplicates()
    .merge(
        PERFIL_ESTRUCTURAL[
            ["Tabla", "Columna", "Filas evaluadas", "Nulos", "% nulos"]
        ],
        on=["Tabla", "Columna"],
        how="left",
        validate="many_to_one"
    )
)

mostrar_est(detalle_tipos)

Tabla,Columna,Tipo SQL,Tipo Parquet,Estado,Detalle,Filas evaluadas,Nulos,% nulos
Procesos,recaudo_comision_conductor,numeric,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,fecha_inicio_liquidacion,date,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,latitud,numeric,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,longitud,numeric,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,valor_operador_relacional_bloqueo_pico,numeric,string,Revisar,Familia de tipo distinta,1,0,0.00
Procesos,valor_tiempo_minimo_viaje,int4,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,id_estacionalidad,int8,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,ultima_estacionalidad,date,string,Revisar,Familia de tipo distinta,1,1,100.00
Procesos,id_diasemana,int4,string,Revisar,Familia de tipo distinta,1,1,100.00
progvehiculos,fechaliq,timestamp,string,Revisar,Familia de tipo distinta,162678,162678,100.00
